# HBB Regulatory Variant Evidence Pipeline (Layers 0–2, Audited v4)

**Purpose:** optionally regenerate the Layer 0 candidate truth set, collect Layer 1 evidence (GWAS Catalog v2, GTEx eQTL, gnomAD, MPRA, optional CRISPR), and build the Layer 2 modality-availability matrix, AlphaGenome trigger queue and evidence-agreement queue.

**Scientific boundary:** this notebook does not retrieve AlphaGenome predictions; Layer 2 only creates a prioritized queue. GWAS evidence is position-level, not allele-specific, and does not count toward evidence tiers. GTEx non-hits are *unverified*, not tested-null. A failed query is a technical unknown, never biological absence. Synthetic fixtures are used only in unit tests.

## What changed in v3/v4
- **Layer 0 is runnable again** (Sections 3b/4b). It is off by default (`layer0.run_layer0: false`); with it on, the notebook rebuilds the candidate set from ClinVar, Ensembl VEP and ENCODE cCREs. Either way it writes `layer0_outputs/HBB_layer0_manifest.json`, so gene metadata is pinned instead of re-fetched.
- **`configs/hbb.yaml` is created automatically** (never overwritten) and holds the ClinVar date, target cell type (HEL92.1.7), tissues, tier thresholds and the failure-gate switch.
- **GWAS failures are no longer silent.** HTTP 410 is retried, an unrecognised response shape is an error (not "not in catalog"), a live probe prints what the server returned, and Layer 2 scans *every* modality for query failures.
- **GWAS 410 fix (v4).** `layer1.gwas_source: auto` tries the REST API first and, if it fails, reads the official GWAS Catalog bulk release file (EBI FTP) instead; the source used is recorded in the Layer 1 manifest. Set `api` or `download` to force one.
- **Data-quality gate.** If any evidence query failed, Layer 2 stops before writing outputs (override: `layer2.halt_on_query_failure: false`, outputs are then marked `DEGRADED`).
- **Context-scope audit** flags MPRA/CRISPR/eQTL evidence that the target cell type/tissue would silently exclude.
- **Tier 1 feasibility report** lists the variants that one extra allele-resolved source would promote to Tier 1.

## Clean-run order
Run every cell top to bottom. The setup cell asks for the real Layer 0 and MPRA CSVs if they are not found (not needed for Layer 0 when `run_layer0` is true). Layer 1 live collection needs internet. After changing code or config, rerun from the top.


## 1. Environment, inputs, and project paths

Expected paths after setup:
- `layer0_outputs/HBB_layer0_regulatory_truthset.csv` (+ `HBB_layer0_manifest.json`, written automatically)
- `data/inputs/hbb_locus_mpra_evidence.csv`
- `configs/hbb.yaml` (created with safe defaults on first run)

The supplied Layer 0 and MPRA tables are real project inputs. No replacement variants or evidence are generated.

**Table schemas (allele-specific; `cell_type` must equal `layer2.target_cell_types`, currently HEL92.1.7):**
- MPRA: `chrom, pos, ref, alt, cell_type, log2_allelic_skew, pvalue, dataset_name`
- CRISPR (optional, drop at `data/inputs/hbb_locus_crispr_evidence.csv` or set `layer1.crispr_evidence_path`): `chrom, pos, ref, alt, target_gene, cell_type, effect_size, pvalue, screen_type, dataset_name`


## 1b. Project configuration (auto-created) and Layer 0 mode

Creates `configs/hbb.yaml` if it does not exist and decides whether Layer 0 is regenerated or the supplied CSV is used.


In [3]:
# Project configuration: created on the first run, never overwritten afterwards (edit it freely).
import os, json, sys, subprocess
from pathlib import Path

CONFIG_PATH = "configs/hbb.yaml"

DEFAULT_CONFIG_YAML = '''# HBB regulatory-evidence pipeline configuration.
# Auto-created by the notebook on first run. Edit freely: the notebook never overwrites this file.
gene_symbol: HBB
flank_bp: 100000

layer0:
  run_layer0: false                   # true = rebuild the candidate set from ClinVar + Ensembl VEP + ENCODE cCREs (needs internet, apt, pip)
  canonical_transcript: ENST00000335295
  promoter_window_bp: 2500
  clinvar_snapshot_date: "20261019"   # pinned for reproducibility; null = auto-discover the newest snapshot (then paste the printed date here)
  clinvar_snapshot_note: "date reported by the original Layer 0 run; not verifiable from the CSV itself"
  vep_batch_size: 200
  vep_timeout_s: 30
  vep_max_retries: 3

layer1:
  gtex_tissues: [Whole_Blood]
  gwas_window_bp: 25000
  gwas_source: auto                   # auto = REST API, then the official bulk release file if the API fails | api | download
  mpra_evidence_path: data/inputs/hbb_locus_mpra_evidence.csv
  crispr_evidence_path: null          # a real CRISPR table (schema in Section 3), or drop it at data/inputs/hbb_locus_crispr_evidence.csv

layer2:
  target_tissues: [Whole_Blood]
  target_cell_types: [HEL92.1.7]      # erythroid; must match the cell_type values in your MPRA/CRISPR tables
  halt_on_query_failure: true         # stop before writing Layer 2 outputs if any evidence query failed (failures are unknowns, not absence)
  tier_thresholds:
    tier_1_min_modalities: 2
    tier_2_modalities: 1
    tier_3_modalities: 0
'''


def ensure_project_config(path=CONFIG_PATH):
    """Write the default config if (and only if) it does not exist. Returns True when created."""
    p = Path(path)
    if p.exists():
        return False
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(DEFAULT_CONFIG_YAML, encoding="utf-8")
    return True


try:
    import yaml
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)
    import yaml

_created = ensure_project_config()
print(("Created default " if _created else "Using existing ") + CONFIG_PATH)
with open(CONFIG_PATH, encoding="utf-8") as _f:
    _project_cfg = yaml.safe_load(_f) or {}

RUN_LAYER0_OVERRIDE = None   # set True/False here to override layer0.run_layer0 from the config file
RUN_LAYER0 = bool(RUN_LAYER0_OVERRIDE if RUN_LAYER0_OVERRIDE is not None
                  else (_project_cfg.get("layer0") or {}).get("run_layer0", False))
print("Layer 0 mode:", "REGENERATE from ClinVar / Ensembl VEP / ENCODE cCREs" if RUN_LAYER0
      else "use the supplied Layer 0 CSV (set layer0.run_layer0: true to regenerate)")


Using existing configs/hbb.yaml
Layer 0 mode: use the supplied Layer 0 CSV (set layer0.run_layer0: true to regenerate)


In [4]:
from pathlib import Path
import os, shutil, sys, hashlib

ROOT = Path.cwd()
LAYER0_DEST = ROOT / "layer0_outputs" / "HBB_layer0_regulatory_truthset.csv"
MPRA_DEST = ROOT / "data" / "inputs" / "hbb_locus_mpra_evidence.csv"
LAYER0_DEST.parent.mkdir(parents=True, exist_ok=True)
MPRA_DEST.parent.mkdir(parents=True, exist_ok=True)

def _sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def _first_existing(candidates):
    for candidate in candidates:
        p = Path(candidate)
        if p.exists() and p.is_file():
            return p
    return None

# Prefer the explicitly uploaded "(1)" file when present, then a top-level upload,
# then the existing project copy. Print hashes so a stale duplicate is visible.
l0_src = _first_existing([
    "/content/HBB_layer0_regulatory_truthset (1).csv",
    "/mnt/data/HBB_layer0_regulatory_truthset (1).csv",
    "/content/HBB_layer0_regulatory_truthset.csv",
    "/mnt/data/HBB_layer0_regulatory_truthset.csv",
    "HBB_layer0_regulatory_truthset (1).csv",
    "HBB_layer0_regulatory_truthset.csv",
    "layer0_outputs/HBB_layer0_regulatory_truthset.csv",
    "/content/layer0_outputs/HBB_layer0_regulatory_truthset.csv",
])
mpra_src = _first_existing([
    "/content/hbb_locus_mpra_evidence(1).csv",
    "/mnt/data/hbb_locus_mpra_evidence(1).csv",
    "/content/hbb_locus_mpra_evidence.csv",
    "/mnt/data/hbb_locus_mpra_evidence.csv",
    "hbb_locus_mpra_evidence(1).csv",
    "hbb_locus_mpra_evidence.csv",
    "data/inputs/hbb_locus_mpra_evidence.csv",
])

# Show all candidate Layer 0 copies and their hashes before selecting the source.
l0_candidates = [
    Path(p) for p in [
        "/content/HBB_layer0_regulatory_truthset (1).csv",
        "/mnt/data/HBB_layer0_regulatory_truthset (1).csv",
        "/content/HBB_layer0_regulatory_truthset.csv",
        "/mnt/data/HBB_layer0_regulatory_truthset.csv",
        "HBB_layer0_regulatory_truthset (1).csv",
        "HBB_layer0_regulatory_truthset.csv",
        "layer0_outputs/HBB_layer0_regulatory_truthset.csv",
        "/content/layer0_outputs/HBB_layer0_regulatory_truthset.csv",
    ] if Path(p).exists() and Path(p).is_file()
]
unique_hashes = {}
if l0_candidates:
    print("Layer 0 source hash audit:")
    for candidate in l0_candidates:
        digest = _sha256_file(candidate)
        unique_hashes.setdefault(digest, []).append(str(candidate))
        print(f"  {digest}  {candidate}")
    if len(unique_hashes) > 1:
        print("WARNING: Layer 0 copies differ. The notebook will prefer the newest explicitly uploaded '(1)' file if present.")
    else:
        print("PASS: all discovered Layer 0 copies are byte-identical.")

if l0_src and l0_src.resolve() != LAYER0_DEST.resolve():
    shutil.copy2(l0_src, LAYER0_DEST)
if mpra_src and mpra_src.resolve() != MPRA_DEST.resolve():
    shutil.copy2(mpra_src, MPRA_DEST)

missing = []
if not LAYER0_DEST.exists() and not RUN_LAYER0: missing.append("HBB_layer0_regulatory_truthset.csv")
if not MPRA_DEST.exists(): missing.append("hbb_locus_mpra_evidence.csv")
if missing:
    try:
        from google.colab import files
        print("Upload the following project file(s):", ", ".join(missing))
        uploaded = files.upload()
        for name in uploaded:
            src = Path(name)
            if "HBB_layer0_regulatory_truthset" in name:
                shutil.copy2(src, LAYER0_DEST)
            elif "hbb_locus_mpra_evidence" in name:
                shutil.copy2(src, MPRA_DEST)
    except ImportError:
        pass

print("Layer 0 input:", "FOUND" if LAYER0_DEST.exists() else "MISSING", LAYER0_DEST)
if LAYER0_DEST.exists():
    print("Layer 0 destination SHA-256:", _sha256_file(LAYER0_DEST))
print("MPRA input:", "FOUND" if MPRA_DEST.exists() else "MISSING", MPRA_DEST)
if not LAYER0_DEST.exists() and not RUN_LAYER0:
    raise FileNotFoundError(f"Required real Layer 0 CSV missing: {LAYER0_DEST}. Upload the supplied project CSV; do not substitute synthetic data.")
if not MPRA_DEST.exists():
    print("WARNING: MPRA CSV missing. Layer 1 will mark MPRA as not configured, not as tested-null.")


Upload the following project file(s): HBB_layer0_regulatory_truthset.csv, hbb_locus_mpra_evidence.csv


Saving HBB_layer0_regulatory_truthset.csv to HBB_layer0_regulatory_truthset.csv
Saving hbb_locus_mpra_evidence.csv to hbb_locus_mpra_evidence.csv
Layer 0 input: FOUND /content/layer0_outputs/HBB_layer0_regulatory_truthset.csv
Layer 0 destination SHA-256: 100797b568d1168f459bb77022414201c8cdfbe80a26b88916141a2da8fd8513
MPRA input: FOUND /content/data/inputs/hbb_locus_mpra_evidence.csv


## 2. Imports and shared utilities


In [5]:
import os
import json
import time
import hashlib
import subprocess
from collections import defaultdict, namedtuple
from dataclasses import dataclass, asdict
from datetime import datetime, timezone, timedelta
from typing import Optional, Callable

import requests
import pandas as pd

## 3. Configuration and reproducible HTTP/cache layer


In [6]:
CONFIG_PATH = "configs/hbb.yaml"

DEFAULT_L1 = {
    "layer0_csv_path": None,            # optional explicit path to Layer 0's candidate CSV
    "strict": False,                    # if True, raises FileNotFoundError if Layer 0 CSV is missing
    "use_http_cache": True,             # caches network responses in data/cache/http to prevent API rate limits
    "http_cache_dir": "data/cache/http",
    "cache_ttl_days": 7,                # cache entries older than 7 days expire automatically
    "refresh_cache": False,             # set True to bypass cache and re-query live endpoints
    "gtex_dataset": "gtex_v8",
    "gtex_gencode_version": "v26",      # must match the dataset (gtex_v8 uses GENCODE v26)
    "gtex_tissues": ["Whole_Blood"],
    "gtex_items_per_page": 250,
    "gtex_max_pages": 40,
    "eqtl_testable_min_maf": 0.01,      # GTEx only tests variants with MAF >= 1% (gnomAD AF is a proxy)
    "gwas_p_threshold": 5.0e-8,
    "gwas_source": "auto",              # auto = REST API first, official bulk file if the API fails; or force: api | download
    "gwas_download_url": "https://ftp.ebi.ac.uk/pub/databases/gwas/releases/latest/gwas-catalog-associations_ontology-annotated-full.zip",
    "gwas_download_ttl_days": 14,       # reuse a downloaded bulk file this long
    "gwas_window_bp": 25000,            # cluster candidate positions to avoid querying massive locus spans
    "gnomad_dataset": "gnomad_r4",
    "gnomad_window_bp": 25000,
    "gnomad_request_delay_s": 6.0,      # public API is rate-limited (~10 req/min); only sleeps on real network calls
    "mpra_evidence_path": "data/inputs/hbb_locus_mpra_evidence.csv",
    "crispr_evidence_path": None,
    "sig_pvalue_threshold": 0.05,       # nominal; supply FDR-adjusted p-values in your tables for real work
}

FALLBACK_CONFIG = {"gene_symbol": "HBB", "flank_bp": 100000, "layer1": {}}

if os.path.exists(CONFIG_PATH):
    import yaml
    with open(CONFIG_PATH) as f:
        CONFIG = yaml.safe_load(f)
else:
    CONFIG = FALLBACK_CONFIG
    print("No config file found -- using fallback CONFIG.")

GENE_SYMBOL = CONFIG["gene_symbol"]
FLANK_BP = CONFIG.get("flank_bp", 100000)
_user_l1 = CONFIG.get("layer1") or {}
L1 = {**DEFAULT_L1, **_user_l1}
_unknown = sorted(set(_user_l1) - set(DEFAULT_L1))
if _unknown:
    print(f"NOTE: ignoring unrecognised layer1 config keys: {_unknown}")
_crispr_default = "data/inputs/hbb_locus_crispr_evidence.csv"
if not L1.get("crispr_evidence_path") and os.path.exists(_crispr_default):
    L1["crispr_evidence_path"] = _crispr_default
    print(f"NOTE: found a CRISPR evidence table at {_crispr_default}; CRISPR is enabled.")
print(json.dumps(L1, indent=2))

{
  "layer0_csv_path": null,
  "strict": false,
  "use_http_cache": true,
  "http_cache_dir": "data/cache/http",
  "cache_ttl_days": 7,
  "refresh_cache": false,
  "gtex_dataset": "gtex_v8",
  "gtex_gencode_version": "v26",
  "gtex_tissues": [
    "Whole_Blood"
  ],
  "gtex_items_per_page": 250,
  "gtex_max_pages": 40,
  "eqtl_testable_min_maf": 0.01,
  "gwas_p_threshold": 5e-08,
  "gwas_source": "auto",
  "gwas_download_url": "https://ftp.ebi.ac.uk/pub/databases/gwas/releases/latest/gwas-catalog-associations_ontology-annotated-full.zip",
  "gwas_download_ttl_days": 14,
  "gwas_window_bp": 25000,
  "gnomad_dataset": "gnomad_r4",
  "gnomad_window_bp": 25000,
  "gnomad_request_delay_s": 6.0,
  "mpra_evidence_path": "data/inputs/hbb_locus_mpra_evidence.csv",
  "crispr_evidence_path": null,
  "sig_pvalue_threshold": 0.05
}


In [7]:
class HTTPCache:
    def __init__(self, cache_dir: str, enabled: bool = True, ttl_days: int = 7, refresh: bool = False):
        self.cache_dir = cache_dir
        self.enabled = enabled
        self.ttl_days = ttl_days
        self.refresh = refresh
        self.hits = 0
        self.misses = 0
        self.oldest_entry_utc: Optional[str] = None
        if self.enabled and self.cache_dir:
            os.makedirs(self.cache_dir, exist_ok=True)

    def _cache_key(self, method: str, url: str, params, json_body) -> str:
        payload = {
            "m": method.upper(),
            "u": url,
            "p": sorted(params.items()) if params else None,
            "b": json_body
        }
        raw = json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
        return hashlib.sha256(raw).hexdigest()

    def get(self, method: str, url: str, params, json_body):
        if not self.enabled or self.refresh or not self.cache_dir:
            return None, False
        ckey = self._cache_key(method, url, params, json_body)
        cpath = os.path.join(self.cache_dir, f"{ckey}.json")
        if not os.path.exists(cpath):
            return None, False
        try:
            with open(cpath, "r", encoding="utf-8") as f:
                entry = json.load(f)
            fetched_at_str = entry.get("fetched_at")
            if fetched_at_str:
                fetched_at = datetime.fromisoformat(fetched_at_str)
                now = datetime.now(timezone.utc)
                if now - fetched_at > timedelta(days=self.ttl_days):
                    return None, False  # expired TTL
                if self.oldest_entry_utc is None or fetched_at_str < self.oldest_entry_utc:
                    self.oldest_entry_utc = fetched_at_str
            self.hits += 1
            return entry, True
        except Exception:
            return None, False

    def put(self, method: str, url: str, params, json_body, data, error):
        if not self.enabled or not self.cache_dir:
            return
        # Do not cache transient failures, only clean data or persistent 404
        if error is not None and error != "http_404":
            return
        # Reject GraphQL responses with errors (cache poisoning prevention)
        if isinstance(data, dict) and data.get("errors"):
            return
        ckey = self._cache_key(method, url, params, json_body)
        cpath = os.path.join(self.cache_dir, f"{ckey}.json")
        now_str = datetime.now(timezone.utc).isoformat()
        try:
            with open(cpath, "w", encoding="utf-8") as f:
                json.dump({"data": data, "error": error, "fetched_at": now_str}, f)
            if self.oldest_entry_utc is None or now_str < self.oldest_entry_utc:
                self.oldest_entry_utc = now_str
        except Exception:
            pass


def make_http_fetch(cache: HTTPCache, timeout=(5, 20), max_retries: int = 3, backoff: float = 1.5):
    """Cached GET/POST client with one persistent session, separate connect/read timeouts,
    bounded exponential backoff, and Retry-After support. Transient failures are never cached."""
    session = requests.Session()
    session.headers.update({"User-Agent": "HBB-regulatory-evidence-pipeline/1.0"})
    retryable_status = {429, 500, 502, 503, 504}

    def fetch(method: str, url: str, params=None, json_body=None):
        cached_entry, hit = cache.get(method, url, params, json_body)
        if hit:
            return cached_entry.get("data"), cached_entry.get("error"), True
        cache.misses += 1
        last_err = "unknown"
        for attempt in range(max_retries + 1):
            retry_after = None
            try:
                r = session.request(method, url, params=params, json=json_body, timeout=timeout)
            except requests.exceptions.RequestException as e:
                last_err = f"{type(e).__name__}: {e}"
            else:
                if r.status_code == 200:
                    try:
                        data = r.json()
                    except ValueError:
                        return None, "invalid_json", False
                    cache.put(method, url, params, json_body, data, None)
                    return data, None, False
                if r.status_code == 404:
                    cache.put(method, url, params, json_body, None, "http_404")
                    return None, "http_404", False
                # 410 Gone is normally permanent, but the GWAS Catalog has returned it transiently from a live
                # endpoint, so retry it a bounded number of times before reporting it as a failure.
                if r.status_code in retryable_status or (r.status_code == 410 and attempt < 2):
                    last_err = f"http_{r.status_code}"
                    retry_after = r.headers.get("Retry-After")
                else:
                    return None, f"http_{r.status_code}", False
            if attempt < max_retries:
                try:
                    delay = float(retry_after) if retry_after is not None else backoff ** attempt
                except (TypeError, ValueError):
                    delay = backoff ** attempt
                time.sleep(min(max(delay, 0.0), 30.0))
        return None, last_err, False
    fetch.session = session
    return fetch


def _file_hash(path, algo="sha256"):
    if not path or not os.path.exists(path):
        return None
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def _git_commit_hash():
    try:
        r = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True, timeout=5)
        return r.stdout.strip()
    except Exception:
        return None


def write_manifest(out_dir, filename, payload):
    manifest = {"generated_at_utc": datetime.now(timezone.utc).isoformat(), "git_commit": _git_commit_hash()}
    manifest.update(payload)
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, default=str)
    return path

## 3b. Layer 0 candidate generation (optional; runs only when `layer0.run_layer0` is true)

Restored from the original Layer 0 workflow as functions, so nothing runs (and no tools are installed) unless regeneration is switched on. Pin `layer0.clinvar_snapshot_date` for reproducibility.


In [8]:
import re, shutil, sys

REGULATORY_UTR_TERMS = [
    "5_prime_UTR_variant",
    "3_prime_UTR_variant",
    "regulatory_region_variant",
    "TF_binding_site_variant",
]

REVIEW_STARS = {
    "practice_guideline": 4,
    "reviewed_by_expert_panel": 3,
    "criteria_provided,_multiple_submitters,_no_conflicts": 2,
    "criteria_provided,_single_submitter": 1,
    "criteria_provided,_conflicting_interpretations": 1,
    "no_assertion_criteria_provided": 0,
    "no_assertion_provided": 0,
}


def _run(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        msg = f"Command failed (exit {r.returncode}): {cmd}\nSTDERR:\n{r.stderr}"
        if check:
            raise RuntimeError(msg)
        print(msg)
    return r


def to_ucsc_chrom(chrom):
    """Ensembl/ClinVar convention ('11') -> UCSC convention ('chr11')."""
    chrom = str(chrom)
    return chrom if chrom.startswith("chr") else f"chr{chrom}"


def to_plain_chrom(chrom):
    """UCSC convention ('chr11') -> Ensembl/ClinVar convention ('11')."""
    chrom = str(chrom)
    return chrom[3:] if chrom.startswith("chr") else chrom


def ensure_layer0_tools():
    """Install the CLI/python dependencies only Layer 0 generation needs (Colab / Ubuntu)."""
    missing = [t for t in ("tabix", "bcftools", "samtools") if shutil.which(t) is None]
    if missing:
        _run("apt-get install -y tabix bcftools samtools -q")
    try:
        import pysam, pyranges  # noqa: F401
    except ImportError:
        _run(f"{sys.executable} -m pip install -q pysam pyranges")


def l0_fetch_gene_metadata(gene_symbol, species="homo_sapiens", timeout=15):
    url = f"https://rest.ensembl.org/lookup/symbol/{species}/{gene_symbol}"
    r = requests.get(url, headers={"Content-Type": "application/json"}, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    return {
        "gene_symbol": gene_symbol,
        "gene_id": data["id"],
        "chrom": str(data["seq_region_name"]),
        "start": int(data["start"]),
        "end": int(data["end"]),
        "strand": int(data["strand"]),            # 1 (+) or -1 (-)
        "assembly_name": data.get("assembly_name", "GRCh38"),
        "metadata_source": "Ensembl REST lookup",
    }


def l0_regions(gene_meta, promoter_window_bp, flank_bp):
    if gene_meta["strand"] == 1:
        promoter_start, promoter_end = gene_meta["start"] - promoter_window_bp, gene_meta["start"]
    else:
        promoter_start, promoter_end = gene_meta["end"], gene_meta["end"] + promoter_window_bp
    flank_start = max(1, gene_meta["start"] - flank_bp)
    flank_end = gene_meta["end"] + flank_bp
    return {"promoter_start": promoter_start, "promoter_end": promoter_end,
            "flank_start": flank_start, "flank_end": flank_end,
            "region_str": f"{gene_meta['chrom']}:{flank_start}-{flank_end}"}


def download_reference(chrom, out_prefix="ref"):
    fa_gz = f"{out_prefix}_chr{chrom}.fa.gz"
    fa = f"{out_prefix}_chr{chrom}.fa"
    url = f"https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr{chrom}.fa.gz"
    _run(f"wget -nc {url} -O {fa_gz}")
    if _run(f"gzip -t {fa_gz}", check=False).returncode != 0:
        print(f"Corrupt download detected, re-fetching {fa_gz}")
        _run(f"rm -f {fa_gz}")
        _run(f"wget {url} -O {fa_gz}")
        _run(f"gzip -t {fa_gz}")
    _run(f"gunzip -kf {fa_gz}")
    _run(f"sed -i 's/^>chr{chrom}/>{chrom}/' {fa}")
    _run(f"samtools faidx {fa}")
    if not os.path.exists(f"{fa}.fai"):
        raise RuntimeError(f"samtools faidx did not produce an index for {fa}")
    return fa


def discover_latest_clinvar_date(timeout=15):
    r = requests.get("https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/", timeout=timeout)
    dates = sorted(set(re.findall(r'clinvar_(\d{8})\.vcf\.gz"', r.text)), reverse=True)
    if not dates:
        raise RuntimeError("Could not find any dated ClinVar snapshot in the directory listing")
    return dates[0]


def clinvar_archive_url(date_str):
    return f"https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar_{date_str}.vcf.gz"


def download_clinvar_snapshot(date_str, out_name="clinvar_snapshot.vcf.gz"):
    url = clinvar_archive_url(date_str)
    _run(f"rm -f {out_name}")
    _run(f"wget {url} -O {out_name}")
    _run(f"gzip -t {out_name}")
    _run(f"tabix -f -p vcf {out_name}")
    return out_name


def extract_region(vcf_gz, region_str, out_name):
    _run(f"bcftools view {vcf_gz} -r {region_str} -Oz -o {out_name}")
    _run(f"tabix -f -p vcf {out_name}")
    return out_name


def normalize_vcf(input_vcf_gz, ref_fasta, out_name):
    split_name = "split_tmp.vcf.gz"
    _run(f"bcftools norm -f {ref_fasta} -m -both {input_vcf_gz} -Oz -o {split_name}")
    _run(f"tabix -f -p vcf {split_name}")
    _run(f"bcftools norm -f {ref_fasta} -d exact {split_name} -Oz -o {out_name}")
    _run(f"tabix -f -p vcf {out_name}")
    return out_name


def extract_clinvar_labels(vcf_path):
    import pysam
    vf = pysam.VariantFile(vcf_path)
    rows = []
    for rec in vf:
        clnsig = rec.info.get("CLNSIG", ("",))
        clnsig = ",".join(clnsig) if isinstance(clnsig, tuple) else str(clnsig)
        if "Conflicting" in clnsig:
            continue
        if "Pathogenic" in clnsig:
            label = "pathogenic"
        elif "Benign" in clnsig:
            label = "benign"
        else:
            continue
        revstat = rec.info.get("CLNREVSTAT", ("",))
        revstat = ",".join(revstat) if isinstance(revstat, tuple) else str(revstat)
        stars = max([v for k, v in REVIEW_STARS.items() if k in revstat], default=0)
        for alt in (rec.alts or []):
            rows.append({"chrom": rec.chrom, "pos": rec.pos, "ref": rec.ref, "alt": alt,
                         "CLNSIG": clnsig, "CLNREVSTAT": revstat, "review_stars": stars, "label": label})
    return pd.DataFrame(rows).drop_duplicates(subset=["chrom", "pos", "ref", "alt"]).reset_index(drop=True)


def fetch_bigbedtobed_tool():
    if not os.path.exists("bigBedToBed"):
        _run("wget -q http://hgdownload.soe.ucsc.edu/admin/exe/linux.x86_64/bigBedToBed")
        _run("chmod +x bigBedToBed")
    return "./bigBedToBed"


def fetch_ccres(chrom, start, end, out_bed="ccre_region.bed"):
    tool = fetch_bigbedtobed_tool()
    remote_bb = "https://hgdownload.soe.ucsc.edu/gbdb/hg38/encode3/ccre/encodeCcreCombined.bb"
    cmd = f"{tool} {remote_bb} -chrom={to_ucsc_chrom(chrom)} -start={start} -end={end} {out_bed}"
    r = _run(cmd, check=False)
    if r.returncode != 0 or not os.path.exists(out_bed) or os.path.getsize(out_bed) == 0:
        print("WARNING: cCRE fetch failed or returned empty for this region")
        return pd.DataFrame(columns=["Chromosome", "Start", "End", "ccre_label"])
    cols = ["Chromosome", "Start", "End", "name", "score", "strand", "thickStart", "thickEnd",
            "reserved", "ccre_group", "ccre_group2", "zscore", "ucscLabel", "accession", "ccre_label"]
    df = pd.read_csv(out_bed, sep="\t", header=None, names=cols)
    df["Chromosome"] = to_plain_chrom(chrom)
    return df[["Chromosome", "Start", "End", "ccre_label"]]


def flag_interval_overlap(variants_df, reg_intervals_df):
    import pyranges as pr
    variants_df = variants_df.reset_index(drop=True).copy()
    variants_df["_row_id"] = variants_df.index
    if reg_intervals_df.empty:
        variants_df["interval_evidence"] = [[] for _ in range(len(variants_df))]
        return variants_df.drop(columns=["_row_id"])
    var_pr = pr.PyRanges(pd.DataFrame({
        "Chromosome": variants_df["chrom"],
        "Start": variants_df["pos"] - 1,   # pyranges is 0-based half-open
        "End": variants_df["pos"],
        "row_id": variants_df["_row_id"],
    }))
    reg_pr = pr.PyRanges(reg_intervals_df)
    joined = var_pr.join(reg_pr).df
    if joined.empty or "row_id" not in joined.columns:
        variants_df["interval_evidence"] = [[] for _ in range(len(variants_df))]
        return variants_df.drop(columns=["_row_id"])
    evidence_map = joined.groupby("row_id")["ccre_label"].apply(list).to_dict()
    variants_df["interval_evidence"] = variants_df["_row_id"].map(lambda i: evidence_map.get(i, []))
    return variants_df.drop(columns=["_row_id"])


def vep_annotate_batch(variants_df, batch_size=200, timeout=30, max_retries=3):
    results = []
    variants_list = variants_df[["chrom", "pos", "ref", "alt"]].to_dict("records")
    n_requested = len(variants_list)
    for i in range(0, n_requested, batch_size):
        batch = variants_list[i:i + batch_size]
        variant_strings = [f"{v['chrom']} {v['pos']} . {v['ref']} {v['alt']} . . ." for v in batch]
        attempt = 0
        while attempt <= max_retries:
            try:
                r = requests.post(
                    "https://rest.ensembl.org/vep/human/region",
                    headers={"Content-Type": "application/json", "Accept": "application/json"},
                    json={"variants": variant_strings, "regulatory": 1},
                    timeout=timeout,
                )
                if r.status_code == 200:
                    results.extend(r.json())
                    break
                elif r.status_code in (429, 503):
                    wait = 2 ** attempt
                    print(f"Batch {i}: rate-limited ({r.status_code}), retrying in {wait}s")
                    time.sleep(wait)
                    attempt += 1
                else:
                    print(f"Batch {i} failed permanently: {r.status_code}, {r.text[:200]}")
                    break
            except requests.exceptions.RequestException as e:
                wait = 2 ** attempt
                print(f"Batch {i}: {type(e).__name__}, retrying in {wait}s")
                time.sleep(wait)
                attempt += 1
        time.sleep(1)
    if len(results) < n_requested:
        print(f"WARNING: requested {n_requested} variants, got {len(results)} VEP results "
              f"({n_requested - len(results)} lost to failed/skipped batches)")
    return results


def parse_vep_results(vep_results, canonical_transcript, target_gene_id):
    rows, n_missing_allele = [], 0
    for res in vep_results:
        pos, chrom, allele = res.get("start"), res.get("seq_region_name"), res.get("allele_string")
        if not allele or "/" not in allele:
            n_missing_allele += 1
            continue
        ref, alt = allele.split("/")[:2]
        gene_restricted_terms, canonical_terms = set(), []   # only transcripts of the target gene
        for tc in res.get("transcript_consequences", []):
            if tc.get("gene_id") == target_gene_id:
                gene_restricted_terms.update(tc.get("consequence_terms", []))
                if tc.get("transcript_id") == canonical_transcript:
                    canonical_terms = tc.get("consequence_terms", [])
        reg_terms = set()   # regulatory features are not gene-specific -- kept unrestricted
        for rfc in res.get("regulatory_feature_consequences", []):
            reg_terms.update(rfc.get("consequence_terms", []))
        rows.append({"chrom": chrom, "pos": pos, "ref": ref, "alt": alt,
                     "vep_canonical_terms": canonical_terms,
                     "vep_all_transcript_terms": sorted(gene_restricted_terms),
                     "vep_regulatory_terms": sorted(reg_terms)})
    if n_missing_allele:
        print(f"NOTE: {n_missing_allele} VEP results had no parseable allele_string and were dropped")
    return pd.DataFrame(rows, columns=["chrom", "pos", "ref", "alt", "vep_canonical_terms",
                                       "vep_all_transcript_terms", "vep_regulatory_terms"])


def build_regulatory_table(variants_df, vep_df):
    """Merge ClinVar labels + interval evidence + VEP terms; keep variants with regulatory evidence."""
    merged = variants_df.merge(vep_df, on=["chrom", "pos", "ref", "alt"], how="left")

    def _terms(row):
        t = row["vep_all_transcript_terms"] if isinstance(row["vep_all_transcript_terms"], list) else []
        r = row["vep_regulatory_terms"] if isinstance(row["vep_regulatory_terms"], list) else []
        return [x for x in (t + r) if any(k in x for k in REGULATORY_UTR_TERMS)]

    def _evidence(row):
        ev_list = [f"interval:{label}" for label in row["interval_evidence"]]
        return ev_list + [f"vep:{term}" for term in _terms(row)]

    merged["regulatory_evidence"] = merged.apply(_evidence, axis=1)
    merged["is_regulatory"] = merged["regulatory_evidence"].str.len() > 0
    final = (merged[merged["is_regulatory"]]
             .drop_duplicates(subset=["chrom", "pos", "ref", "alt"]).reset_index(drop=True))
    return merged, final


def layer0_sanity_checks(final_df, merged_df, gene_meta):
    lcr_start, lcr_end = 5_269_925, 5_304_186   # HBB LCR, GRCh38, informational reference range
    in_lcr = final_df[(final_df["chrom"] == gene_meta["chrom"]) &
                      (final_df["pos"] >= lcr_start) & (final_df["pos"] <= lcr_end)]
    if in_lcr.empty:
        print("INFO: no labeled ClinVar variants fall inside the LCR window for this run (not an error).")
    else:
        assert in_lcr["is_regulatory"].all(), "LCR-region variant failed to be flagged regulatory"
        assert any("interval:" in e for row in in_lcr["regulatory_evidence"] for e in row), \
            "LCR variant flagged regulatory but not via interval evidence -- check cCRE fetch"
        print(f"PASS: {len(in_lcr)} LCR-region variant(s) retained via interval-first classification")

    def _s(val):
        return set(val) if isinstance(val, list) else set()
    disagree = merged_df[merged_df.apply(
        lambda r: _s(r.get("vep_canonical_terms")) != _s(r.get("vep_all_transcript_terms")), axis=1)]
    if disagree.empty:
        print("INFO: no canonical/union VEP disagreements found in this run.")
    else:
        n_reg = int(disagree["is_regulatory"].sum())
        print(f"{len(disagree)} canonical/union disagreement variant(s); {n_reg} flagged regulatory, "
              f"{len(disagree) - n_reg} not (visible here, not silently resolved by canonical-only logic).")


def run_layer0(cfg, gene_symbol, flank_bp, out_dir="layer0_outputs"):
    """Regenerate the Layer 0 candidate truth set. Returns (final_df, gene_meta, info)."""
    ensure_layer0_tools()
    promoter_bp = int(cfg.get("promoter_window_bp", 2500))
    canonical = cfg.get("canonical_transcript", "ENST00000335295")
    gene_meta = l0_fetch_gene_metadata(gene_symbol)
    reg = l0_regions(gene_meta, promoter_bp, flank_bp)
    print(gene_meta)
    print(f"Promoter window: {reg['promoter_start']}-{reg['promoter_end']} | search region: {reg['region_str']}")

    ref_fasta = download_reference(gene_meta["chrom"])
    clinvar_date = cfg.get("clinvar_snapshot_date")
    if not clinvar_date:
        clinvar_date = discover_latest_clinvar_date()
        print(f"No pinned ClinVar date -- auto-discovered {clinvar_date}. "
              f"Paste it into layer0.clinvar_snapshot_date in configs/hbb.yaml for reproducible re-runs.")
    else:
        print(f"Using pinned ClinVar snapshot: {clinvar_date}")
    clinvar_full = download_clinvar_snapshot(clinvar_date)
    clinvar_region = extract_region(clinvar_full, reg["region_str"], "clinvar_region.vcf.gz")
    clinvar_norm = normalize_vcf(clinvar_region, ref_fasta, "clinvar_normalized.vcf.gz")

    labeled = extract_clinvar_labels(clinvar_norm)
    n_labeled = len(labeled)
    labeled = labeled[labeled["review_stars"] >= 1].reset_index(drop=True)   # drop unreviewed (0-star)
    print(f"Labeled ClinVar variants: {n_labeled} -> {len(labeled)} after review_stars >= 1")

    ccre_df = fetch_ccres(gene_meta["chrom"], reg["flank_start"], reg["flank_end"])
    promoter_df = pd.DataFrame({"Chromosome": [gene_meta["chrom"]], "Start": [reg["promoter_start"]],
                                "End": [reg["promoter_end"]], "ccre_label": ["computed_promoter_window"]})
    reg_intervals = pd.concat([ccre_df, promoter_df], ignore_index=True)
    labeled = flag_interval_overlap(labeled, reg_intervals)

    vep_results = vep_annotate_batch(labeled, int(cfg.get("vep_batch_size", 200)),
                                     int(cfg.get("vep_timeout_s", 30)), int(cfg.get("vep_max_retries", 3)))
    vep_df = parse_vep_results(vep_results, canonical, gene_meta["gene_id"])
    merged, final = build_regulatory_table(labeled, vep_df)
    print(f"Regulatory candidates: {len(final)} / {len(merged)} labeled variants")
    layer0_sanity_checks(final, merged, gene_meta)
    info = {"clinvar_snapshot_date": clinvar_date, "region": reg["region_str"],
            "n_labeled_variants": n_labeled, "n_after_review_filter": int(len(labeled)),
            "n_vep_results": len(vep_results), "n_regulatory_candidates": int(len(final)),
            "canonical_transcript": canonical, "promoter_window_bp": promoter_bp}
    return final, gene_meta, info


## 4. Evidence schemas, variant normalization, and Layer 0 intake


In [9]:
SOURCE_TYPES = ("experimental", "computational", "curated_clinical")
MISSINGNESS_FLAGS = ("observed_positive", "observed_null", "absent")
MODALITIES = ("gwas", "eqtl", "gnomad_af", "mpra", "crispr")

MODALITY_META = {
    "gwas":      {"kind": "association",          "completable": False, "use_in_fusion": True,
                  "direction": "supports_regulatory_relevance"},
    "eqtl":      {"kind": "expression_effect",    "completable": True,  "use_in_fusion": True,
                  "direction": "supports_regulatory_relevance"},
    "mpra":      {"kind": "reporter_activity",    "completable": True,  "use_in_fusion": True,
                  "direction": "supports_regulatory_relevance"},
    "crispr":    {"kind": "perturbation_effect",  "completable": True,  "use_in_fusion": True,
                  "direction": "supports_regulatory_relevance"},
    "gnomad_af": {"kind": "population_frequency", "completable": False, "use_in_fusion": False,
                  "direction": "benign_leaning_when_observed"},
}


def _clean(o):
    """Make values JSON-safe: NaN/inf -> None, numpy scalars -> python, recurse into containers."""
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [_clean(v) for v in o]
    if hasattr(o, "item") and not isinstance(o, (str, bytes)):
        try:
            o = o.item()
        except Exception:
            pass
    if isinstance(o, float) and (o != o or o in (float("inf"), float("-inf"))):
        return None
    return o


def ev(**kw):
    return json.dumps(_clean(kw), sort_keys=True, allow_nan=False)


@dataclass
class EvidenceRecord:
    chrom: str
    pos: int
    ref: str
    alt: str
    modality: str
    source_type: str
    evidence_value: str
    context: Optional[str] = None
    missingness_flag: str = "observed_positive"

    def __post_init__(self):
        if self.modality not in MODALITIES:
            raise ValueError(f"modality must be one of {MODALITIES}, got {self.modality!r}")
        if self.source_type not in SOURCE_TYPES:
            raise ValueError(f"source_type must be one of {SOURCE_TYPES}, got {self.source_type!r}")
        if self.missingness_flag not in MISSINGNESS_FLAGS:
            raise ValueError(f"missingness_flag must be one of {MISSINGNESS_FLAGS}, got {self.missingness_flag!r}")


LAYER1_COLUMNS = ["chrom", "pos", "ref", "alt", "modality", "source_type",
                  "evidence_value", "context", "missingness_flag"]


def records_to_dataframe(records):
    if not records:
        return pd.DataFrame(columns=LAYER1_COLUMNS)
    return pd.DataFrame([asdict(r) for r in records])[LAYER1_COLUMNS]


def validate_layer1_output(df):
    missing = [c for c in LAYER1_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Layer 1 output missing required column(s): {missing}")
    for col, allowed in (("source_type", SOURCE_TYPES), ("missingness_flag", MISSINGNESS_FLAGS),
                         ("modality", MODALITIES)):
        bad = set(df[col].dropna()) - set(allowed)
        if bad:
            raise ValueError(f"Layer 1 output has invalid {col} value(s): {bad}")
    for s in df["evidence_value"]:
        json.loads(s)   # raises if any evidence_value is not valid JSON


def validate_matrix_complete(evidence_df, variants, modalities=MODALITIES):
    """Every candidate variant must have >= 1 record for every modality (downstream layers depend on this)."""
    expected = {v.key for v in variants}
    problems = []
    for m in modalities:
        sub = evidence_df[evidence_df["modality"] == m]
        got = {canon_key(r.chrom, r.pos, r.ref, r.alt) for r in sub[["chrom", "pos", "ref", "alt"]].itertuples(index=False)}
        missing = expected - got
        if missing:
            problems.append(f"{m}: {len(missing)} candidate variant(s) have no record")
    if problems:
        raise ValueError("Evidence matrix incomplete -- " + "; ".join(problems))

In [10]:
Variant = namedtuple("Variant", "chrom pos ref alt key")


def canon_key(chrom, pos, ref, alt):
    ref, alt, pos = str(ref).upper(), str(alt).upper(), int(pos)
    while ref and alt and ref[-1] == alt[-1]:
        ref, alt = ref[:-1], alt[:-1]
    while ref and alt and ref[0] == alt[0]:
        ref, alt, pos = ref[1:], alt[1:], pos + 1
    return (str(chrom).replace("chr", ""), pos, ref, alt)


def plan_windows(positions, max_span):
    """Cluster sorted genomic positions into windows of at most max_span."""
    windows = []
    for p in sorted(set(positions)):
        if windows and p - windows[-1][0] < max_span:
            windows[-1][1] = p
        else:
            windows.append([p, p])
    return [(s, e) for s, e in windows]


def to_variants(df):
    out = []
    for r in df[["chrom", "pos", "ref", "alt"]].itertuples(index=False):
        chrom = str(r.chrom).replace("chr", "")
        out.append(Variant(chrom, int(r.pos), str(r.ref), str(r.alt), canon_key(chrom, r.pos, r.ref, r.alt)))
    return out


def make_record(v, modality, source_type, flag, context=None, **value):
    return EvidenceRecord(chrom=v.chrom, pos=v.pos, ref=v.ref, alt=v.alt, modality=modality,
                          source_type=source_type, evidence_value=ev(**value),
                          context=context, missingness_flag=flag)

In [11]:
def find_layer0_inputs(gene_symbol, explicit_csv=None):
    candidates = []
    if explicit_csv:
        _d = os.path.dirname(explicit_csv)
        _cands = [os.path.join(_d, f"{gene_symbol}_layer0_manifest.json"), os.path.join(_d, "manifest.json")]
        candidates.append((explicit_csv, next((m for m in _cands if os.path.exists(m)), _cands[-1]), "config path"))
    candidates.append((f"data/outputs/{gene_symbol}/layer0_regulatory_truthset.csv",
                       f"data/outputs/{gene_symbol}/manifest.json", "repo layout"))
    candidates.append((f"layer0_outputs/{gene_symbol}_layer0_regulatory_truthset.csv",
                       f"layer0_outputs/{gene_symbol}_layer0_manifest.json", "layer-0 notebook layout"))
    for csv_path, manifest_path, label in candidates:
        if os.path.exists(csv_path):
            return csv_path, manifest_path, label
    raise FileNotFoundError("No Layer 0 candidate CSV found. Looked in: " + ", ".join(c[0] for c in candidates))


def read_candidate_csv(csv_path):
    df = pd.read_csv(csv_path, dtype={"chrom": str})
    need = ["chrom", "pos", "ref", "alt"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} is missing required column(s): {missing}")
    df["chrom"] = df["chrom"].astype(str).str.replace("chr", "", regex=False)
    df["pos"] = df["pos"].astype(int)
    return df.drop_duplicates(subset=need).sort_values(["chrom", "pos"]).reset_index(drop=True)


def fetch_gene_metadata_fallback(gene_symbol, fetch):
    """Resolve live Ensembl metadata; only use the explicitly validated HBB GRCh38 fallback on failure."""
    url = f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene_symbol}"
    data, err, from_cache = fetch("GET", url, params={"content-type": "application/json"})
    if not err and isinstance(data, dict):
        meta = {"gene_symbol": gene_symbol, "gene_id": data.get("id"),
                "chrom": str(data.get("seq_region_name")), "start": int(data.get("start")),
                "end": int(data.get("end")), "strand": int(data.get("strand")),
                "metadata_source": "Ensembl REST lookup", "metadata_from_cache": bool(from_cache),
                "assembly_name": data.get("assembly_name", "unknown"),
                "annotation_provenance": "Ensembl REST lookup/symbol; response metadata"}
        if gene_symbol == "HBB" and meta["gene_id"] != "ENSG00000244734":
            raise ValueError(f"Ensembl returned unexpected HBB gene ID {meta['gene_id']!r}; refusing fallback substitution.")
        return meta
    if gene_symbol == "HBB":
        # Previously observed HBB locus; only for GRCh38 and only after exact gene-ID/coordinate validation.
        fallback = {"gene_symbol": "HBB", "gene_id": "ENSG00000244734", "chrom": "11",
                    "start": 5225464, "end": 5229395, "strand": -1,
                    "metadata_source": "validated_historical_fallback",
                    "metadata_from_cache": False, "assembly_name": "GRCh38",
                    "annotation_provenance": "Validated against Ensembl REST lookup/symbol and Alliance of Genome Resources HBB record; 1-based inclusive genomic coordinates; live lookup failed: " + str(err)}
        if (fallback["gene_id"] != "ENSG00000244734" or fallback["chrom"] != "11"
                or fallback["start"] != 5225464 or fallback["end"] != 5229395
                or fallback["strand"] != -1):
            raise ValueError("HBB fallback metadata failed internal validation.")
        print("WARNING: using validated historical HBB GRCh38 metadata fallback; confirm the Layer 0 manifest/assembly.")
        return fallback
    raise RuntimeError(f"Layer 0 manifest missing and Ensembl metadata lookup failed for {gene_symbol}: {err}")


def load_layer0_output(gene_symbol, explicit_csv=None, fetch=None):
    csv_path, manifest_path, layout = find_layer0_inputs(gene_symbol, explicit_csv)
    variants_df = read_candidate_csv(csv_path)
    info = {"csv_path": csv_path, "csv_sha256": _file_hash(csv_path), "layout": layout,
            "manifest_path": None, "n_variants": len(variants_df)}
    if manifest_path and os.path.exists(manifest_path):
        with open(manifest_path) as f:
            l0_manifest = json.load(f)
        gene_meta = l0_manifest["gene_metadata"]
        info["manifest_path"] = manifest_path
        info["clinvar_snapshot_date"] = l0_manifest.get("clinvar_snapshot_date")
    else:
        print("WARNING: no Layer 0 manifest found -- fetching gene metadata fresh from Ensembl. "
              "This is NOT pinned to the gene build/version Layer 0 used.")
        if fetch is None:
            raise ValueError("fetch function must be provided for fallback Ensembl gene metadata query.")
        gene_meta = fetch_gene_metadata_fallback(gene_symbol, fetch)
        info["gene_metadata_source"] = "fresh Ensembl lookup (not pinned to Layer 0)"
    return variants_df, gene_meta, info

In [12]:
GNOMAD_API = "https://gnomad.broadinstitute.org/api"

GNOMAD_REGION_QUERY = """
query($chrom: String!, $start: Int!, $stop: Int!, $dataset: DatasetId!) {
  region(chrom: $chrom, start: $start, stop: $stop, reference_genome: GRCh38) {
    variants(dataset: $dataset) {
      variant_id
      pos
      ref
      alt
      genome { ac an }
    }
  }
}
"""


def collect_gnomad_evidence(variants, dataset, window_bp, delay_s, fetch):
    result = {"windows": [], "by_key": {}}
    by_chrom = defaultdict(list)
    for v in variants:
        by_chrom[v.chrom].append(v.pos)
    first_network_call = True
    for chrom, positions in by_chrom.items():
        for s, e in plan_windows(positions, window_bp):
            start, stop = max(1, s - 1), e + 1
            payload, err, from_cache = fetch("POST", GNOMAD_API, json_body={
                "query": GNOMAD_REGION_QUERY,
                "variables": {"chrom": chrom, "start": start, "stop": stop, "dataset": dataset}})
            # Delay only after actual network calls, never on cache hits
            if not from_cache and delay_s:
                if not first_network_call:
                    time.sleep(delay_s)
                first_network_call = False

            if not err and payload and payload.get("errors"):
                err = "graphql_error: " + str(payload["errors"][0].get("message", ""))[:120]
            win = {"chrom": chrom, "start": s, "stop": e, "error": err, "n_variants": 0, "from_cache": from_cache}
            if not err:
                region = ((payload or {}).get("data") or {}).get("region") or {}
                for gv in region.get("variants") or []:
                    genome = gv.get("genome")
                    if not genome or not genome.get("an") or not genome.get("ac"):
                        continue
                    key = canon_key(chrom, gv["pos"], gv["ref"], gv["alt"])
                    result["by_key"][key] = {"ac": genome["ac"], "an": genome["an"],
                                             "af": genome["ac"] / genome["an"],
                                             "variant_id": gv.get("variant_id")}
                    win["n_variants"] += 1
            result["windows"].append(win)
    return result


def build_gnomad_records(gnomad, variants, dataset):
    records = []
    for v in variants:
        hit = gnomad["by_key"].get(v.key)
        if hit:
            records.append(make_record(v, "gnomad_af", "curated_clinical", "observed_positive",
                                       context=f"{dataset}:genomes", af=hit["af"], ac=hit["ac"], an=hit["an"],
                                       gnomad_variant_id=hit["variant_id"]))
            continue
        failed = [w for w in gnomad["windows"] if w["chrom"] == v.chrom and w["start"] <= v.pos <= w["stop"] and w["error"]]
        if failed:
            records.append(make_record(v, "gnomad_af", "curated_clinical", "absent",
                                       context=f"{dataset}:genomes", reason="query_failed", error=failed[0]["error"]))
        else:
            records.append(make_record(v, "gnomad_af", "curated_clinical", "absent",
                                       context=f"{dataset}:genomes", reason="not_observed_in_genome_callset"))
    return records

In [13]:

GWAS_BASE = "https://www.ebi.ac.uk/gwas/rest/api/v2"


def _hal_embedded(payload, key):
    """Item list from a HAL (_embedded[key]) or plain (key / content / results / data / items) payload."""
    if isinstance(payload, list):
        return payload
    if not isinstance(payload, dict):
        return []
    emb = payload.get("_embedded")
    if isinstance(emb, dict) and isinstance(emb.get(key), list):
        return emb[key]
    for k in (key, "content", "results", "data", "items"):
        if isinstance(payload.get(k), list):
            return payload[k]
    return []


def _unrecognized_payload(payload, key):
    """True when a 200 response is a dict with neither the expected item container nor any HAL/paging
    marker -- the API shape changed. That must be an ERROR, never silently read as 'no results'."""
    if payload is None or isinstance(payload, list):
        return False
    if not isinstance(payload, dict):
        return True
    markers = ("_embedded", "_links", "links", "page", key, "content", "results", "data", "items")
    return bool(payload) and not any(m in payload for m in markers)


def _next_href(payload):
    if not isinstance(payload, dict):
        return None
    for container in ("_links", "links"):
        links = payload.get(container)
        nxt = links.get("next") if isinstance(links, dict) else None
        if isinstance(nxt, dict) and nxt.get("href"):
            return nxt["href"]
        if isinstance(nxt, str) and nxt:
            return nxt
    nxt = payload.get("next")
    return nxt if isinstance(nxt, str) and nxt else None


def _clean_href(href):
    return href.split("{")[0] if href else href


def fetch_paginated_hal(url, params, embedded_key, fetch, max_pages=200):
    """Follow HAL _links.next and collect all pages."""
    items = []
    next_url = url
    next_params = dict(params or {})
    pages = 0

    while next_url and pages < max_pages:
        payload, err, _ = fetch("GET", next_url, params=next_params)
        if err:
            return items, err

        if _unrecognized_payload(payload, embedded_key):
            return items, "unrecognized_response_shape"
        items.extend(_hal_embedded(payload, embedded_key))

        next_url = _clean_href(_next_href(payload))
        next_params = None
        pages += 1

    if next_url:
        return items, f"truncated_after_{max_pages}_pages"

    return items, None


def gwas_snp_position(snp, chrom):
    for loc in snp.get("locations") or []:
        if (
            str(loc.get("chromosome_name")) == str(chrom)
            and loc.get("chromosome_position") is not None
        ):
            return int(loc["chromosome_position"])
    return None


def assoc_pvalue(assoc):
    p = assoc.get("p_value")
    if p is not None:
        try:
            return float(p)
        except (TypeError, ValueError):
            pass

    m = assoc.get("pvalue_mantissa")
    e = assoc.get("pvalue_exponent")

    if m is not None and e is not None:
        try:
            return float(m) * 10 ** float(e)
        except (TypeError, ValueError):
            pass

    return None


def resolve_traits(assoc, fetch=None):
    traits = assoc.get("efo_traits") or []
    names = [
        t.get("efo_trait")
        for t in traits
        if isinstance(t, dict) and t.get("efo_trait")
    ]

    if not names:
        names = [
            t for t in (assoc.get("reported_trait") or [])
            if isinstance(t, str) and t
        ]

    return names


def diagnose_gwas_endpoint(chrom, start, stop, timeout=20):
    """One raw probe of the SNP-by-region call. Runs automatically when a GWAS window fails and prints
    what the server actually returned (status, final URL, content type, top-level JSON keys)."""
    url = f"{GWAS_BASE}/single-nucleotide-polymorphisms"
    params = {"chromosome": str(chrom), "bp_start": int(start), "bp_end": int(stop), "size": 5, "page": 0}
    try:
        r = requests.get(url, params=params, timeout=timeout, headers={"Accept": "application/json"})
    except requests.exceptions.RequestException as e:
        print(f"GWAS diagnostic probe: request failed ({type(e).__name__}: {e})")
        return None
    try:
        j = r.json()
        keys = sorted(j.keys()) if isinstance(j, dict) else f"list[{len(j)}]"
    except ValueError:
        keys = None
    info = {"status": r.status_code, "url": r.url, "content_type": r.headers.get("Content-Type"),
            "top_level_keys": keys, "body_head": r.text[:200]}
    print("GWAS diagnostic probe:")
    for k, v in info.items():
        print(f"  {k}: {v}")
    if r.status_code in (404, 410):
        print("  -> The endpoint/path is gone or renamed. Open https://www.ebi.ac.uk/gwas/rest/api/v2/swagger-ui/index.html, "
              "find the SNP/variant region filter, and or keep layer1.gwas_source: auto so the official bulk release file is used instead.")
    return info


def collect_gwas_evidence(chrom, positions, fetch, size=500, window_bp=25000):
    result = {"windows": [], "by_pos": {}, "n_snps_in_region": 0}

    if not positions:
        return result

    wanted = set(positions)
    windows = plan_windows(positions, window_bp)
    all_snps = []

    for s, e in windows:
        start, stop = max(1, s - 1), e + 1

        snps, err = fetch_paginated_hal(
            f"{GWAS_BASE}/single-nucleotide-polymorphisms",
            {
                "chromosome": str(chrom),
                "bp_start": start,
                "bp_end": stop,
                "size": size,
                "page": 0,
            },
            "snps",
            fetch,
        )

        result["windows"].append({
            "chrom": str(chrom),
            "start": s,
            "stop": e,
            "error": err,
            "n_snps": len(snps),
        })

        all_snps.extend(snps)

    # Deduplicate SNPs because adjacent windows can overlap.
    unique_snps = {}
    for snp in all_snps:
        rs = snp.get("rs_id")
        pos = gwas_snp_position(snp, chrom)
        key = (rs, pos) if rs else (None, pos, str(snp))
        unique_snps[key] = snp

    all_snps = list(unique_snps.values())
    result["n_snps_in_region"] = len(all_snps)

    for snp in all_snps:
        pos = gwas_snp_position(snp, chrom)

        if pos not in wanted:
            continue

        entry = result["by_pos"].setdefault(
            pos, {"rs_ids": [], "assocs": [], "error": None}
        )

        rs = snp.get("rs_id")
        if rs and rs not in entry["rs_ids"]:
            entry["rs_ids"].append(rs)

        if not rs:
            entry["error"] = entry["error"] or "missing_rs_id"
            continue

        assocs, aerr = fetch_paginated_hal(
            f"{GWAS_BASE}/associations",
            {"rs_id": rs, "size": size, "page": 0},
            "associations",
            fetch,
        )

        if aerr:
            entry["error"] = aerr

        for assoc in assocs:
            entry["assocs"].append({
                "traits": resolve_traits(assoc) or ["unresolved"],
                "pvalue": assoc_pvalue(assoc),
                "rs_id": rs,
            })

    return result


def _gwas_norm_col(c):
    return re.sub(r"\s+", "_", str(c).strip().upper())


_GWAS_WANTED = {"CHR_ID", "CHR_POS", "SNPS", "P-VALUE", "PVALUE_MLOG", "DISEASE/TRAIT", "MAPPED_TRAIT"}


def _split_multi(value):
    """CHR_ID / CHR_POS / SNPS hold several values for multi-SNP rows ('a; b' or 'a x b')."""
    if not isinstance(value, str):
        return []
    return [t.strip() for t in re.split(r"\s*;\s*|\s+x\s+", value) if t.strip()]


def download_gwas_catalog(url, cache_dir, ttl_days=14, timeout=300):
    """Download (or reuse) the official GWAS Catalog bulk associations file. Returns (path, error).
    A stale cached copy is reused if the download fails (its age is recorded by the caller)."""
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, os.path.basename(url))
    fresh = os.path.exists(path) and (time.time() - os.path.getmtime(path)) < ttl_days * 86400
    if fresh:
        return path, None
    tmp = path + ".part"
    try:
        with requests.get(url, stream=True, timeout=timeout) as r:
            r.raise_for_status()
            with open(tmp, "wb") as fh:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
        os.replace(tmp, path)
        return path, None
    except (requests.exceptions.RequestException, OSError) as e:
        if os.path.exists(path):
            print(f"WARNING: bulk GWAS download failed ({type(e).__name__}); using the older cached copy.")
            return path, None
        return None, f"bulk_download_failed:{type(e).__name__}"


def collect_gwas_evidence_from_download(chrom, positions, path, window_bp=25000):
    """Same output shape as collect_gwas_evidence(), read from the official bulk TSV (zip).
    Still position-level (the file lists the strongest SNP per association), not allele-specific."""
    import zipfile
    result = {"windows": [], "by_pos": {}, "n_snps_in_region": 0,
              "source_label": "GWAS Catalog bulk download (associations file)"}
    if not positions:
        return result
    wanted = set(positions)
    windows = plan_windows(positions, window_bp)
    chrom = str(chrom)
    frames = []
    try:
        with zipfile.ZipFile(path) as zf:
            members = [m for m in zf.namelist() if m.lower().endswith((".tsv", ".txt"))]
            if not members:
                raise ValueError("no tsv member in zip")
            with zf.open(members[0]) as fh:
                for chunk in pd.read_csv(fh, sep="\t", dtype=str, chunksize=200_000, low_memory=False,
                                         usecols=lambda c: _gwas_norm_col(c) in _GWAS_WANTED,
                                         on_bad_lines="skip"):
                    chunk.columns = [_gwas_norm_col(c) for c in chunk.columns]
                    keep = chunk["CHR_ID"].fillna("").str.contains(rf"(?:^|\s|;){re.escape(chrom)}(?:$|\s|;)", regex=True)
                    if keep.any():
                        frames.append(chunk[keep])
    except (OSError, ValueError, KeyError, zipfile.BadZipFile, pd.errors.ParserError) as e:
        err = f"bulk_file_unreadable:{type(e).__name__}"
        result["windows"] = [{"chrom": chrom, "start": s, "stop": e_, "error": err, "n_snps": 0} for s, e_ in windows]
        return result

    snps = {}   # (rs_id, pos) -> list of association dicts
    for df in frames:
        for rec in df.to_dict("records"):
            chrs, poss, rss = (_split_multi(rec.get(k)) for k in ("CHR_ID", "CHR_POS", "SNPS"))
            n = max(len(chrs), len(poss), len(rss), 1)
            chrs = chrs * n if len(chrs) == 1 else chrs
            rss = rss + [None] * (n - len(rss))
            try:
                p = float(rec.get("P-VALUE"))
            except (TypeError, ValueError):
                p = None
            if p is None or p != p:          # missing / NaN -> derive from -log10(p) when available
                try:
                    p = 10 ** (-float(rec.get("PVALUE_MLOG")))
                except (TypeError, ValueError):
                    p = None
                if p is not None and p != p:
                    p = None
            trait = rec.get("MAPPED_TRAIT") or rec.get("DISEASE/TRAIT")
            trait = trait if isinstance(trait, str) and trait else "unresolved"
            for c, ps, rs in zip(chrs, poss, rss):
                if c != chrom or not str(ps).isdigit():
                    continue
                snps.setdefault((rs, int(ps)), []).append({"traits": [trait], "pvalue": p, "rs_id": rs})

    lo, hi = min(s for s, _ in windows) - 1, max(e for _, e in windows) + 1
    result["n_snps_in_region"] = len({k for k in snps if lo <= k[1] <= hi})
    for s, e_ in windows:
        n_here = len({k for k in snps if s - 1 <= k[1] <= e_ + 1})
        result["windows"].append({"chrom": chrom, "start": s, "stop": e_, "error": None, "n_snps": n_here})
    for (rs, pos), assocs in snps.items():
        if pos in wanted:
            entry = result["by_pos"].setdefault(pos, {"rs_ids": [], "assocs": [], "error": None})
            if rs and rs not in entry["rs_ids"]:
                entry["rs_ids"].append(rs)
            entry["assocs"].extend(assocs)
    return result


def gwas_via_bulk_file(chrom, positions, L1):
    """Download/reuse the bulk file and extract GWAS evidence. Returns (result, file_info, error)."""
    import datetime as _dt
    cache_dir = os.path.join(os.path.dirname(str(L1["http_cache_dir"]).rstrip("/")) or ".", "gwas_bulk")
    path, err = download_gwas_catalog(L1["gwas_download_url"], cache_dir, L1.get("gwas_download_ttl_days", 14))
    if err:
        return None, None, err
    res = collect_gwas_evidence_from_download(chrom, positions, path, window_bp=L1.get("gwas_window_bp", 25000))
    info = {"url": L1["gwas_download_url"], "path": path, "sha256": _file_hash(str(path)),
            "file_mtime_utc": _dt.datetime.fromtimestamp(os.path.getmtime(path), _dt.timezone.utc).isoformat()}
    return res, info, None


def _gwas_problems(g):
    errs = sorted({str(w["error"]) for w in g["windows"] if w["error"]})
    pos_errs = [p for p, e in g["by_pos"].items() if e.get("error")]
    return errs, pos_errs


def _gwas_warnings(g, label):
    errs, pos_errs = _gwas_problems(g)
    out = []
    if errs:
        bad = sum(1 for w in g["windows"] if w["error"])
        out.append(f"GWAS ({label}): {bad} of {len(g['windows'])} window(s) failed query ({', '.join(errs)})")
    if pos_errs:
        out.append(f"GWAS ({label}): association query errors at {len(pos_errs)} candidate position(s)")
    return out


def collect_gwas_with_fallback(chrom, positions, fetch, L1, bulk_fn=None, probe_fn=None):
    """layer1.gwas_source: 'api' | 'download' | 'auto' (REST API first, official bulk file if it fails).
    Returns {'gwas', 'used', 'file_info', 'warnings', 'notes'}; only unresolved failures become warnings."""
    bulk_fn = bulk_fn or gwas_via_bulk_file
    probe_fn = probe_fn or diagnose_gwas_endpoint
    mode = str(L1.get("gwas_source", "auto")).lower()
    if mode not in ("auto", "api", "download"):
        raise ValueError(f"layer1.gwas_source must be auto, api or download (got {mode!r})")
    out = {"gwas": None, "used": None, "file_info": None, "warnings": [], "notes": []}
    api_bad, api_errs = False, []
    if mode in ("auto", "api"):
        api = collect_gwas_evidence(chrom, positions, fetch=fetch, window_bp=L1.get("gwas_window_bp", 25000))
        out["gwas"], out["used"] = api, "api"
        api_errs, api_pos_errs = _gwas_problems(api)
        api_bad = bool(api_errs or api_pos_errs)
        if api_errs:
            w0 = next(w for w in api["windows"] if w["error"])
            probe_fn(w0["chrom"], w0["start"], w0["stop"])
        if mode == "api" or not api_bad:
            out["warnings"] = _gwas_warnings(api, "REST API")
            return out
    bulk, info, derr = bulk_fn(chrom, positions, L1)
    bulk_errs = _gwas_problems(bulk)[0] if bulk else []
    if not derr and not bulk_errs:
        out.update(gwas=bulk, used="download", file_info=info)
        if api_bad:
            out["notes"].append(f"GWAS REST API failed ({', '.join(api_errs) or 'position-level errors'}); evidence taken from "
                                f"the official bulk release file instead (recorded in the manifest).")
        return out
    detail = derr or ", ".join(bulk_errs)
    if out["gwas"] is None:   # download-only mode, and it failed
        out["gwas"] = bulk if bulk else {"windows": [{"chrom": str(chrom), "start": min(positions), "stop": max(positions),
                                                      "error": derr, "n_snps": 0}], "by_pos": {}, "n_snps_in_region": 0,
                                         "source_label": "GWAS Catalog bulk download (associations file)"}
        out["used"] = "download"
        out["warnings"] = _gwas_warnings(out["gwas"], "bulk file")
    else:
        out["warnings"] = _gwas_warnings(out["gwas"], "REST API") + [f"GWAS: bulk release file also unavailable ({detail})"]
    return out


def build_gwas_records(gwas, variants, p_threshold):
    """Create one evidence record per candidate variant from GWAS Catalog position matches.

    The API query identifies SNPs by chromosome/position, not by the candidate's ref/alt alleles.
    Therefore all emitted records explicitly mark the match as position-level, not allele-specific.
    """
    records = []
    src_label = gwas.get("source_label") or "GWAS Catalog REST API v2"
    by_pos = gwas.get("by_pos") or {}
    windows = gwas.get("windows") or []
    for v in variants:
        entry = by_pos.get(v.pos)
        if entry:
            assocs = entry.get("assocs") or []
            details = []
            valid_pvalues = []
            for assoc in assocs:
                p = assoc.get("pvalue")
                if p is not None:
                    try:
                        p = float(p)
                        valid_pvalues.append(p)
                    except (TypeError, ValueError):
                        p = None
                details.append({
                    "rs_id": assoc.get("rs_id"),
                    "traits": assoc.get("traits") or ["unresolved"],
                    "pvalue": p,
                })
            common = {
                "matching_level": "position",
                "allele_specific": False,
                "candidate_alleles": {"ref": v.ref, "alt": v.alt},
                "candidate_position": int(v.pos),
                "rs_ids": entry.get("rs_ids") or [],
                "associations": details,
                "source": src_label,
                "p_value_threshold": float(p_threshold),
            }
            if entry.get("error") and not assocs:
                records.append(make_record(v, "gwas", "experimental", "absent",
                    context="GWAS Catalog", reason="query_failed", error=entry["error"], **common))
            elif not assocs:
                records.append(make_record(v, "gwas", "experimental", "absent",
                    context="GWAS Catalog", reason="no_usable_association_records", **common))
            elif not valid_pvalues:
                records.append(make_record(v, "gwas", "experimental", "absent",
                    context="GWAS Catalog", reason="p_value_missing", **common))
            elif any(p <= p_threshold for p in valid_pvalues):
                sig_traits = sorted({trait for a in details if a["pvalue"] is not None
                                     and a["pvalue"] <= p_threshold for trait in a["traits"]})
                records.append(make_record(v, "gwas", "experimental", "observed_positive",
                    context="GWAS Catalog", traits=sig_traits, min_pvalue=min(valid_pvalues), **common))
            else:
                records.append(make_record(v, "gwas", "experimental", "observed_null",
                    context="GWAS Catalog", reason="subthreshold_association",
                    traits=sorted({t for a in details for t in a["traits"]}),
                    min_pvalue=min(valid_pvalues), **common))
            continue

        failed = [w for w in windows if w.get("chrom") == v.chrom
                  and int(w.get("start", 0)) <= v.pos <= int(w.get("stop", -1)) and w.get("error")]
        if failed:
            records.append(make_record(v, "gwas", "experimental", "absent",
                context="GWAS Catalog", reason="incomplete_region_query",
                errors=[w.get("error") for w in failed],
                matching_level="position", allele_specific=False,
                candidate_alleles={"ref": v.ref, "alt": v.alt},
                source=src_label))
        else:
            records.append(make_record(v, "gwas", "experimental", "absent",
                context="GWAS Catalog", reason="not_in_catalog",
                matching_level="position", allele_specific=False,
                candidate_alleles={"ref": v.ref, "alt": v.alt},
                source=src_label))
    return records


In [14]:
GTEX_BASE = "https://gtexportal.org/api/v2"


def resolve_versioned_gencode_id(gene_symbol, ensembl_id, gencode_version, fetch):
    attempts = ({"geneId": gene_symbol, "gencodeVersion": gencode_version, "genomeBuild": "GRCh38/hg38"},
                {"geneId": gene_symbol})
    for params in attempts:
        payload, err, _ = fetch("GET", f"{GTEX_BASE}/reference/gene", params=params)
        if err:
            continue
        records = (payload or {}).get("data") or []
        for rec in records:
            gid = rec.get("gencodeId") or ""
            if ensembl_id and gid.split(".")[0] == ensembl_id:
                return gid, None
        for rec in records:
            if rec.get("geneSymbol") == gene_symbol and rec.get("gencodeId"):
                return rec["gencodeId"], "matched_by_symbol_not_ensembl_id"
    return None, "gene_not_resolved"


def parse_gtex_variant_id(variant_id):
    if not variant_id:
        return None
    parts = variant_id.split("_")
    if len(parts) < 4:
        return None
    try:
        pos = int(parts[1])
    except ValueError:
        return None
    return parts[0].replace("chr", ""), pos, parts[2], parts[3]


def fetch_tissue_eqtls(gencode_id, tissue, dataset, fetch, items_per_page, max_pages):
    rows, seen = [], set()
    for page in range(max_pages):
        payload, err, _ = fetch("GET", f"{GTEX_BASE}/association/singleTissueEqtl",
                                params={"gencodeId": gencode_id, "tissueSiteDetailId": tissue,
                                        "datasetId": dataset, "itemsPerPage": items_per_page, "page": page})
        if err:
            return rows, err
        data = (payload or {}).get("data") or []
        new = 0
        for rec in data:
            vid = rec.get("variantId")
            if vid in seen:
                continue
            seen.add(vid)
            rows.append(rec)
            new += 1
        n_pages = ((payload or {}).get("paging_info") or {}).get("numberOfPages")
        if len(data) < items_per_page:
            return rows, None
        if new == 0:
            return rows, "pagination_not_advancing"
        if n_pages is not None and page + 1 >= n_pages:
            return rows, None
    return rows, f"truncated_after_{max_pages}_pages"


def collect_eqtl_evidence(gene_meta, tissues, dataset, gencode_version, items_per_page, max_pages, fetch):
    out = {"gencode_id": None, "resolve_note": None, "resolve_error": None, "tissues": {}}
    gid, note = resolve_versioned_gencode_id(gene_meta["gene_symbol"], gene_meta.get("gene_id"), gencode_version, fetch)
    if gid is None:
        out["resolve_error"] = note
        return out
    out["gencode_id"], out["resolve_note"] = gid, note
    for tissue in tissues:
        rows, err = fetch_tissue_eqtls(gid, tissue, dataset, fetch, items_per_page, max_pages)
        by_key = defaultdict(list)
        for rec in rows:
            parsed = parse_gtex_variant_id(rec.get("variantId"))
            if not parsed:
                continue
            by_key[canon_key(*parsed)].append({"slope": rec.get("nes", rec.get("slope")),
                                                "pvalue": rec.get("pValue", rec.get("pval")),
                                                "variant_id": rec.get("variantId")})
        out["tissues"][tissue] = {"error": err, "n_rows": len(rows), "by_key": dict(by_key)}
    return out


def build_eqtl_records(eqtl, variants, tissues, maf_by_key, min_maf):
    records = []
    for tissue in tissues:
        tinfo = eqtl["tissues"].get(tissue)
        for v in variants:
            if eqtl.get("resolve_error") or tinfo is None:
                records.append(make_record(v, "eqtl", "experimental", "absent", context=tissue,
                                           reason="query_failed",
                                           error=eqtl.get("resolve_error") or "tissue_not_queried"))
                continue
            hits = tinfo["by_key"].get(v.key)
            if hits:
                for h in hits:
                    records.append(make_record(v, "eqtl", "experimental", "observed_positive", context=tissue,
                                               slope=h["slope"], pvalue=h["pvalue"], gtex_variant_id=h["variant_id"]))
                continue
            if tinfo["error"]:
                records.append(make_record(v, "eqtl", "experimental", "absent", context=tissue,
                                           reason="incomplete_tissue_query", error=tinfo["error"]))
                continue
            maf = maf_by_key.get(v.key)
            # GTEx's singleTissueEqtl endpoint returns significant associations, not a complete
            # per-variant tested-null table. Therefore, absence of an exact candidate allele from
            # the significant results cannot establish a tested negative, even if other gene rows
            # exist. Only returned exact associations are observed_positive; non-hits stay absent
            # with test status unverified (or not-testable when frequency is rare/unknown).
            if maf is not None and maf >= min_maf:
                records.append(make_record(
                    v, "eqtl", "experimental", "absent", context=tissue,
                    reason="no_significant_eqtl_association_returned_test_status_unverified",
                    maf=maf, gtex_gene_rows=int(tinfo.get("n_rows", 0)), test_status="unverified"
                ))
            else:
                records.append(make_record(v, "eqtl", "experimental", "absent", context=tissue,
                                           reason="not_testable_rare_or_unknown_frequency", maf=maf))
    return records

In [15]:
MPRA_COLUMNS = ["chrom", "pos", "ref", "alt", "cell_type", "log2_allelic_skew", "pvalue", "dataset_name"]
CRISPR_COLUMNS = ["chrom", "pos", "ref", "alt", "target_gene", "cell_type", "effect_size", "pvalue",
                  "screen_type", "dataset_name"]


def load_local_evidence_table(path, expected_columns, label, numeric_columns=()):
    if not path:
        return None
    if not os.path.exists(path):
        raise FileNotFoundError(f"{label} evidence path is configured but the file does not exist: {path}")
    df = pd.read_csv(path, sep=None, engine="python", dtype=str)
    missing = [c for c in expected_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{label} evidence table at {path} is missing expected column(s): {missing}. "
                         f"Expected: {expected_columns}")
    df["chrom"] = df["chrom"].astype(str).str.replace("chr", "", regex=False)
    df["pos"] = pd.to_numeric(df["pos"], errors="coerce")
    n_bad = int(df["pos"].isna().sum())
    if n_bad:
        print(f"WARNING: {label} table has {n_bad} row(s) with a non-numeric pos; dropped.")
        df = df.dropna(subset=["pos"])
    df["pos"] = df["pos"].astype(int)
    df["ref"], df["alt"] = df["ref"].str.upper(), df["alt"].str.upper()
    for c in ["pvalue", *numeric_columns]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    index = defaultdict(list)
    for row in df.to_dict("records"):
        index[canon_key(row["chrom"], row["pos"], row["ref"], row["alt"])].append(row)
    return dict(index)


def build_local_table_records(modality, variants, index, p_threshold, value_fn):
    records = []
    for v in variants:
        if index is None:
            records.append(make_record(v, modality, "experimental", "absent", reason="no_data_source_configured"))
            continue
        rows = index.get(v.key)
        if not rows:
            records.append(make_record(v, modality, "experimental", "absent", reason="not_tested_in_provided_table"))
            continue
        for row in rows:
            p, ctx = row.get("pvalue"), row.get("cell_type")
            if p is None or p != p:
                records.append(make_record(v, modality, "experimental", "absent", context=ctx, reason="p_value_missing"))
            elif p <= p_threshold:
                records.append(make_record(v, modality, "experimental", "observed_positive", context=ctx,
                                           **value_fn(row)))
            else:
                records.append(make_record(v, modality, "experimental", "observed_null", context=ctx,
                                           reason="tested_not_significant", pvalue=p))
    return records


def mpra_value(row):
    return {"log2_allelic_skew": row.get("log2_allelic_skew"), "pvalue": row.get("pvalue"),
            "dataset": row.get("dataset_name")}


def crispr_value(row):
    return {"target_gene": row.get("target_gene"), "effect_size": row.get("effect_size"),
            "pvalue": row.get("pvalue"), "screen_type": row.get("screen_type"),
            "dataset": row.get("dataset_name")}

## 4b. Layer 0 run and provenance manifest

Regenerates Layer 0 (if enabled) and always writes `layer0_outputs/HBB_layer0_manifest.json` (gene metadata, assembly, ClinVar snapshot, CSV hash) so Layer 1 can pin the gene build.


In [16]:
# Layer 0: generate (optional) and ALWAYS write a provenance manifest next to the candidate CSV,
# so Layer 1 can pin the gene build/version instead of re-fetching it unpinned.
_l0_cfg = CONFIG.get("layer0") or {}
LAYER0_MANIFEST_PATH = Path("layer0_outputs") / f"{GENE_SYMBOL}_layer0_manifest.json"

_l0_cache = HTTPCache(cache_dir=L1["http_cache_dir"], enabled=L1["use_http_cache"],
                      ttl_days=L1.get("cache_ttl_days", 7), refresh=L1.get("refresh_cache", False))
_l0_fetch = make_http_fetch(_l0_cache)


def write_layer0_manifest(mode, gene_meta, csv_path, extra=None, manifest_path=LAYER0_MANIFEST_PATH):
    cand = read_candidate_csv(csv_path)
    flank = FLANK_BP
    lo, hi = int(gene_meta["start"]) - flank, int(gene_meta["end"]) + flank
    outside = cand[(cand["chrom"] != str(gene_meta["chrom"])) | (cand["pos"] < lo) | (cand["pos"] > hi)]
    labels = pd.read_csv(csv_path, dtype={"chrom": str})
    payload = {
        "layer": 0,
        "notebook_version": "layer0_v1",
        "mode": mode,                         # "generated" (this notebook) or "supplied_csv"
        "gene_symbol": GENE_SYMBOL,
        "gene_metadata": gene_meta,
        "genome_assembly": "GRCh38",
        "canonical_transcript": _l0_cfg.get("canonical_transcript"),
        "flank_bp": flank,
        "promoter_window_bp": _l0_cfg.get("promoter_window_bp"),
        "clinvar_snapshot_date": _l0_cfg.get("clinvar_snapshot_date"),
        "clinvar_snapshot_provenance": _l0_cfg.get("clinvar_snapshot_note"),
        "csv_path": str(csv_path),
        "csv_sha256": _file_hash(str(csv_path)),
        "n_variants": int(len(cand)),
        "label_counts": ({str(k): int(v) for k, v in labels["label"].value_counts().items()}
                         if "label" in labels.columns else {}),
        "n_variants_outside_search_region": int(len(outside)),
    }
    payload.update(extra or {})
    path = write_manifest(str(manifest_path.parent), manifest_path.name, payload)
    return path, payload


if RUN_LAYER0:
    print("Regenerating Layer 0 candidate truth set (this downloads ClinVar + chr11 and calls Ensembl VEP)...")
    _final_df, _gene_meta, _info = run_layer0(_l0_cfg, GENE_SYMBOL, FLANK_BP)
    LAYER0_DEST.parent.mkdir(parents=True, exist_ok=True)
    _final_df.to_csv(LAYER0_DEST, index=False)
    _l0_cfg = {**_l0_cfg, "clinvar_snapshot_date": _info["clinvar_snapshot_date"],
               "clinvar_snapshot_note": "pinned/discovered by this run"}
    _path, _payload = write_layer0_manifest("generated", _gene_meta, LAYER0_DEST, extra=_info)
    print(f"Layer 0 written: {LAYER0_DEST} ({len(_final_df)} candidates); manifest: {_path}")
else:
    _existing = None
    if LAYER0_MANIFEST_PATH.exists():
        try:
            _existing = json.loads(LAYER0_MANIFEST_PATH.read_text())
        except ValueError:
            _existing = None
    if _existing and _existing.get("csv_sha256") == _file_hash(str(LAYER0_DEST)):
        print(f"Layer 0 manifest already matches the supplied CSV: {LAYER0_MANIFEST_PATH}")
    else:
        _gene_meta = fetch_gene_metadata_fallback(GENE_SYMBOL, _l0_fetch)
        _path, _payload = write_layer0_manifest("supplied_csv", _gene_meta, LAYER0_DEST)
        print(f"Wrote Layer 0 manifest for the supplied CSV: {_path}")
        if _payload["n_variants_outside_search_region"]:
            print(f"WARNING: {_payload['n_variants_outside_search_region']} supplied variant(s) lie outside the "
                  f"gene +/- {FLANK_BP} bp search region or on another chromosome -- check the assembly of the CSV.")
        print(f"ClinVar snapshot recorded as {_payload['clinvar_snapshot_date']} "
              f"({_payload['clinvar_snapshot_provenance']}). Edit layer0.clinvar_snapshot_date if that is wrong.")


Wrote Layer 0 manifest for the supplied CSV: layer0_outputs/HBB_layer0_manifest.json
ClinVar snapshot recorded as 20261019 (date reported by the original Layer 0 run; not verifiable from the CSV itself). Edit layer0.clinvar_snapshot_date if that is wrong.


## 5. Layer 1 collection and output validation


In [17]:
L1["layer0_csv_path"] = str(LAYER0_DEST)
L1["strict"] = True
# Initialize cache and bound fetch function
http_cache = HTTPCache(
    cache_dir=L1["http_cache_dir"],
    enabled=L1["use_http_cache"],
    ttl_days=L1.get("cache_ttl_days", 7),
    refresh=L1.get("refresh_cache", False)
)
fetch = make_http_fetch(http_cache)

try:
    variants_df, gene_meta, layer0_info = load_layer0_output(GENE_SYMBOL, L1["layer0_csv_path"], fetch=fetch)
    variants = to_variants(variants_df)
    print(f"Loaded {len(variants)} candidate variants for {GENE_SYMBOL} from {layer0_info['csv_path']} ({layer0_info['layout']})")
    print(f"Gene metadata: {gene_meta}")
except FileNotFoundError as e:
    if L1.get("strict"):
        raise
    variants_df, gene_meta, layer0_info, variants = None, None, None, None
    print(f"INFO: Layer 0 outputs not found yet ({e}).")
    print("To run live collection, execute Layer 0 first to produce candidate CSVs.")
    print("You can still proceed to Section 10 to run the full self-contained test suite.")

Loaded 112 candidate variants for HBB from /content/layer0_outputs/HBB_layer0_regulatory_truthset.csv (config path)
Gene metadata: {'gene_symbol': 'HBB', 'gene_id': 'ENSG00000244734', 'chrom': '11', 'start': 5225464, 'end': 5229395, 'strand': -1, 'metadata_source': 'Ensembl REST lookup', 'metadata_from_cache': False, 'assembly_name': 'GRCh38', 'annotation_provenance': 'Ensembl REST lookup/symbol; response metadata'}


In [18]:
if variants is None:
    print("Skipping live collection: Layer 0 candidate variants not loaded.")
    evidence_df = None
else:
    RUN_WARNINGS = []
    if {v.chrom for v in variants} != {str(gene_meta["chrom"])}:
        raise ValueError("Candidate variants span more than the gene's chromosome; "
                         "multi-chromosome candidate sets are not supported in this notebook.")
    chrom = str(gene_meta["chrom"])

    # --- gnomAD ---
    print("Collecting gnomAD allele frequencies...")
    gnomad = collect_gnomad_evidence(variants, L1["gnomad_dataset"], L1["gnomad_window_bp"],
                                     L1["gnomad_request_delay_s"], fetch=fetch)
    maf_by_key = {k: min(h["af"], 1 - h["af"]) for k, h in gnomad["by_key"].items()}
    for w in gnomad["windows"]:
        if w["error"]:
            RUN_WARNINGS.append(f"gnomAD window {w['chrom']}:{w['start']}-{w['stop']} failed: {w['error']}")
    print(f"gnomAD: {len(gnomad['windows'])} window(s), {len(gnomad['by_key'])} variant(s) with genome AF")

    # --- GWAS Catalog (Windowed) ---
    print("Collecting GWAS Catalog associations (windowed)...")
    positions = sorted({v.pos for v in variants})
    _g = collect_gwas_with_fallback(chrom, positions, fetch, L1)
    gwas, gwas_used, gwas_file_info = _g["gwas"], _g["used"], _g["file_info"]
    RUN_WARNINGS.extend(_g["warnings"])
    for _n in _g["notes"]:
        print("INFO:", _n)
    print(f"GWAS source used: {gwas_used}")
    print(f"GWAS: {gwas['n_snps_in_region']} catalog SNP(s) across {len(gwas['windows'])} window(s), "
          f"{len(gwas['by_pos'])} at candidate positions")

    # --- GTEx eQTL ---
    print("Collecting GTEx eQTLs...")
    tissues = list(L1["gtex_tissues"])
    eqtl = collect_eqtl_evidence(gene_meta, tissues, L1["gtex_dataset"], L1["gtex_gencode_version"],
                                 L1["gtex_items_per_page"], L1["gtex_max_pages"], fetch=fetch)
    if eqtl["resolve_error"]:
        RUN_WARNINGS.append(f"GTEx gene resolution failed: {eqtl['resolve_error']}")
    if eqtl["resolve_note"]:
        RUN_WARNINGS.append(f"GTEx gene resolution note: {eqtl['resolve_note']}")
    for t, info in eqtl["tissues"].items():
        if info["error"]:
            RUN_WARNINGS.append(f"GTEx tissue {t}: {info['error']} after {info['n_rows']} row(s)")
        elif info["n_rows"] == 0:
            print(f"INFO: GTEx tissue {t}: query succeeded but returned 0 significant eQTL rows for "
                  f"{gene_meta['gene_symbol']}; absence is not treated as a tested-null result.")
    print(f"GTEx: gencode_id={eqtl['gencode_id']}, " +
          ", ".join(f"{t}={i['n_rows']} rows" for t, i in eqtl["tissues"].items()))

    # --- Local Tables (MPRA & CRISPR) ---
    mpra_index = load_local_evidence_table(L1["mpra_evidence_path"], MPRA_COLUMNS, "MPRA", ("log2_allelic_skew",))
    crispr_index = load_local_evidence_table(L1["crispr_evidence_path"], CRISPR_COLUMNS, "CRISPR", ("effect_size",))
    for _lbl, _idx in (("MPRA", mpra_index), ("CRISPR", crispr_index)):
        if _idx is None:
            print(f"{_lbl}: no table configured")
        else:
            _cts = sorted({str(r.get("cell_type")) for rows in _idx.values() for r in rows})
            print(f"{_lbl}: table cell type(s) {_cts} (Layer 2 targets: {(CONFIG.get('layer2') or {}).get('target_cell_types')})")

    # --- Assemble All Records ---
    sig_p = L1["sig_pvalue_threshold"]
    all_records = (
        build_gwas_records(gwas, variants, L1["gwas_p_threshold"])
        + build_eqtl_records(eqtl, variants, tissues, maf_by_key, L1["eqtl_testable_min_maf"])
        + build_gnomad_records(gnomad, variants, L1["gnomad_dataset"])
        + build_local_table_records("mpra", variants, mpra_index, sig_p, mpra_value)
        + build_local_table_records("crispr", variants, crispr_index, sig_p, crispr_value)
    )
    evidence_df = records_to_dataframe(all_records)
    validate_layer1_output(evidence_df)
    validate_matrix_complete(evidence_df, variants)
    print(f"Total evidence records: {len(evidence_df)} (complete matrix for all {len(variants)} candidates x {len(MODALITIES)} modalities)")
    print("DATA QUALITY (Layer 1):", f"DEGRADED -- {len(RUN_WARNINGS)} query warning(s), listed after the save cell" if RUN_WARNINGS else "OK")

gnomAD: 1 window(s), 2569 variant(s) with genome AF
GWAS source used: api
GWAS: 16 catalog SNP(s) across 1 window(s), 3 at candidate positions
INFO: GTEx tissue Whole_Blood: query succeeded but returned 0 significant eQTL rows for HBB; absence is not treated as a tested-null result.
GTEx: gencode_id=ENSG00000244734.3, Whole_Blood=0 rows
MPRA: table cell type(s) ['HEL92.1.7'] (Layer 2 targets: ['HEL92.1.7'])
CRISPR: no table configured
Total evidence records: 560 (complete matrix for all 112 candidates x 5 modalities)
DATA QUALITY (Layer 1): OK


In [19]:
if evidence_df is not None:
    out_dir = os.path.join("data/outputs", GENE_SYMBOL)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "layer1_evidence.csv")
    evidence_df.to_csv(out_path, index=False)

    flag_counts = {m: {f: int(n) for f, n in g["missingness_flag"].value_counts().items()}
                   for m, g in evidence_df.groupby("modality")}

    manifest_path = write_manifest(out_dir, "layer1_manifest.json", {
        "layer": 1,
        "notebook_version": "layer1_v7",
        "gene_symbol": GENE_SYMBOL,
        "gene_metadata": gene_meta,
        "layer0_input": layer0_info,
        "config_path": CONFIG_PATH,
        "config_sha256": _file_hash(CONFIG_PATH),
        "layer1_effective_config": L1,
        "modality_meta": MODALITY_META,
        "n_candidate_variants": len(variants),
        "n_evidence_records": len(evidence_df),
        "flag_counts": flag_counts,
        "cache": {
            "enabled": L1["use_http_cache"],
            "cache_dir": L1["http_cache_dir"],
            "hits": http_cache.hits,
            "misses": http_cache.misses,
            "oldest_entry_utc": http_cache.oldest_entry_utc
        },
        "gtex": {"gencode_id": eqtl["gencode_id"], "dataset": L1["gtex_dataset"],
                 "tissues": {t: {"n_rows": i["n_rows"], "error": i["error"]} for t, i in eqtl["tissues"].items()}},
        "gnomad": {"dataset": L1["gnomad_dataset"], "windows": gnomad["windows"]},
        "gwas": {"windows": gwas["windows"], "n_snps_in_span": gwas["n_snps_in_region"]},
        "mpra_evidence_path": L1["mpra_evidence_path"], "mpra_sha256": _file_hash(L1["mpra_evidence_path"]),
        "crispr_evidence_path": L1["crispr_evidence_path"], "crispr_sha256": _file_hash(L1["crispr_evidence_path"]),
        "gwas_source_used": globals().get("gwas_used"),
        "gwas_bulk_file": globals().get("gwas_file_info"),
        "data_quality": "DEGRADED" if RUN_WARNINGS else "OK",
        "any_query_failed": bool(RUN_WARNINGS),
        "warnings": RUN_WARNINGS,
    })
    print(f"Saved {len(evidence_df)} records to {out_path}")
    print(f"Manifest: {manifest_path}")
    print(f"Cache performance: {http_cache.hits} hits, {http_cache.misses} misses (oldest entry: {http_cache.oldest_entry_utc})")
    if RUN_WARNINGS:
        print(f"\n{len(RUN_WARNINGS)} WARNING(S) -- affected variants are marked absent, not null:")
        for w in RUN_WARNINGS:
            print("  -", w)
else:
    print("No evidence DataFrame to save.")

Saved 560 records to data/outputs/HBB/layer1_evidence.csv
Manifest: data/outputs/HBB/layer1_manifest.json
Cache performance: 0 hits, 7 misses (oldest entry: 2026-11-10T12:25:05.277141+00:00)


In [20]:
if evidence_df is not None:
    display(evidence_df.groupby(["modality", "missingness_flag"]).size().unstack(fill_value=0))

missingness_flag,absent,observed_null,observed_positive
modality,,,
crispr,112,0,0
eqtl,112,0,0
gnomad_af,69,0,43
gwas,106,0,6
mpra,90,4,18


In [21]:
if evidence_df is not None:
    reasons = evidence_df.assign(reason=evidence_df["evidence_value"].map(lambda s: json.loads(s).get("reason", "(hit)")))
    display(reasons.groupby(["modality", "missingness_flag", "reason"]).size())

modality   missingness_flag   reason                                                         
crispr     absent             no_data_source_configured                                          112
eqtl       absent             no_significant_eqtl_association_returned_test_status_unverified      7
                              not_testable_rare_or_unknown_frequency                             105
gnomad_af  absent             not_observed_in_genome_callset                                      69
           observed_positive  (hit)                                                               43
gwas       absent             not_in_catalog                                                     106
           observed_positive  (hit)                                                                6
mpra       absent             not_tested_in_provided_table                                        90
           observed_null      tested_not_significant                                               4
           observed_positive  (hit)                                                               18
dtype: int64

## 6. Layer 1 regression tests


In [22]:
import tempfile


def _fake_fetch(routes):
    calls = []
    def fetch(method, url, params=None, json_body=None):
        calls.append((method, url, dict(params) if params else None))
        for match, respond in routes:
            if match(method, url, params, json_body):
                data, err = respond(method, url, params, json_body)
                return data, err, False
        return None, "http_404", False
    fetch.calls = calls
    return fetch


def _variants(rows):
    return to_variants(pd.DataFrame([{"chrom": "11", "pos": p, "ref": r, "alt": a} for p, r, a in rows]))


def _flag(recs, pos, context=None):
    return [r for r in recs if r.pos == pos and (context is None or r.context == context)]


def test_http_retry_and_transient_failures_not_cached():
    class FakeResponse:
        def __init__(self, status, payload=None):
            self.status_code, self._payload, self.headers = status, payload, {}
        def json(self): return self._payload
    class FakeSession:
        def __init__(self, statuses):
            self.statuses, self.headers, self.calls = list(statuses), {}, 0
        def request(self, *args, **kwargs):
            self.calls += 1
            status = self.statuses.pop(0) if self.statuses else 503
            return FakeResponse(status, {"ok": True} if status == 200 else None)
    original_session, original_sleep = requests.Session, time.sleep
    try:
        session_ok = FakeSession([503, 200])
        requests.Session = lambda: session_ok
        time.sleep = lambda *_args, **_kwargs: None
        with tempfile.TemporaryDirectory() as d:
            fetcher = make_http_fetch(HTTPCache(d, enabled=True), timeout=(1, 2), max_retries=2, backoff=0)
            data, err, cached = fetcher("GET", "https://example.test/retry")
            assert data == {"ok": True} and err is None and not cached and session_ok.calls == 2
            _, _, cached_again = fetcher("GET", "https://example.test/retry")
            assert cached_again and session_ok.calls == 2
        session_bad = FakeSession([503, 503, 503, 503])
        requests.Session = lambda: session_bad
        with tempfile.TemporaryDirectory() as d:
            fetcher = make_http_fetch(HTTPCache(d, enabled=True), timeout=(1, 2), max_retries=1, backoff=0)
            assert fetcher("GET", "https://example.test/transient")[1] == "http_503"
            assert fetcher("GET", "https://example.test/transient")[1] == "http_503"
            assert session_bad.calls == 4
        print("PASS: bounded HTTP retry, successful-response cache, transient-failure non-caching")
    finally:
        requests.Session, time.sleep = original_session, original_sleep


def test_hbb_metadata_fallback_provenance():
    def failed_fetch(method, url, params=None, json_body=None):
        return None, "ConnectTimeout", False
    meta = fetch_gene_metadata_fallback("HBB", failed_fetch)
    assert meta["gene_id"] == "ENSG00000244734" and meta["chrom"] == "11"
    assert (meta["start"], meta["end"], meta["strand"], meta["assembly_name"]) == (5225464, 5229395, -1, "GRCh38")
    assert meta["metadata_source"] == "validated_historical_fallback"
    try:
        fetch_gene_metadata_fallback("HBD", failed_fetch)
        raise AssertionError("Fallback must not be applied to a different gene.")
    except RuntimeError:
        pass
    print("PASS: HBB-only metadata fallback provenance and no cross-gene substitution")


def test_canon_key_and_gtex_ids():
    assert canon_key("chr11", 100, "A", "G") == ("11", 100, "A", "G")
    assert canon_key("11", 5227100, "G", "GA") == canon_key("11", 5227099, "AG", "AGA")
    assert parse_gtex_variant_id("chr11_5227002_T_C_b38") == ("11", 5227002, "T", "C")
    assert parse_gtex_variant_id("chr1_55516888_G_GA_b38") == ("1", 55516888, "G", "GA")
    assert parse_gtex_variant_id("nope") is None and parse_gtex_variant_id(None) is None
    print("PASS: canonical keys and GTEx variant-id parsing")


def test_http_cache_and_poisoning_prevention():
    with tempfile.TemporaryDirectory() as d:
        cache = HTTPCache(cache_dir=d, enabled=True, ttl_days=7)
        # 1. Successful response cached
        cache.put("GET", "https://api.test/data", {"a": "1"}, None, {"test": 123}, None)
        entry, hit = cache.get("GET", "https://api.test/data", {"a": "1"}, None)
        assert hit and entry.get("data") == {"test": 123}

        # 2. GraphQL response with errors must NOT be cached (poisoning prevention)
        cache.put("POST", "https://api.test/gql", None, {"q": 1}, {"errors": [{"msg": "fail"}]}, None)
        _, hit = cache.get("POST", "https://api.test/gql", None, {"q": 1})
        assert not hit, "GraphQL response with errors must not be cached"

        # 3. 404 response cached and retrieved
        cache.put("GET", "https://api.test/404", None, None, None, "http_404")
        entry, hit = cache.get("GET", "https://api.test/404", None, None)
        assert hit and entry.get("error") == "http_404"

        # 4. TTL expiration
        old_time = (datetime.now(timezone.utc) - timedelta(days=8)).isoformat()
        cache.put("GET", "https://api.test/old", None, None, {"old": True}, None)
        ckey = cache._cache_key("GET", "https://api.test/old", None, None)
        with open(os.path.join(d, f"{ckey}.json"), "w") as f:
            json.dump({"data": {"old": True}, "error": None, "fetched_at": old_time}, f)
        _, hit = cache.get("GET", "https://api.test/old", None, None)
        assert not hit, "Expired TTL must yield cache miss"
    print("PASS: HTTP cache storage, 404 caching, TTL expiration, and poisoning prevention")


def test_cache_config_toggle():
    with tempfile.TemporaryDirectory() as d:
        cache_off = HTTPCache(cache_dir=d, enabled=False)
        cache_off.put("GET", "https://api.test/x", None, None, {"x": 1}, None)
        assert len(os.listdir(d)) == 0, "Disabled cache must write 0 files"
    print("PASS: Cache config toggle respects enabled=False")



def test_gwas_multi_window_failure_isolation():
    variants = _variants([
        (100, "A", "G"),
        (50000, "C", "T"),
        (100000, "G", "A"),
    ])

    def fake_fetch(method, url, params=None, json_body=None):
        params = params or {}

        if url.endswith("/single-nucleotide-polymorphisms"):
            start = params["bp_start"]

            if start < 1000:
                # Window 1 fails.
                return None, "http_500", False

            elif start < 60000:
                # Window 2 succeeds but contains no SNPs.
                return {"_embedded": {"snps": []}}, None, False

            else:
                # Window 3 succeeds and contains a SNP.
                return {
                    "_embedded": {
                        "snps": [{
                            "rs_id": "rs3",
                            "locations": [{
                                "chromosome_name": "11",
                                "chromosome_position": 100000,
                            }],
                        }]
                    }
                }, None, False

        if url.endswith("/associations"):
            return {
                "_embedded": {
                    "associations": [{
                        "p_value": 1e-9,
                        "efo_traits": [{"efo_trait": "HbF"}],
                    }]
                }
            }, None, False

        return None, "http_404", False

    gwas = collect_gwas_evidence(
        "11",
        [100, 50000, 100000],
        fetch=fake_fetch,
        window_bp=25000,
    )

    recs = build_gwas_records(gwas, variants, 5e-8)

    r100 = [r for r in recs if r.pos == 100][0]
    r50k = [r for r in recs if r.pos == 50000][0]
    r100k = [r for r in recs if r.pos == 100000][0]

    assert (
        r100.missingness_flag == "absent"
        and json.loads(r100.evidence_value)["reason"]
        == "incomplete_region_query"
    )

    assert (
        r50k.missingness_flag == "absent"
        and json.loads(r50k.evidence_value)["reason"]
        == "not_in_catalog"
    )

    assert r100k.missingness_flag == "observed_positive"

    print("PASS: Multi-window GWAS failure isolated to failed window only")




def test_gwas_pagination_scope_and_semantics():
    def snp(rs, pos):
        return {
            "rs_id": rs,
            "locations": [{
                "chromosome_name": "11",
                "chromosome_position": pos,
            }],
        }

    def page1(m, u, p, b):
        return {
            "_embedded": {
                "snps": [
                    snp("rs1", 100),
                    snp("rs2", 999),
                ]
            },
            "_links": {
                "next": {
                    "href": "https://x/single-nucleotide-polymorphisms?page=1"
                }
            },
        }, None

    def page2(m, u, p, b):
        return {
            "_embedded": {
                "snps": [snp("rs3", 300)]
            },
            "_links": {},
        }, None

    def assoc(m, u, p, b):
        rs = (p or {}).get("rs_id")
        pv = {"rs1": 1e-9, "rs3": 1e-6}[rs]

        return {
            "_embedded": {
                "associations": [{
                    "p_value": pv,
                    "efo_traits": [{"efo_trait": "HbF, level"}],
                }]
            },
            "_links": {},
        }, None

    fetch = _fake_fetch([
        (
            lambda m, u, p, b:
                u.endswith("/single-nucleotide-polymorphisms"),
            page1,
        ),
        (
            lambda m, u, p, b:
                "single-nucleotide-polymorphisms?page=1" in u,
            page2,
        ),
        (
            lambda m, u, p, b:
                u.endswith("/associations"),
            assoc,
        ),
    ])

    res = collect_gwas_evidence(
        "11",
        [100, 300, 500],
        fetch=fetch,
        window_bp=25000,
    )

    assert set(res["by_pos"]) == {100, 300}, (
        "SNP on the second page must be found"
    )

    assoc_calls = [
        c for c in fetch.calls
        if c[1].endswith("/associations")
    ]

    assert len(assoc_calls) == 2, (
        "Associations should only be queried for candidate positions"
    )

    assert not any("{" in c[1] for c in fetch.calls), (
        "No templated URLs should be passed to fetch"
    )

    recs = build_gwas_records(
        res,
        _variants([
            (100, "A", "G"),
            (300, "C", "T"),
            (500, "G", "A"),
        ]),
        5e-8,
    )

    r100 = _flag(recs, 100)[0]
    r300 = _flag(recs, 300)[0]
    r500 = _flag(recs, 500)[0]

    assert (
        r100.missingness_flag == "observed_positive"
        and json.loads(r100.evidence_value)["traits"] == ["HbF, level"]
    )

    assert (
        r300.missingness_flag == "observed_null"
        and json.loads(r300.evidence_value)["reason"]
        == "subthreshold_association"
    )

    assert (
        r500.missingness_flag == "absent"
        and json.loads(r500.evidence_value)["reason"] == "not_in_catalog"
    )

    print(
        "PASS: GWAS pagination, windowing scope, positive / "
        "sub-threshold null / not-in-catalog absent"
    )


def test_eqtl_testability_and_failed_tissue():
    variants = _variants([(100, "A", "G"), (200, "C", "T"), (300, "G", "A"), (400, "T", "C")])
    eqtl = {"gencode_id": "ENSG1.1", "resolve_note": None, "resolve_error": None, "tissues": {
        "Whole_Blood": {"error": None, "n_rows": 1, "by_key": {
            canon_key("11", 100, "A", "G"): [{"slope": 0.4, "pvalue": 1e-8, "variant_id": "chr11_100_A_G_b38"}]}},
        "Liver": {"error": "http_500", "n_rows": 0, "by_key": {}}}}
    maf = {canon_key("11", 200, "C", "T"): 0.2, canon_key("11", 300, "G", "A"): 0.0001}
    recs = build_eqtl_records(eqtl, variants, ["Whole_Blood", "Liver"], maf, 0.01)
    assert _flag(recs, 100, "Whole_Blood")[0].missingness_flag == "observed_positive"
    common_nonhit = _flag(recs, 200, "Whole_Blood")[0]
    assert common_nonhit.missingness_flag == "absent"
    common_reason = json.loads(common_nonhit.evidence_value)
    assert common_reason["reason"] == "no_significant_eqtl_association_returned_test_status_unverified"
    assert common_reason["test_status"] == "unverified"
    assert _flag(recs, 300, "Whole_Blood")[0].missingness_flag == "absent"
    assert json.loads(_flag(recs, 300, "Whole_Blood")[0].evidence_value)["reason"].startswith("not_testable")
    assert _flag(recs, 400, "Whole_Blood")[0].missingness_flag == "absent"
    assert all(r.missingness_flag == "absent" for r in recs if r.context == "Liver")
    print("PASS: eQTL exact hit -> positive, common non-hit -> absent/unverified, rare/unknown -> absent, failed tissue -> absent")


def test_eqtl_pagination_truncation_and_stuck_pages():
    rows = [{"variantId": f"chr11_{i}_A_G_b38", "nes": 0.1, "pValue": 1e-6} for i in range(5)]

    def paged(m, u, p, b):
        chunk = rows[p["page"] * p["itemsPerPage"]:(p["page"] + 1) * p["itemsPerPage"]]
        return {"data": chunk, "paging_info": {"numberOfPages": 3, "page": p["page"]}}, None

    route = [(lambda m, u, p, b: "singleTissueEqtl" in u, paged)]
    got, err = fetch_tissue_eqtls("E.1", "Whole_Blood", "gtex_v8", _fake_fetch(route), 2, 10)
    assert err is None and len(got) == 5, "all pages must be collected"
    got, err = fetch_tissue_eqtls("E.1", "Whole_Blood", "gtex_v8", _fake_fetch(route), 2, 2)
    assert err and err.startswith("truncated"), "hitting the page cap must be reported"
    stuck = [(lambda m, u, p, b: True, lambda m, u, p, b: ({"data": rows[:2]}, None))]
    got, err = fetch_tissue_eqtls("E.1", "Whole_Blood", "gtex_v8", _fake_fetch(stuck), 2, 10)
    assert err == "pagination_not_advancing"
    print("PASS: GTEx pagination, truncation flag, non-advancing-page flag")


def test_gnomad_windows_af_and_delay_skipping():
    variants = _variants([(100, "A", "G"), (200, "C", "T"), (900000, "G", "A")])
    assert plan_windows([v.pos for v in variants], 25000) == [(100, 200), (900000, 900000)]

    def fake_cached_fetch(method, url, params=None, json_body=None):
        if json_body and json_body.get("variables", {}).get("start", 0) > 800000:
            return None, "http_503", False
        return {"data": {"region": {"variants": [
            {"variant_id": "11-100-A-G", "pos": 100, "ref": "A", "alt": "G", "genome": {"ac": 3, "an": 150000}},
            {"variant_id": "11-150-A-G", "pos": 150, "ref": "A", "alt": "G", "genome": None}]}}}, None, True  # from_cache=True

    t0 = time.time()
    res = collect_gnomad_evidence(variants, "gnomad_r4", 25000, delay_s=5.0, fetch=fake_cached_fetch)
    elapsed = time.time() - t0
    assert elapsed < 0.5, "gnomAD cache hits must skip rate-limit sleep"

    recs = {r.pos: r for r in build_gnomad_records(res, variants, "gnomad_r4")}
    assert recs[100].missingness_flag == "observed_positive"
    assert abs(json.loads(recs[100].evidence_value)["af"] - 3 / 150000) < 1e-12
    assert json.loads(recs[200].evidence_value)["reason"] == "not_observed_in_genome_callset"
    assert json.loads(recs[900000].evidence_value)["reason"] == "query_failed"
    print("PASS: gnomAD windows, AF calculation, failed window isolation, and cache delay skip")


def test_local_tables():
    variants = _variants([(100, "A", "G"), (200, "C", "T"), (300, "G", "A"), (400, "T", "C")])
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "mpra.csv")
        pd.DataFrame([
            {"chrom": "11", "pos": 100, "ref": "A", "alt": "G", "cell_type": "K562", "log2_allelic_skew": 1.8, "pvalue": 0.001, "dataset_name": "t"},
            {"chrom": "11", "pos": 200, "ref": "C", "alt": "T", "cell_type": "K562", "log2_allelic_skew": 0.1, "pvalue": 0.4, "dataset_name": "t"},
            {"chrom": "11", "pos": 300, "ref": "G", "alt": "A", "cell_type": "K562", "log2_allelic_skew": 0.1, "pvalue": None, "dataset_name": "t"},
        ]).to_csv(path, index=False)
        idx = load_local_evidence_table(path, MPRA_COLUMNS, "MPRA", ("log2_allelic_skew",))
        recs = build_local_table_records("mpra", variants, idx, 0.05, mpra_value)
        by = {r.pos: r for r in recs}
        assert by[100].missingness_flag == "observed_positive"
        assert by[200].missingness_flag == "observed_null"
        assert by[300].missingness_flag == "absent" and json.loads(by[300].evidence_value)["reason"] == "p_value_missing"
        assert by[400].missingness_flag == "absent" and json.loads(by[400].evidence_value)["reason"] == "not_tested_in_provided_table"

        bad = os.path.join(d, "bad.csv")
        pd.DataFrame([{"chrom": "11", "pos": 1}]).to_csv(bad, index=False)
        for fn, exc in ((lambda: load_local_evidence_table(bad, MPRA_COLUMNS, "MPRA"), ValueError),
                        (lambda: load_local_evidence_table(os.path.join(d, "nope.csv"), MPRA_COLUMNS, "MPRA"), FileNotFoundError)):
            try:
                fn()
                raise AssertionError("expected " + exc.__name__)
            except exc:
                pass
    none_recs = build_local_table_records("crispr", variants, None, 0.05, crispr_value)
    assert all(r.missingness_flag == "absent" and json.loads(r.evidence_value)["reason"] == "no_data_source_configured" for r in none_recs)
    print("PASS: local tables -- positive / tested-null / missing-p / not-in-table absent / schema enforcement")


def test_validators_and_json():
    variants = _variants([(100, "A", "G"), (200, "C", "T")])
    full = build_local_table_records("mpra", variants, None, 0.05, mpra_value)
    df = records_to_dataframe(full)
    try:
        validate_matrix_complete(df.iloc[:1], variants, modalities=("mpra",))
        raise AssertionError("expected incomplete-matrix error")
    except ValueError:
        pass
    validate_matrix_complete(df, variants, modalities=("mpra",))
    assert json.loads(ev(x=float("nan"), y=[1, float("inf")])) == {"x": None, "y": [1, None]}
    bad = df.copy()
    bad.loc[0, "source_type"] = "made_up"
    try:
        validate_layer1_output(bad)
        raise AssertionError("expected invalid source_type error")
    except ValueError:
        pass
    print("PASS: matrix-completeness validator, NaN-safe JSON, schema validation")


def test_layer0_discovery_and_strict_mode():
    old = os.getcwd()
    with tempfile.TemporaryDirectory() as d:
        os.chdir(d)
        try:
            os.makedirs("layer0_outputs")
            pd.DataFrame([{"chrom": 11, "pos": 5, "ref": "A", "alt": "G"}]).to_csv(
                "layer0_outputs/HBB_layer0_regulatory_truthset.csv", index=False)
            _, manifest, label = find_layer0_inputs("HBB")
            assert label == "layer-0 notebook layout" and manifest == "layer0_outputs/HBB_layer0_manifest.json"
            os.makedirs("data/outputs/HBB")
            pd.DataFrame([{"chrom": 11, "pos": 5, "ref": "A", "alt": "G"}]).to_csv(
                "data/outputs/HBB/layer0_regulatory_truthset.csv", index=False)
            path, _, label = find_layer0_inputs("HBB")
            assert label == "repo layout"
            df = read_candidate_csv(path)
            assert df["chrom"].iloc[0] == "11" and isinstance(df["chrom"].iloc[0], str)
            try:
                find_layer0_inputs("MISSING_GENE")
                raise AssertionError("expected FileNotFoundError")
            except FileNotFoundError:
                pass
        finally:
            os.chdir(old)
    print("PASS: Layer 0 discovery and string chrom dtypes")


test_http_retry_and_transient_failures_not_cached()
test_hbb_metadata_fallback_provenance()
test_canon_key_and_gtex_ids()
test_http_cache_and_poisoning_prevention()
test_cache_config_toggle()
test_gwas_multi_window_failure_isolation()
test_gwas_pagination_scope_and_semantics()
test_eqtl_testability_and_failed_tissue()
test_eqtl_pagination_truncation_and_stuck_pages()
test_gnomad_windows_af_and_delay_skipping()
test_local_tables()
test_validators_and_json()
test_layer0_discovery_and_strict_mode()
print("\nALL 11 REGRESSION AND SANITY CHECKS PASSED!")

PASS: bounded HTTP retry, successful-response cache, transient-failure non-caching
PASS: HBB-only metadata fallback provenance and no cross-gene substitution
PASS: canonical keys and GTEx variant-id parsing
PASS: HTTP cache storage, 404 caching, TTL expiration, and poisoning prevention
PASS: Cache config toggle respects enabled=False
PASS: Multi-window GWAS failure isolated to failed window only
PASS: GWAS pagination, windowing scope, positive / sub-threshold null / not-in-catalog absent
PASS: eQTL exact hit -> positive, common non-hit -> absent/unverified, rare/unknown -> absent, failed tissue -> absent
PASS: GTEx pagination, truncation flag, non-advancing-page flag
PASS: gnomAD windows, AF calculation, failed window isolation, and cache delay skip
PASS: local tables -- positive / tested-null / missing-p / not-in-table absent / schema enforcement
PASS: matrix-completeness validator, NaN-safe JSON, schema validation
PASS: Layer 0 discovery and string chrom dtypes

ALL 11 REGRESSION AND

## 7. Layer 2 configuration and logic


In [23]:
# Setup & Standard Library Imports
import os
import sys
import json
import hashlib
import subprocess
from datetime import datetime, timezone
from dataclasses import dataclass, asdict
from collections import defaultdict, Counter, namedtuple
from typing import Optional, List, Dict, Any, Tuple, Set

import pandas as pd
import numpy as np
import yaml

print(f"Python {sys.version}")
print(f"Pandas {pd.__version__}")


Python 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Pandas 2.2.3


In [24]:
CONFIG_PATH = "configs/hbb.yaml"

if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, "r") as f:
        config = yaml.safe_load(f)
    print(f"Loaded config from {CONFIG_PATH}: gene_symbol={config.get('gene_symbol')}")
else:
    config = {"gene_symbol": "HBB", "canonical_transcript": "ENST00000335295"}
    print(f"Config {CONFIG_PATH} not found; using fallback defaults.")

GENE_SYMBOL = config.get("gene_symbol", "HBB")

# AlphaGenome output modalities as described in the AlphaGenome paper (prose names, NOT API enum values).
# Layer 3 must bind each of these to the exact output-type enum of the installed client; until then every
# trigger carries alphagenome_binding_status = "unverified".
AG_OUTPUT_TYPES = ("gene_expression", "transcription_initiation", "chromatin_accessibility",
                   "histone_modification", "tf_binding", "contact_map", "splicing")

# Modality metadata defining completability, usage in fusion, and directional impact
MODALITY_META = {
    "gwas": {
        "kind": "association",
        "completable": False,
        "use_in_fusion": True,
        "direction": "supports_regulatory_relevance",
        "description": "GWAS Catalog association / fine-mapping",
    },
    "eqtl": {
        "kind": "expression_effect",
        "completable": True,
        "use_in_fusion": True,
        "direction": "supports_regulatory_relevance",
        "description": "GTEx expression QTL in target tissues",
        "alphagenome_output_types": ["gene_expression"],
        "proxy_quality": "direct",
        "receptive_field_bp": 1048576,   # requested context; Layer 3 must snap to a supported length
    },
    "mpra": {
        "kind": "reporter_activity",
        "completable": True,
        "use_in_fusion": True,
        "direction": "supports_regulatory_relevance",
        "description": "Massively Parallel Reporter Assay allelic skew",
        "alphagenome_output_types": ["chromatin_accessibility", "transcription_initiation"],
        "proxy_quality": "indirect_proxy",   # reporter allelic skew has no direct AlphaGenome track
        "receptive_field_bp": 2048,
    },
    "crispr": {
        "kind": "perturbation_effect",
        "completable": True,
        "use_in_fusion": True,
        "direction": "supports_regulatory_relevance",
        "description": "CRISPRi / flowFISH perturbation screen",
        "alphagenome_output_types": ["gene_expression"],
        "proxy_quality": "indirect_proxy",   # CRISPRi perturbs the element; the model scores the allele
        "receptive_field_bp": 1048576,
    },
    "gnomad_af": {
        "kind": "population_frequency",
        "completable": False,
        "use_in_fusion": False,
        "direction": "benign_leaning_when_observed",
        "description": "gnomAD genome allele frequency (annotation only)",
    },
}

# Layer 2 Execution Configuration
# Layer 1 stores its settings under `layer1:`; Layer 2 may override under `layer2:`.
_l1_cfg = config.get("layer1") or {}
_l2_cfg = config.get("layer2") or {}

L2 = {
    "layer1_csv_path": config.get("layer1_csv_path", None),
    "layer1_manifest_path": config.get("layer1_manifest_path", None),
    "layer0_csv_path": config.get("layer0_csv_path", None),

    # Target tissues for eQTL completion. Must match what Layer 1 actually queried, so default to
    # layer1.gtex_tissues (v1 read a top-level key Layer 1 never writes and fell back to Blood + Liver).
    "target_tissues": list(_l2_cfg.get("target_tissues") or _l1_cfg.get("gtex_tissues") or ["Whole_Blood"]),

    # Target cell types for MPRA / CRISPR completion (erythroid for HBB).
   "target_cell_types": list(
    _l2_cfg.get("target_cell_types") or ["HEL92.1.7"]
),

    # Completability mapping
    # Do not create AlphaGenome jobs for CRISPR when no CRISPR evidence source is configured.
    # It can be re-enabled automatically by configuring a real existing table in Layer 1.
    "completable_modalities": [
        m for m, meta in MODALITY_META.items()
        if meta.get("completable") and not (
            m == "crispr" and not (
                globals().get("L1", {}).get("crispr_evidence_path")
                and os.path.exists(globals().get("L1", {}).get("crispr_evidence_path"))
            )
        )
    ],

    # Stratification thresholds
    # Defaults below can be overridden under layer2.tier_thresholds in configs/hbb.yaml.
    "tier_thresholds": {**{
        "tier_1_min_modalities": 2,  # >= 2 allele-resolved modalities tested -> Tier 1 (Rich)
        "tier_2_modalities": 1,      # exactly 1 modality tested -> Tier 2 (Moderate)
        "tier_3_modalities": 0,      # 0 modalities tested -> Tier 3 (Sparse)
    }, **(_l2_cfg.get("tier_thresholds") or {})},

    # Stop before writing outputs when any evidence query failed (unknowns are not biological absence).
    "halt_on_query_failure": bool(_l2_cfg.get("halt_on_query_failure", True)),

    # Research runs fail closed if real Layer 1 output is missing; synthetic data is test-only.
    "strict": True,
}

if "crispr" not in L2["completable_modalities"]:
    print("CRISPR trigger jobs disabled: no configured, existing CRISPR evidence table.")
print(f"Completable modalities for AlphaGenome trigger: {L2['completable_modalities']}")
print(f"Target tissues for eQTL completion: {L2['target_tissues']}")
print(f"Target cell types for MPRA/CRISPR completion: {L2['target_cell_types']}")

Loaded config from configs/hbb.yaml: gene_symbol=HBB
CRISPR trigger jobs disabled: no configured, existing CRISPR evidence table.
Completable modalities for AlphaGenome trigger: ['eqtl', 'mpra']
Target tissues for eQTL completion: ['Whole_Blood']
Target cell types for MPRA/CRISPR completion: ['HEL92.1.7']


In [25]:
Variant = namedtuple("Variant", "chrom pos ref alt key")

def canon_key(chrom, pos, ref, alt):
    """Normalize chromosome (remove 'chr'), strip whitespace, uppercase alleles, and fully
    normalize indels by trimming a matching SUFFIX first, then a matching PREFIX, down to empty
    if needed -- matching the canon_key used (and verified) in the Layer 1 notebooks.

    v2 only trimmed a matching prefix and stopped once either allele reached length 1, which is
    weaker than Layer 1's: e.g. ref='CAT', alt='GAT' (a shared 'AT' suffix) was left completely
    unreduced in v2. This was dormant (not an observed bug) only because Layer 1 always echoes the
    candidate's own already-bcftools-normalized ref/alt into every record rather than a source's
    raw representation -- but two independently-maintained implementations of the same concept is
    exactly the kind of thing that silently drifts. Aligning them removes the risk entirely rather
    than relying on that upstream guarantee continuing to hold.
    """
    c = str(chrom).replace("chr", "").strip()
    p = int(pos)
    r = str(ref).strip().upper()
    a = str(alt).strip().upper()

    # Trim a matching suffix first, then a matching prefix (standard VCF normalization order).
    while r and a and r[-1] == a[-1]:
        r, a = r[:-1], a[:-1]
    while r and a and r[0] == a[0]:
        r, a, p = r[1:], a[1:], p + 1
    return (c, p, r, a)


def to_variants(df):
    """Convert a DataFrame with chrom, pos, ref, alt into a deduplicated list of Variant namedtuples."""
    variants = []
    seen = set()
    for row in df[["chrom", "pos", "ref", "alt"]].itertuples(index=False):
        key = canon_key(row.chrom, row.pos, row.ref, row.alt)
        if key not in seen:
            seen.add(key)
            variants.append(Variant(key[0], key[1], key[2], key[3], key))
    return sorted(variants, key=lambda v: (v.chrom, v.pos, v.ref, v.alt))


def _git_commit_hash():
    try:
        r = subprocess.run(
            ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True, timeout=5
        )
        return r.stdout.strip()
    except Exception:
        return None


def _file_hash(path, algo="sha256"):
    if not path or not os.path.exists(path):
        return None
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def write_manifest(out_dir, filename, payload):
    os.makedirs(out_dir, exist_ok=True)
    manifest_path = os.path.join(out_dir, filename)
    payload["generated_at_utc"] = datetime.now(timezone.utc).isoformat()
    payload["git_commit"] = _git_commit_hash()
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    return manifest_path


def resolve_out_dir(gene_symbol, is_synthetic):
    """Real runs write to data/outputs/<gene>/; synthetic demos are segregated so Layer 3 can never pick them up."""
    base = os.path.join("data/outputs", gene_symbol)
    return os.path.join(base, "synthetic_demo") if is_synthetic else base


def _norm_ctx(ctx):
    """pandas reads an empty CSV cell as float NaN (truthy); normalize None/NaN/'' to 'default'."""
    if ctx is None or (isinstance(ctx, float) and ctx != ctx) or str(ctx).strip() == "":
        return "default"
    return ctx

In [26]:
# Output schemas
LAYER2_AVAILABILITY_COLUMNS = [
    "chrom", "pos", "ref", "alt", "variant_key",
    "gwas_status", "eqtl_status", "mpra_status", "crispr_status", "gnomad_af_status",
    "n_positive", "n_null", "n_absent", "n_tested_experimental",
    "evidence_tier", "primary_gap_reasons"
]

LAYER2_TRIGGER_COLUMNS = [
    "trigger_id", "chrom", "pos", "ref", "alt", "variant_key",
    "modality", "context", "target_gene",
    "alphagenome_output_types", "alphagenome_proxy_quality", "alphagenome_binding_status",
    "receptive_field_bp", "trigger_reason", "source_evidence_value"
]

LAYER2_AGREEMENT_COLUMNS = [
    "check_id", "chrom", "pos", "ref", "alt", "variant_key",
    "modality", "context", "observed_flag", "observed_value",
    "validation_purpose"
]

EVIDENCE_TIERS = ("Tier 1 (Evidence-Rich)", "Tier 2 (Moderate Evidence)", "Tier 3 (Evidence-Sparse)")
MODALITY_STATUSES = ("observed_positive", "observed_null", "absent")


def validate_layer2_availability_output(df):
    missing = [c for c in LAYER2_AVAILABILITY_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Layer 2 availability DataFrame missing required column(s): {missing}")
    # v2 validated only 4 of the 5 status columns -- gnomad_af_status was silently unchecked.
    for col in ["gwas_status", "eqtl_status", "mpra_status", "crispr_status", "gnomad_af_status"]:
        invalid = set(df[col].dropna()) - set(MODALITY_STATUSES)
        if invalid:
            raise ValueError(f"Invalid status value(s) in {col}: {invalid}")
    invalid_tiers = set(df["evidence_tier"].dropna()) - set(EVIDENCE_TIERS)
    if invalid_tiers:
        raise ValueError(f"Invalid evidence tier value(s): {invalid_tiers}")


def validate_layer2_triggers_output(df):
    missing = [c for c in LAYER2_TRIGGER_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Layer 2 triggers DataFrame missing required column(s): {missing}")
    if not df.empty:
        non_completable = set(df["modality"]) - set(L2["completable_modalities"])
        if non_completable:
            raise ValueError(f"AlphaGenome triggers contain non-completable modality: {non_completable}")
        empty_types = df[df["alphagenome_output_types"].astype(str).str.strip() == ""]
        if not empty_types.empty:
            bad_mods = sorted(set(empty_types["modality"]))
            raise ValueError(f"AlphaGenome triggers for modality {bad_mods} have no alphagenome_output_types "
                             f"-- every completable modality in MODALITY_META must declare at least one.")
        bad_types = {t for s in df["alphagenome_output_types"] for t in str(s).split(";")} - set(AG_OUTPUT_TYPES)
        if bad_types:
            raise ValueError(f"AlphaGenome triggers contain unknown output type(s): {bad_types}")
        for s in df["source_evidence_value"]:
            if not isinstance(json.loads(s), dict):
                raise ValueError("source_evidence_value must serialize a JSON object (double-encoded?)")


def validate_layer2_agreement_output(df):
    missing = [c for c in LAYER2_AGREEMENT_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Layer 2 agreement checks DataFrame missing required column(s): {missing}")
    if not df.empty:
        invalid = set(df["observed_flag"]) - {"observed_positive", "observed_null"}
        if invalid:
            raise ValueError(f"Agreement check contains non-observed flag: {invalid}")


def validate_layer1_input(df):
    """Validate the Layer 1 evidence table BEFORE Layer 2 trusts it.

    v2 ingested the CSV unchecked. The risk isn't just a cryptic KeyError on a missing column: the
    reason classifier treats ANY flag that isn't exactly 'observed_positive' / 'observed_null' as
    absent, so a typo'd or corrupted flag (e.g. 'observed_postive') would be silently read as a gap
    and could queue an AlphaGenome job for a variant that actually had a hit.
    """
    required = ["chrom", "pos", "ref", "alt", "modality", "missingness_flag"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Layer 1 evidence is missing required column(s): {missing} "
                         f"(found: {sorted(df.columns)}). Was this produced by the Layer 1 notebook?")
    if df["missingness_flag"].isna().any():
        raise ValueError(f"Layer 1 evidence has {int(df['missingness_flag'].isna().sum())} row(s) with an empty "
                         f"missingness_flag; refusing to guess whether they are gaps.")
    bad_flags = set(df["missingness_flag"]) - set(MODALITY_STATUSES)
    if bad_flags:
        raise ValueError(f"Layer 1 evidence has invalid missingness_flag value(s): {sorted(bad_flags)} "
                         f"(allowed: {list(MODALITY_STATUSES)}). Refusing to treat them as gaps.")
    unknown_mods = sorted(set(df["modality"].dropna()) - set(MODALITY_META))
    if unknown_mods:
        print(f"NOTE: Layer 1 contains modality/modalities not in MODALITY_META: {unknown_mods}. "
              f"Layer 2 ignores them (not counted toward tiers, never completed).")

In [27]:
REASON_CATEGORIES = {
    # 1. True biological or dataset coverage gaps -> Trigger AlphaGenome
    "not_testable_rare_or_unknown_frequency": "GAP_BIOLOGICAL",
    "no_significant_eqtl_association_returned_test_status_unverified": "GAP_BIOLOGICAL",
    "not_tested_in_provided_table": "GAP_BIOLOGICAL",
    "not_in_catalog": "GAP_BIOLOGICAL",
    "not_observed_in_genome_callset": "GAP_BIOLOGICAL",   # gnomAD (non-completable; kept for completeness)
    "no_data_source_configured": "GAP_BIOLOGICAL",
    "no_usable_association_records": "GAP_BIOLOGICAL",
    "p_value_missing": "GAP_BIOLOGICAL",

    # 2. Tested null -> Experimental negative measurement (NEVER trigger AlphaGenome)
    "common_variant_not_significant": "OBSERVED_NULL",
    "tested_not_significant": "OBSERVED_NULL",
    "subthreshold_association": "OBSERVED_NULL",

    # 3. Technical / query failures -> Actionable transient warnings (NEVER trigger AlphaGenome)
    "query_failed": "ERROR_TRANSIENT",
    "incomplete_region_query": "ERROR_TRANSIENT",
    "incomplete_tissue_query": "ERROR_TRANSIENT",
}

# Categories that must NEVER produce an AlphaGenome trigger. ERROR_TRANSIENT means "ask Layer 1 to
# retry"; UNKNOWN_REASON means "a reason string was given that isn't in REASON_CATEGORIES above" --
# i.e. Layer 1's evidence vocabulary has drifted ahead of what Layer 2 knows how to interpret. v2
# silently treated any unrecognized reason as GAP_BIOLOGICAL (safe to complete), which is backwards:
# an unrecognized reason is precisely the situation where Layer 2 should NOT assume it's safe.
NEVER_COMPLETE_CATEGORIES = {"ERROR_TRANSIENT", "UNKNOWN_REASON"}


def parse_evidence_value(val):
    """Safely parse JSON evidence_value payload or return empty dict."""
    if isinstance(val, dict):
        return val
    if not isinstance(val, str) or not val.strip():
        return {}
    try:
        return json.loads(val)
    except Exception:
        return {"raw": str(val)}


def classify_evidence_reason(missingness_flag, reason=None, error=None):
    """Classify an evidence observation into one of:
    - POSITIVE (observed_positive hit)
    - NULL_EXPERIMENTAL (observed negative result)
    - GAP_BIOLOGICAL (true missingness eligible for computational fill)
    - ERROR_TRANSIENT (network/query failure requiring pipeline rerun)
    - UNKNOWN_REASON (a reason string was given but isn't in REASON_CATEGORIES -- treated
      conservatively, same as ERROR_TRANSIENT: never auto-completed, always surfaced as a warning)

    Note the deliberate distinction between reason=None (no reason field at all, e.g. a minimal
    test fixture -- treated as an ordinary gap) and reason="some_unrecognized_string" (a reason WAS
    supplied but doesn't match Layer 2's known vocabulary -- treated as a vocabulary-drift signal,
    not a gap). Real Layer 1 output always sets a reason for every absent record, so in practice
    this distinction only matters for genuinely malformed or newly-introduced reason strings.
    """
    if missingness_flag == "observed_positive":
        return "POSITIVE"
    if missingness_flag == "observed_null":
        return "NULL_EXPERIMENTAL"

    # If absent, inspect reason & error
    if error or reason in ("query_failed", "incomplete_region_query", "incomplete_tissue_query"):
        return "ERROR_TRANSIENT"
    if reason is None:
        return "GAP_BIOLOGICAL"
    if reason in REASON_CATEGORIES:
        return REASON_CATEGORIES[reason]
    return "UNKNOWN_REASON"


def aggregate_modality_status(records):
    """Given a list of EvidenceRecords for one (variant, modality) across multiple contexts:
    - Positive if ANY record is observed_positive
    - Null if no positive, but AT LEAST ONE record is observed_null
    - Absent if ALL records are absent
    Returns: (aggregate_status, primary_category, list_of_reasons, dict_by_context)
    """
    if not records:
        return "absent", "GAP_BIOLOGICAL", ["untested"], {}

    by_context = {}
    flags = [r["missingness_flag"] for r in records]

    reasons = []
    errors = []

    for r in records:
        ctx = _norm_ctx(r.get("context"))
        ev = parse_evidence_value(r.get("evidence_value"))
        reason = ev.get("reason")
        err = ev.get("error")
        cat = classify_evidence_reason(r["missingness_flag"], reason, err)
        by_context[ctx] = {
            "flag": r["missingness_flag"],
            "reason": reason,
            "category": cat,
            "error": err,
            "evidence_value": ev
        }
        if reason:
            reasons.append(reason)
        if err:
            errors.append(err)

    if "observed_positive" in flags:
        status = "observed_positive"
        category = "POSITIVE"
    elif "observed_null" in flags:
        status = "observed_null"
        category = "NULL_EXPERIMENTAL"
    else:
        status = "absent"
        context_cats = {c["category"] for c in by_context.values()}
        if errors or "ERROR_TRANSIENT" in context_cats:
            category = "ERROR_TRANSIENT"
        elif "UNKNOWN_REASON" in context_cats:
            category = "UNKNOWN_REASON"
        else:
            category = "GAP_BIOLOGICAL"

    return status, category, reasons, by_context


def filter_records_to_targets(records, target_tissues=None, target_cell_types=None):
    """Keep only evidence in scope for this gene: eQTL in target tissues, MPRA/CRISPR in target cell types.
    Records with no context (None / NaN / 'default') are kept -- they carry no-data-source reasons.
    Prevents e.g. a Liver eQTL hit from lifting an erythroid gene's availability status or tier."""

    target_tissues = L2["target_tissues"] if target_tissues is None else target_tissues
    target_cell_types = L2["target_cell_types"] if target_cell_types is None else target_cell_types

    kept = []
    for r in records:
        ctx = _norm_ctx(r.get("context"))
        mod = r["modality"]

        if ctx != "default":
            if mod == "eqtl" and ctx not in target_tissues:
                continue
            if mod in ("mpra", "crispr") and ctx not in target_cell_types:
                continue

        kept.append(r)

    return kept

In [28]:
def _trigger_row(counter, v, mod, ctx, gene_meta, reason, source_ev):
    meta = MODALITY_META.get(mod, {})
    return {
        "trigger_id": f"AG_TRIG_{counter:06d}",
        "chrom": v.chrom, "pos": v.pos, "ref": v.ref, "alt": v.alt,
        "variant_key": f"{v.chrom}:{v.pos}:{v.ref}:{v.alt}",
        "modality": mod,
        "context": ctx,
        "target_gene": gene_meta.get("gene_symbol", "UNKNOWN"),
        "alphagenome_output_types": ";".join(meta.get("alphagenome_output_types", [])),
        "alphagenome_proxy_quality": meta.get("proxy_quality", "unknown"),
        "alphagenome_binding_status": "unverified",     # Layer 3 must bind to real API enum values
        "receptive_field_bp": meta.get("receptive_field_bp", 1048576),
        "trigger_reason": reason,
        "source_evidence_value": json.dumps(source_ev if isinstance(source_ev, dict) else parse_evidence_value(source_ev)),
    }


def _queried_contexts(evidence_by_key, modality):
    """All non-default contexts actually queried for this modality, across every variant."""
    return {_norm_ctx(r.get("context")) for recs in evidence_by_key.values() for r in recs
           if r["modality"] == modality} - {"default"}


def compile_alphagenome_triggers(variants, evidence_by_key, completable_modalities, target_tissues,
                                 head_mapping, gene_meta, target_cell_types=None):
    """Scan all candidate variants and compile AlphaGenome execution tasks for missing completable modalities.
    Returns (triggers, warnings). Warnings cover transient query failures, unrecognized reason strings, and
    CONFIG mismatches; none of these are ever routed to AlphaGenome.

    eQTL, MPRA and CRISPR are all handled through the SAME per-context loop below (one iteration per
    target tissue/cell type, looked up via aggregate_modality_status's by_context dict) -- v2 only did
    this for eQTL and used one aggregate status across ALL target cell types for MPRA/CRISPR, which
    meant a single positive hit in one cell type (e.g. K562) silently suppressed completion for a
    SECOND, never-tested cell type (e.g. HUDEP2). With a single configured cell type (the v2 default)
    this bug was invisible; it was confirmed with two.
    """
    if target_cell_types is None:
        target_cell_types = L2["target_cell_types"]
    triggers, warnings_out, counter = [], [], 0

    # A target TISSUE that Layer 1 never queried at all is a config mismatch for eQTL specifically:
    # Layer 1's GTEx collector iterates a fixed tissues list per gene, so "never appears in the
    # evidence at all" genuinely means "Layer 2 and Layer 1 were configured with different tissue
    # lists." MPRA/CRISPR have no equivalent signal: Layer 1's local-table collector doesn't iterate
    # target_cell_types at all, it just checks whether a variant appears in whatever cell types the
    # supplied table happens to contain -- so "this cell type never appears in the evidence" for
    # MPRA/CRISPR is exactly the real gap (e.g. HUDEP2 configured as a target but not yet sourced),
    # not a config error. Suppressing it here would silently cancel out the per-context fix above, so
    # MPRA/CRISPR get every configured target cell type unconditionally, with no mismatch check.
    usable_contexts = {}
    for mod in completable_modalities:
        if mod == "eqtl":
            queried = _queried_contexts(evidence_by_key, "eqtl")
            usable = []
            for t in target_tissues:
                if queried and t not in queried:
                    warnings_out.append(f"CONFIG: eqtl target tissue '{t}' was not queried by Layer 1 "
                                        f"(queried: {sorted(queried)}); no AlphaGenome jobs queued for it.")
                else:
                    usable.append(t)
            usable_contexts[mod] = usable
        elif mod in ("mpra", "crispr"):
            usable_contexts[mod] = list(target_cell_types)
        else:
            usable_contexts[mod] = []

    fallback_reason = {"eqtl": "tissue_not_measured", "mpra": "cell_type_not_measured",
                       "crispr": "cell_type_not_measured"}

    for v in variants:
        in_scope = filter_records_to_targets(evidence_by_key.get(v.key, []), target_tissues, target_cell_types)
        by_modality = defaultdict(list)
        for r in in_scope:
            by_modality[r["modality"]].append(r)

        for mod in completable_modalities:
            records = by_modality.get(mod, [])
            _, _, _, by_context = aggregate_modality_status(records)

            for ctx in usable_contexts.get(mod, []):
                # A context-specific record wins; otherwise fall back to a context-less ("default")
                # record, which applies to every target context. Layer 1 emits exactly this shape for
                # MPRA/CRISPR gaps ("no_data_source_configured", "not_tested_in_provided_table"): they
                # aren't tied to any one cell type. Without this fallback the trigger would lose its
                # real reason and evidence payload (a regression v2 didn't have).
                rec = by_context.get(ctx, by_context.get("default"))
                if rec is not None and rec["flag"] != "absent":
                    continue                                   # observed (positive or null) -> never complete
                if rec is not None and rec["category"] in NEVER_COMPLETE_CATEGORIES:
                    kind = "query failure" if rec["category"] == "ERROR_TRANSIENT" else "unrecognized reason"
                    warnings_out.append(f"Variant {v.chrom}:{v.pos} {v.ref}>{v.alt} {mod} {ctx} had a "
                                        f"{kind}: {rec.get('error') or rec.get('reason')}. "
                                        f"Skipping AlphaGenome trigger.")
                    continue
                counter += 1
                triggers.append(_trigger_row(counter, v, mod, ctx, gene_meta,
                                             rec["reason"] if rec else fallback_reason.get(mod, "not_measured"),
                                             rec["evidence_value"] if rec else {}))
    return triggers, warnings_out


def compile_agreement_check_queue(variants, evidence_by_key, completable_modalities,
                                  target_tissues=None, target_cell_types=None):
    """Variants that HAVE in-scope experimental evidence on completable modalities -> separate validation queue.
    Used exclusively for model-vs-experiment benchmarking (never used in the fused score)."""
    checks, counter = [], 0
    for v in variants:
        for r in filter_records_to_targets(evidence_by_key.get(v.key, []), target_tissues, target_cell_types):
            mod, flag = r["modality"], r["missingness_flag"]
            if mod in completable_modalities and flag in ("observed_positive", "observed_null"):
                counter += 1
                checks.append({
                    "check_id": f"AG_AGREE_{counter:06d}",
                    "chrom": v.chrom, "pos": v.pos, "ref": v.ref, "alt": v.alt,
                    "variant_key": f"{v.chrom}:{v.pos}:{v.ref}:{v.alt}",
                    "modality": mod,
                    "context": _norm_ctx(r.get("context")),
                    "observed_flag": flag,
                    "observed_value": json.dumps(parse_evidence_value(r.get("evidence_value"))),
                    "validation_purpose": "internal_agreement_benchmark_only",
                })
    return checks

In [29]:
def compute_evidence_tiers(variants, evidence_by_key, clinvar_labels=None, regulatory_modalities=("eqtl", "mpra", "crispr")):
    """Assemble per-variant availability and tiers.

    Tier counts use allele-resolved experimental modalities (eQTL, MPRA, CRISPR).
    GWAS Catalog evidence is retained as locus/position-level context, but is not counted
    as allele-specific evidence because the Layer 1 match is positional, not allele-specific.
    """
    rows = []

    for v in variants:
        records_for_var = filter_records_to_targets(evidence_by_key.get(v.key, []))
        by_modality = defaultdict(list)
        for r in records_for_var:
            by_modality[r["modality"]].append(r)

        status_dict = {}
        all_reasons = []
        n_pos = 0
        n_null = 0
        n_abs = 0
        n_tested_experimental = 0

        for mod in ("gwas", "eqtl", "mpra", "crispr", "gnomad_af"):
            records = by_modality.get(mod, [])
            status, cat, reasons, _ = aggregate_modality_status(records)
            status_dict[f"{mod}_status"] = status

            if mod in regulatory_modalities:
                if status == "observed_positive":
                    n_pos += 1
                    n_tested_experimental += 1
                elif status == "observed_null":
                    n_null += 1
                    n_tested_experimental += 1
                else:
                    n_abs += 1

            if reasons and status == "absent":
                all_reasons.extend(reasons)

        # Tier assignment:
        # Tier 1 (Rich): >= 2 experimental regulatory modalities tested
        # Tier 2 (Moderate): exactly 1 experimental regulatory modality tested
        # Tier 3 (Sparse): 0 experimental regulatory modalities tested
        if n_tested_experimental >= L2["tier_thresholds"]["tier_1_min_modalities"]:
            tier = "Tier 1 (Evidence-Rich)"
        elif n_tested_experimental == L2["tier_thresholds"]["tier_2_modalities"]:
            tier = "Tier 2 (Moderate Evidence)"
        else:
            tier = "Tier 3 (Evidence-Sparse)"

        row = {
            "chrom": v.chrom,
            "pos": v.pos,
            "ref": v.ref,
            "alt": v.alt,
            "variant_key": f"{v.chrom}:{v.pos}:{v.ref}:{v.alt}",
            "gwas_status": status_dict.get("gwas_status", "absent"),
            "eqtl_status": status_dict.get("eqtl_status", "absent"),
            "mpra_status": status_dict.get("mpra_status", "absent"),
            "crispr_status": status_dict.get("crispr_status", "absent"),
            "gnomad_af_status": status_dict.get("gnomad_af_status", "absent"),
            "n_positive": n_pos,
            "n_null": n_null,
            "n_absent": n_abs,
            "n_tested_experimental": n_tested_experimental,
            "evidence_tier": tier,
            "primary_gap_reasons": ";".join(sorted(set(all_reasons))) if all_reasons else "none"
        }

        if clinvar_labels and v.key in clinvar_labels:
            cl = clinvar_labels[v.key]
            row["CLNSIG"] = cl.get("CLNSIG")
            row["label"] = cl.get("label")
            row["review_stars"] = cl.get("review_stars")

        rows.append(row)

    df = pd.DataFrame(rows)
    extra_cols = [c for c in ["CLNSIG", "label", "review_stars"] if clinvar_labels]
    for c in extra_cols:
        if c not in df.columns:          # labels loaded but none matched any variant
            df[c] = None
    return df[LAYER2_AVAILABILITY_COLUMNS + extra_cols]


def tier_balance_warnings(df, min_per_tier=5):
    """Layer 6 compares tiers; flag empty or tiny ones now rather than discovering it at calibration time."""
    out, counts = [], df["evidence_tier"].value_counts()
    for tier in EVIDENCE_TIERS:
        n = int(counts.get(tier, 0))
        if n == 0:
            out.append(f"{tier}: 0 variants -- any Layer 6 comparison involving this tier is impossible on this run.")
        elif n < min_per_tier:
            out.append(f"{tier}: only {n} variant(s) -- too few for stable per-tier calibration (want >= {min_per_tier}).")
    return out


def label_coverage_warnings(df):
    """Layer 6 needs ClinVar pathogenic AND benign examples inside each tier."""
    if "label" not in df.columns or df["label"].isna().all():
        return ["No ClinVar pathogenic/benign labels attached to any variant -- Layer 6 cannot calibrate. "
                "Check that Layer 0's CSV has a 'label' column and that its variant keys match Layer 1's."]
    out = []
    for tier, sub in df.groupby("evidence_tier"):
        lab = sub["label"].dropna()
        for cls in ("pathogenic", "benign"):
            if (lab == cls).sum() == 0:
                out.append(f"{tier}: no '{cls}' variants -- per-tier calibration needs both classes.")
    return out

### Layer 2 diagnostics: query-failure scan, context-scope audit, Tier 1 feasibility, data-quality gate


In [30]:
def scan_query_errors(evidence_df):
    """Aggregate technical query failures across EVERY modality (including non-completable GWAS/gnomAD).
    A failed query is an unknown, never a biological absence."""
    groups = defaultdict(lambda: defaultdict(int))
    for r in evidence_df.to_dict("records"):
        if r["missingness_flag"] != "absent":
            continue
        ev = parse_evidence_value(r.get("evidence_value"))
        reason, err, errs = ev.get("reason"), ev.get("error"), ev.get("errors")
        if err or errs or classify_evidence_reason("absent", reason, None) == "ERROR_TRANSIENT":
            label = str(err or (errs[0] if errs else None) or reason)
            groups[r["modality"]][label] += 1
    out = []
    for mod in sorted(groups):
        detail = ", ".join(f"{k} x{n}" for k, n in sorted(groups[mod].items()))
        out.append(f"QUERY FAILURE: {mod}: {sum(groups[mod].values())} record(s) failed ({detail}). "
                   f"These are unknowns, not biological absence: rerun Layer 1 with refresh_cache=True "
                   f"and check the endpoint before trusting tiers or triggers.")
    return out


def enforce_data_quality_gate(blocking_warnings, halt=True):
    """Return 'OK' or 'DEGRADED'; raise instead of silently producing degraded outputs when halt=True."""
    if not blocking_warnings:
        return "OK"
    if halt:
        raise RuntimeError(
            "Layer 2 halted: evidence queries failed, so tiers/triggers would be built on unknowns.\n  - "
            + "\n  - ".join(blocking_warnings)
            + "\nFix the failing source and rerun Layer 1, or set layer2.halt_on_query_failure: false in "
              "configs/hbb.yaml to continue with outputs explicitly marked DEGRADED.")
    return "DEGRADED"


def audit_context_scope(evidence_df, target_tissues, target_cell_types):
    """Report evidence that filter_records_to_targets() silently excludes (e.g. MPRA in K562 when the
    target is HEL92.1.7). Returns (warnings, per-modality context table)."""
    warnings_out, rows = [], []
    for mod, targets in (("eqtl", target_tissues), ("mpra", target_cell_types), ("crispr", target_cell_types)):
        sub = evidence_df[evidence_df["modality"] == mod]
        ctxs = Counter(_norm_ctx(c) for c in sub["context"])
        named = {c: n for c, n in ctxs.items() if c != "default"}
        inside = {c: n for c, n in named.items() if c in targets}
        outside = {c: n for c, n in named.items() if c not in targets}
        rows.append({"modality": mod, "targets": ";".join(targets),
                     "contexts_in_evidence": ";".join(f"{c}:{n}" for c, n in sorted(named.items())) or "(none)",
                     "records_in_scope": int(sum(inside.values())), "records_excluded": int(sum(outside.values()))})
        if outside:
            warnings_out.append(
                f"SCOPE: {mod} evidence in context(s) {sorted(outside)} is outside the configured target(s) "
                f"{list(targets)} and is EXCLUDED from tiers and triggers. If that is the wrong target, "
                f"fix layer2.target_{'tissues' if mod == 'eqtl' else 'cell_types'} in configs/hbb.yaml.")
        if mod in ("mpra", "crispr") and named and not inside:
            warnings_out.append(f"SCOPE: NONE of the {mod} table's cell types match the target "
                                f"{list(targets)} -- all {mod} evidence is being ignored.")
    return warnings_out, pd.DataFrame(rows)


def tier1_feasibility(availability_df, min_modalities=2):
    """Explain Tier 1 sparsity and list the variants a second allele-resolved source would promote."""
    allele_cols = [c for c in ("eqtl_status", "mpra_status", "crispr_status") if c in availability_df.columns]
    observed = availability_df[allele_cols].isin(["observed_positive", "observed_null"])
    n_obs = observed.sum(axis=1)
    cand = availability_df[(n_obs == min_modalities - 1) & (n_obs >= 1)].copy()
    if cand.empty:
        return pd.DataFrame(columns=["variant_key", "has_modalities", "needs_one_of"])
    cand["has_modalities"] = [";".join(c.replace("_status", "") for c in allele_cols if observed.loc[i, c])
                              for i in cand.index]
    cand["needs_one_of"] = [";".join(c.replace("_status", "") for c in allele_cols if not observed.loc[i, c])
                            for i in cand.index]
    keep = ["variant_key", "has_modalities", "needs_one_of"] + [c for c in ("label", "CLNSIG") if c in cand.columns]
    return cand[keep].reset_index(drop=True)


def print_tier1_feasibility(availability_df, candidates_df, min_modalities=2):
    allele_cols = [c for c in ("eqtl_status", "mpra_status", "crispr_status") if c in availability_df.columns]
    print("\n5. Tier 1 feasibility (needs >= %d allele-resolved modalities among eQTL / MPRA / CRISPR):" % min_modalities)
    for c in allele_cols:
        n = int(availability_df[c].isin(["observed_positive", "observed_null"]).sum())
        print(f"   {c.replace('_status', ''):7s}: {n} / {len(availability_df)} variants with an observed result")
    n_t1 = int((availability_df["evidence_tier"] == "Tier 1 (Evidence-Rich)").sum())
    print(f"   Tier 1 variants now: {n_t1}. Variants one extra source away from Tier 1: {len(candidates_df)}")
    if len(candidates_df):
        print("   -> adding a second allele-resolved table (another MPRA/MAVE dataset, a CRISPR screen, or a "
              "second eQTL resource) covering these variants would promote them; see "
              "layer2_tier1_candidates_needing_second_source.csv")
    elif n_t1 == 0:
        print("   -> no variant has even one observed allele-resolved result; no source can reach Tier 1 until "
              "at least two cover the same variants.")


In [31]:
def find_layer1_inputs(gene_symbol, explicit_csv=None, explicit_manifest=None):
    candidates = []
    if explicit_csv:
        candidates.append((explicit_csv, explicit_manifest or os.path.join(os.path.dirname(explicit_csv), "layer1_manifest.json"), "explicit config"))
    candidates.append((f"data/outputs/{gene_symbol}/layer1_evidence.csv", f"data/outputs/{gene_symbol}/layer1_manifest.json", "standard repo layout"))
    candidates.append((f"layer1_outputs/{gene_symbol}_layer1_evidence.csv", None, "layer1 notebook layout"))

    for csv_path, manifest_path, label in candidates:
        if os.path.exists(csv_path):
            return csv_path, manifest_path, label
    return None, None, None


def find_layer0_csv(gene_symbol, explicit_csv=None):
    candidates = [
        f"data/outputs/{gene_symbol}/layer0_regulatory_truthset.csv",
        f"layer0_outputs/{gene_symbol}_layer0_regulatory_truthset.csv"
    ]
    if explicit_csv:
        candidates.insert(0, explicit_csv)
    for p in candidates:
        if os.path.exists(p):
            return p
    return None


# Real Layer 1 reason strings per modality, used by the synthetic fallback below. v2's synthetic data
# (and one of its tests) used the placeholder "not_tested" for some records -- a string that is NOT part
# of Layer 1's actual vocabulary. v3 treats unrecognized reasons conservatively (see
# NEVER_COMPLETE_CATEGORIES), so the fixtures now use reasons Layer 1 would genuinely emit.
SYNTH_ABSENT_REASON = {
    "gwas": "not_in_catalog",
    "eqtl": "not_testable_rare_or_unknown_frequency",
    "gnomad_af": "not_observed_in_genome_callset",
    "mpra": "no_data_source_configured",
    "crispr": "no_data_source_configured",
}


def derive_fallback_gene_meta(variants, gene_symbol):
    """Minimal gene_meta used when no Layer 1 manifest is available. Raises a clear error instead of a
    bare IndexError when Layer 1's evidence CSV loaded but contained zero candidate variants."""
    if not variants:
        raise ValueError(f"Layer 1 evidence for {gene_symbol} loaded but produced zero candidate variants; "
                         f"nothing to analyse. Check that Layer 1 ran successfully and its output isn't empty.")
    return {"gene_symbol": gene_symbol, "chrom": str(variants[0].chrom)}


def generate_synthetic_hbb_evidence():
    """Generates a realistic synthetic Layer 1 evidence table for HBB regulatory candidates,
    covering:
    - 5' UTR promoter variants (common benign with eQTL + GWAS vs rare pathogenic with coverage gap)
    - Distal LCR enhancer variants (tested in MPRA/CRISPR vs untested)
    - Transient query error edge cases
    """
    print("Generating realistic synthetic HBB regulatory evidence matrix...")
    synthetic_vars = [
        # 1. 5' UTR promoter variant: common benign (eQTL null, GWAS subthreshold, gnomAD AF 0.12)
        ("11", 5227002, "T", "C", "Benign", "benign", 2),
        # 2. Promoter pathogenic: rare beta-thalassemia variant (rare/unknown AF -> eQTL absent, GWAS absent)
        ("11", 5227172, "G", "A", "Pathogenic", "pathogenic", 2),
        # 3. LCR cCRE distal enhancer variant: tested in MPRA (observed_positive allelic skew)
        ("11", 5280000, "A", "G", "Pathogenic", "pathogenic", 1),
        # 4. Promoter variant with tested null MPRA and null eQTL
        ("11", 5227250, "C", "T", "Benign", "benign", 1),
        # 5. Deep non-coding candidate: zero coverage across all experimental modalities (Tier 3 sparse)
        ("11", 5229000, "G", "T", "Uncertain_significance", "vus", 0),
        # 6. Variant affected by transient query failure on GWAS
        ("11", 5226500, "A", "C", "Likely_benign", "benign", 1),
    ]

    variants = [Variant(c, p, r, a, canon_key(c, p, r, a)) for c, p, r, a, _, _, _ in synthetic_vars]
    clinvar_labels = {canon_key(c, p, r, a): {"CLNSIG": sig, "label": lbl, "review_stars": stars}
                      for c, p, r, a, sig, lbl, stars in synthetic_vars}

    records = []
    # Var 1: eQTL null in blood, GWAS null, gnomAD AF 0.12
    v1 = variants[0]
    records.append({"chrom": v1.chrom, "pos": v1.pos, "ref": v1.ref, "alt": v1.alt, "modality": "eqtl",
                    "source_type": "experimental", "context": "Whole_Blood", "missingness_flag": "observed_null",
                    "evidence_value": json.dumps({"reason": "common_variant_not_significant", "maf": 0.12})})
    records.append({"chrom": v1.chrom, "pos": v1.pos, "ref": v1.ref, "alt": v1.alt, "modality": "eqtl",
                    "source_type": "experimental", "context": "Liver", "missingness_flag": "observed_null",
                    "evidence_value": json.dumps({"reason": "common_variant_not_significant", "maf": 0.12})})
    records.append({"chrom": v1.chrom, "pos": v1.pos, "ref": v1.ref, "alt": v1.alt, "modality": "gwas",
                    "source_type": "experimental", "context": "rs12345", "missingness_flag": "observed_null",
                    "evidence_value": json.dumps({"reason": "subthreshold_association", "best_pvalue": 0.04})})
    records.append({"chrom": v1.chrom, "pos": v1.pos, "ref": v1.ref, "alt": v1.alt, "modality": "gnomad_af",
                    "source_type": "curated_clinical", "context": "genome", "missingness_flag": "observed_positive",
                    "evidence_value": json.dumps({"af": 0.12})})
    records.append({"chrom": v1.chrom, "pos": v1.pos, "ref": v1.ref, "alt": v1.alt, "modality": "mpra",
                    "source_type": "experimental", "context": None, "missingness_flag": "absent",
                    "evidence_value": json.dumps({"reason": "not_tested_in_provided_table"})})
    records.append({"chrom": v1.chrom, "pos": v1.pos, "ref": v1.ref, "alt": v1.alt, "modality": "crispr",
                    "source_type": "experimental", "context": None, "missingness_flag": "absent",
                    "evidence_value": json.dumps({"reason": "no_data_source_configured"})})

    # Var 2: Pathogenic promoter - rare, absent everywhere except ClinVar
    v2 = variants[1]
    records.append({"chrom": v2.chrom, "pos": v2.pos, "ref": v2.ref, "alt": v2.alt, "modality": "eqtl",
                    "source_type": "experimental", "context": "Whole_Blood", "missingness_flag": "absent",
                    "evidence_value": json.dumps({"reason": "not_testable_rare_or_unknown_frequency", "maf": None})})
    records.append({"chrom": v2.chrom, "pos": v2.pos, "ref": v2.ref, "alt": v2.alt, "modality": "eqtl",
                    "source_type": "experimental", "context": "Liver", "missingness_flag": "absent",
                    "evidence_value": json.dumps({"reason": "not_testable_rare_or_unknown_frequency", "maf": None})})
    records.append({"chrom": v2.chrom, "pos": v2.pos, "ref": v2.ref, "alt": v2.alt, "modality": "gwas",
                    "source_type": "experimental", "context": None, "missingness_flag": "absent",
                    "evidence_value": json.dumps({"reason": "not_in_catalog"})})
    records.append({"chrom": v2.chrom, "pos": v2.pos, "ref": v2.ref, "alt": v2.alt, "modality": "gnomad_af",
                    "source_type": "curated_clinical", "context": "genome", "missingness_flag": "absent",
                    "evidence_value": json.dumps({"reason": "variant_not_found_in_gnomad"})})
    records.append({"chrom": v2.chrom, "pos": v2.pos, "ref": v2.ref, "alt": v2.alt, "modality": "mpra",
                    "source_type": "experimental", "context": None, "missingness_flag": "absent",
                    "evidence_value": json.dumps({"reason": "not_tested_in_provided_table"})})
    records.append({"chrom": v2.chrom, "pos": v2.pos, "ref": v2.ref, "alt": v2.alt, "modality": "crispr",
                    "source_type": "experimental", "context": None, "missingness_flag": "absent",
                    "evidence_value": json.dumps({"reason": "no_data_source_configured"})})

    # Var 3: LCR cCRE - MPRA positive hit, CRISPR hit
    v3 = variants[2]
    records.append({"chrom": v3.chrom, "pos": v3.pos, "ref": v3.ref, "alt": v3.alt, "modality": "eqtl",
                    "source_type": "experimental", "context": "Whole_Blood", "missingness_flag": "absent",
                    "evidence_value": json.dumps({"reason": "not_testable_rare_or_unknown_frequency"})})
    records.append({"chrom": v3.chrom, "pos": v3.pos, "ref": v3.ref, "alt": v3.alt, "modality": "eqtl",
                    "source_type": "experimental", "context": "Liver", "missingness_flag": "absent",
                    "evidence_value": json.dumps({"reason": "not_testable_rare_or_unknown_frequency"})})
    records.append({"chrom": v3.chrom, "pos": v3.pos, "ref": v3.ref, "alt": v3.alt, "modality": "gwas",
                    "source_type": "experimental", "context": "rs9999", "missingness_flag": "observed_positive",
                    "evidence_value": json.dumps({"traits": ["Red blood cell count"], "pvalue": 1e-9})})
    records.append({"chrom": v3.chrom, "pos": v3.pos, "ref": v3.ref, "alt": v3.alt, "modality": "gnomad_af",
                    "source_type": "curated_clinical", "context": "genome", "missingness_flag": "observed_positive",
                    "evidence_value": json.dumps({"af": 0.005})})
    records.append({"chrom": v3.chrom, "pos": v3.pos, "ref": v3.ref, "alt": v3.alt, "modality": "mpra",
                    "source_type": "experimental", "context": "K562", "missingness_flag": "observed_positive",
                    "evidence_value": json.dumps({"log2_allelic_skew": 1.85, "pvalue": 1e-6})})
    records.append({"chrom": v3.chrom, "pos": v3.pos, "ref": v3.ref, "alt": v3.alt, "modality": "crispr",
                    "source_type": "experimental", "context": "K562", "missingness_flag": "observed_positive",
                    "evidence_value": json.dumps({"effect_size": -0.74, "pvalue": 2e-5})})

    # Var 4: Tested null MPRA
    v4 = variants[3]
    records.append({"chrom": v4.chrom, "pos": v4.pos, "ref": v4.ref, "alt": v4.alt, "modality": "eqtl",
                    "source_type": "experimental", "context": "Whole_Blood", "missingness_flag": "observed_null",
                    "evidence_value": json.dumps({"reason": "common_variant_not_significant", "maf": 0.08})})
    records.append({"chrom": v4.chrom, "pos": v4.pos, "ref": v4.ref, "alt": v4.alt, "modality": "eqtl",
                    "source_type": "experimental", "context": "Liver", "missingness_flag": "observed_null",
                    "evidence_value": json.dumps({"reason": "common_variant_not_significant", "maf": 0.08})})
    records.append({"chrom": v4.chrom, "pos": v4.pos, "ref": v4.ref, "alt": v4.alt, "modality": "gwas",
                    "source_type": "experimental", "context": None, "missingness_flag": "absent",
                    "evidence_value": json.dumps({"reason": "not_in_catalog"})})
    records.append({"chrom": v4.chrom, "pos": v4.pos, "ref": v4.ref, "alt": v4.alt, "modality": "gnomad_af",
                    "source_type": "curated_clinical", "context": "genome", "missingness_flag": "observed_positive",
                    "evidence_value": json.dumps({"af": 0.08})})
    records.append({"chrom": v4.chrom, "pos": v4.pos, "ref": v4.ref, "alt": v4.alt, "modality": "mpra",
                    "source_type": "experimental", "context": "K562", "missingness_flag": "observed_null",
                    "evidence_value": json.dumps({"reason": "tested_not_significant", "pvalue": 0.42})})
    records.append({"chrom": v4.chrom, "pos": v4.pos, "ref": v4.ref, "alt": v4.alt, "modality": "crispr",
                    "source_type": "experimental", "context": None, "missingness_flag": "absent",
                    "evidence_value": json.dumps({"reason": "no_data_source_configured"})})

    # Var 5: VUS - completely absent
    v5 = variants[4]
    for m in ("eqtl", "gwas", "gnomad_af", "mpra", "crispr"):
        records.append({"chrom": v5.chrom, "pos": v5.pos, "ref": v5.ref, "alt": v5.alt, "modality": m,
                        "source_type": "experimental" if m != "gnomad_af" else "curated_clinical",
                        "context": None, "missingness_flag": "absent",
                        "evidence_value": json.dumps({"reason": SYNTH_ABSENT_REASON[m]})})

    # Var 6: Transient query failure on GWAS
    v6 = variants[5]
    records.append({"chrom": v6.chrom, "pos": v6.pos, "ref": v6.ref, "alt": v6.alt, "modality": "gwas",
                    "source_type": "experimental", "context": None, "missingness_flag": "absent",
                    "evidence_value": json.dumps({"reason": "query_failed", "error": "HTTP 504 Gateway Timeout"})})
    for m in ("eqtl", "gnomad_af", "mpra", "crispr"):
        records.append({"chrom": v6.chrom, "pos": v6.pos, "ref": v6.ref, "alt": v6.alt, "modality": m,
                        "source_type": "experimental" if m != "gnomad_af" else "curated_clinical",
                        "context": None, "missingness_flag": "absent",
                        "evidence_value": json.dumps({"reason": SYNTH_ABSENT_REASON[m]})})

    gene_meta = {"gene_symbol": "HBB", "gene_id": "ENSG00000244734", "chrom": "11",
                 "start": 5225464, "end": 5229395, "strand": -1}
    info = {"csv_path": "synthetic_in_memory", "csv_sha256": None, "layout": "synthetic fallback", "manifest_path": None}

    return pd.DataFrame(records), variants, clinvar_labels, gene_meta, info


# --- Execution of Intake ---
l1_csv, l1_manifest, layout = find_layer1_inputs(GENE_SYMBOL, L2["layer1_csv_path"], L2["layer1_manifest_path"])
l0_csv = find_layer0_csv(GENE_SYMBOL, L2["layer0_csv_path"])

clinvar_labels = {}
if l0_csv:
    l0_df = pd.read_csv(l0_csv, dtype={"chrom": str})
    for row in l0_df.to_dict("records"):
        k = canon_key(row["chrom"], row["pos"], row["ref"], row["alt"])
        clinvar_labels[k] = {
            "CLNSIG": row.get("CLNSIG"),
            "label": row.get("label"),
            "review_stars": row.get("review_stars")
        }
    print(f"Loaded {len(clinvar_labels)} variant annotations from Layer 0 ({l0_csv})")

layer1_manifest = None
if l1_csv:
    print(f"Loading live Layer 1 evidence from {l1_csv} ({layout})...")
    evidence_df = pd.read_csv(l1_csv, dtype={"chrom": str})
    validate_layer1_input(evidence_df)
    IS_SYNTHETIC = False
    variants = to_variants(evidence_df)

    gene_meta = derive_fallback_gene_meta(variants, GENE_SYMBOL)
    if l1_manifest and os.path.exists(l1_manifest):
        with open(l1_manifest) as f:
            m = json.load(f)
        gene_meta = m.get("gene_metadata", gene_meta)
        layer1_manifest = m
    layer1_info = {"csv_path": l1_csv, "csv_sha256": _file_hash(l1_csv), "layout": layout, "manifest_path": l1_manifest}
    print(f"Loaded {len(evidence_df)} evidence records across {len(variants)} candidates.")
else:
    raise FileNotFoundError(
        f"Real Layer 1 evidence CSV not found for {GENE_SYMBOL}. Run the Layer 1 collection and save its output first. "
        "Synthetic fixtures are restricted to tests and are not allowed in research output tables."
    )


Loaded 112 variant annotations from Layer 0 (layer0_outputs/HBB_layer0_regulatory_truthset.csv)
Loading live Layer 1 evidence from data/outputs/HBB/layer1_evidence.csv (standard repo layout)...
Loaded 560 evidence records across 112 candidates.


## 8. Layer 2 assembly, outputs, and manifest


In [32]:
# Index evidence records by canonical variant key
evidence_by_key = defaultdict(list)
for rec in evidence_df.to_dict("records"):
    k = canon_key(rec["chrom"], rec["pos"], rec["ref"], rec["alt"])
    evidence_by_key[k].append(rec)

print(f"Indexed evidence for {len(evidence_by_key)} unique variant keys.")

# 0. Context-scope audit: evidence outside the configured target tissues / cell types is excluded from tiers and triggers.
scope_warnings, scope_table = audit_context_scope(evidence_df, L2["target_tissues"], L2["target_cell_types"])
display(scope_table)

# 1. Modality Availability & Stratification
availability_df = compute_evidence_tiers(variants, evidence_by_key, clinvar_labels=clinvar_labels)
validate_layer2_availability_output(availability_df)

# 2. AlphaGenome Trigger Queue
triggers, transient_warnings = compile_alphagenome_triggers(
    variants=variants,
    evidence_by_key=evidence_by_key,
    completable_modalities=L2["completable_modalities"],
    target_tissues=L2["target_tissues"],
    head_mapping=MODALITY_META,
    gene_meta=gene_meta
)
triggers_df = pd.DataFrame(triggers)
validate_layer2_triggers_output(triggers_df)

# 3. Agreement Check Queue
agreements = compile_agreement_check_queue(
    variants=variants,
    evidence_by_key=evidence_by_key,
    completable_modalities=L2["completable_modalities"]
)
agreement_df = pd.DataFrame(agreements)
validate_layer2_agreement_output(agreement_df)

tier_warnings = tier_balance_warnings(availability_df)
label_warnings = label_coverage_warnings(availability_df)

# Query failures in ANY modality (including non-completable GWAS / gnomAD) plus Layer 1's own warnings.
layer1_manifest_warnings = [f"Layer 1 manifest: {w}" for w in ((layer1_manifest or {}).get("warnings") or [])]
query_failure_warnings = scan_query_errors(evidence_df)
blocking_warnings = query_failure_warnings + layer1_manifest_warnings
transient_warnings = list(transient_warnings) + blocking_warnings + scope_warnings

tier1_candidates_df = tier1_feasibility(availability_df, L2["tier_thresholds"]["tier_1_min_modalities"])

print(f"\nLayer 2 Assembly Complete:")
print(f" - Modality Availability Matrix: {len(availability_df)} variants")
print(f" - AlphaGenome Trigger Queue:    {len(triggers_df)} inference jobs")
print(f" - Agreement Check Queue:        {len(agreement_df)} validation benchmarks")
print(f" - Pipeline Warnings (query/config): {len(transient_warnings)}")
print(f" - Tier Balance Warnings:        {len(tier_warnings)}")
print(f" - Label Coverage Warnings:      {len(label_warnings)}")


Indexed evidence for 112 unique variant keys.


,modality,targets,contexts_in_evidence,records_in_scope,records_excluded
0,eqtl,Whole_Blood,Whole_Blood:112,112,0
1,mpra,HEL92.1.7,HEL92.1.7:22,22,0
2,crispr,HEL92.1.7,(none),0,0



Layer 2 Assembly Complete:
 - Modality Availability Matrix: 112 variants
 - AlphaGenome Trigger Queue:    202 inference jobs
 - Agreement Check Queue:        22 validation benchmarks
 - Pipeline Warnings (query/config): 0
 - Tier Balance Warnings:        1
 - Label Coverage Warnings:      1


In [33]:
DATA_QUALITY = enforce_data_quality_gate(blocking_warnings, halt=L2["halt_on_query_failure"])
print(f"DATA QUALITY: {DATA_QUALITY}")
if DATA_QUALITY != "OK":
    print("!! Outputs below are marked DEGRADED in the manifest. Do not feed them to Layer 3 until the failures are fixed.")
for _w in scope_warnings:
    print("WARNING:", _w)


DATA QUALITY: OK


In [34]:
out_dir = resolve_out_dir(GENE_SYMBOL, IS_SYNTHETIC)
os.makedirs(out_dir, exist_ok=True)

avail_path = os.path.join(out_dir, "layer2_modality_availability.csv")
triggers_path = os.path.join(out_dir, "layer2_alphagenome_triggers.csv")
agreement_path = os.path.join(out_dir, "layer2_agreement_checks.csv")

availability_df.to_csv(avail_path, index=False)
triggers_df.to_csv(triggers_path, index=False)
agreement_df.to_csv(agreement_path, index=False)
tier1_path = os.path.join(out_dir, "layer2_tier1_candidates_needing_second_source.csv")
tier1_candidates_df.to_csv(tier1_path, index=False)

tier_counts = {k: int(v) for k, v in availability_df["evidence_tier"].value_counts().items()}
trigger_by_modality = {k: int(v) for k, v in triggers_df["modality"].value_counts().items()} if not triggers_df.empty else {}
trigger_by_context = {f"{m}|{c}": int(n) for (m, c), n in triggers_df.groupby(["modality", "context"]).size().items()} if not triggers_df.empty else {}
trigger_by_output_type = {}
if not triggers_df.empty:
    for s in triggers_df["alphagenome_output_types"]:
        for t in str(s).split(";"):
            trigger_by_output_type[t] = trigger_by_output_type.get(t, 0) + 1

manifest_payload = {
    "layer": 2,
    "notebook_version": "layer2_v3",
    "is_synthetic": bool(IS_SYNTHETIC),        # Layer 3 MUST refuse manifests where this is true
    "gene_symbol": GENE_SYMBOL,
    "gene_metadata": gene_meta,
    "layer1_input": layer1_info,
    "config_path": CONFIG_PATH,
    "config_sha256": _file_hash(CONFIG_PATH),
    "layer2_effective_config": L2,
    "modality_meta": MODALITY_META,
    "n_candidate_variants": len(variants),
    "tier_distribution": tier_counts,
    "alphagenome_triggers": {
        "total_jobs": len(triggers_df),
        "by_modality": trigger_by_modality,
        "by_context": trigger_by_context,
        "by_output_type": trigger_by_output_type,
        "binding_status": "unverified: Layer 3 must map output types to the real AlphaGenome API enums",
    },
    "agreement_checks": {
        "total_checks": len(agreement_df)
    },
    "data_quality": DATA_QUALITY,               # Layer 3 should refuse DEGRADED manifests
    "blocking_warnings": blocking_warnings,
    "scope_warnings": scope_warnings,
    "tier1_candidates_needing_second_source": int(len(tier1_candidates_df)),
    "pipeline_warnings_count": len(transient_warnings),
    "pipeline_warnings": transient_warnings,
    "tier_warnings": tier_warnings,
    "label_warnings": label_warnings,
}

manifest_path = write_manifest(out_dir, "layer2_manifest.json", manifest_payload)

print(f"Saved Modality Availability to: {avail_path}")
print(f"Saved AlphaGenome Triggers to:  {triggers_path}")
print(f"Saved Agreement Checks to:      {agreement_path}")
print(f"Saved Tier-1 candidates to:     {tier1_path}")
print(f"Saved Manifest to:              {manifest_path}")
print(f"DATA QUALITY: {DATA_QUALITY}")
if IS_SYNTHETIC:
    print("\n!! These outputs are SYNTHETIC (demo mode) and were written to synthetic_demo/ -- not real results.")

Saved Modality Availability to: data/outputs/HBB/layer2_modality_availability.csv
Saved AlphaGenome Triggers to:  data/outputs/HBB/layer2_alphagenome_triggers.csv
Saved Agreement Checks to:      data/outputs/HBB/layer2_agreement_checks.csv
Saved Tier-1 candidates to:     data/outputs/HBB/layer2_tier1_candidates_needing_second_source.csv
Saved Manifest to:              data/outputs/HBB/layer2_manifest.json
DATA QUALITY: OK


In [35]:
print("=" * 70)
print("LAYER 2 SUMMARY: EVIDENCE AVAILABILITY & ALPHAGENOME WORKLOAD")
print(f"DATA QUALITY: {DATA_QUALITY}")
if IS_SYNTHETIC:
    print("*** SYNTHETIC DEMO DATA -- NOT REAL RESULTS ***")
print("=" * 70)

print("\n1. Candidate Variant Stratification Tiers:")
tier_summary = availability_df["evidence_tier"].value_counts().rename_axis("Tier").reset_index(name="Count")
tier_summary["Percentage"] = (tier_summary["Count"] / len(availability_df) * 100).round(1).astype(str) + "%"
display(tier_summary)

print("\n2. Modality Availability Status Breakdown (Variants):")
status_summary = []
for mod in ["gwas", "eqtl", "mpra", "crispr", "gnomad_af"]:
    col = f"{mod}_status"
    c = Counter(availability_df[col])
    status_summary.append({
        "Modality": mod,
        "Observed Positive": c.get("observed_positive", 0),
        "Observed Null": c.get("observed_null", 0),
        "Absent": c.get("absent", 0)
    })
display(pd.DataFrame(status_summary))

print("\n3. ClinVar Label Distribution Across Evidence Tiers:")
if "label" in availability_df.columns and availability_df["label"].notna().any():
    crosstab = pd.crosstab(availability_df["evidence_tier"], availability_df["label"], margins=True)
    display(crosstab)
else:
    print("  (no labels attached -- see label warnings below)")

print("\n4. AlphaGenome Trigger Queue by Modality & Context:")
if not triggers_df.empty:
    trig_summary = triggers_df.groupby(["modality", "context", "alphagenome_output_types", "alphagenome_proxy_quality"]).size().reset_index(name="Job Count")
    display(trig_summary)
else:
    print("No AlphaGenome triggers generated.")

if transient_warnings:
    print(f"\nWARNING: {len(transient_warnings)} pipeline warning(s) (query failures need a Layer 1 retry; CONFIG: needs a config fix):")
    for w in transient_warnings:
        print(f"  - {w}")
if tier_warnings:
    print(f"\nWARNING: {len(tier_warnings)} tier-balance warning(s) -- Layer 6 stratification will be affected:")
    for w in tier_warnings:
        print(f"  - {w}")
if label_warnings:
    print(f"\nWARNING: {len(label_warnings)} label-coverage warning(s):")
    for w in label_warnings:
        print(f"  - {w}")

print_tier1_feasibility(availability_df, tier1_candidates_df, L2["tier_thresholds"]["tier_1_min_modalities"])


LAYER 2 SUMMARY: EVIDENCE AVAILABILITY & ALPHAGENOME WORKLOAD
DATA QUALITY: OK

1. Candidate Variant Stratification Tiers:


,Tier,Count,Percentage
0,Tier 3 (Evidence-Sparse),90,80.4%
1,Tier 2 (Moderate Evidence),22,19.6%



2. Modality Availability Status Breakdown (Variants):


,Modality,Observed Positive,Observed Null,Absent
0,gwas,6,0,106
1,eqtl,0,0,112
2,mpra,18,4,90
3,crispr,0,0,112
4,gnomad_af,43,0,69



3. ClinVar Label Distribution Across Evidence Tiers:


label,benign,pathogenic,All
evidence_tier,,,
Tier 2 (Moderate Evidence),0,22,22
Tier 3 (Evidence-Sparse),23,67,90
All,23,89,112



4. AlphaGenome Trigger Queue by Modality & Context:


,modality,context,alphagenome_output_types,alphagenome_proxy_quality,Job Count
0,eqtl,Whole_Blood,gene_expression,direct,112
1,mpra,HEL92.1.7,chromatin_accessibility;transcription_initiation,indirect_proxy,90



  - Tier 1 (Evidence-Rich): 0 variants -- any Layer 6 comparison involving this tier is impossible on this run.

  - Tier 2 (Moderate Evidence): no 'benign' variants -- per-tier calibration needs both classes.

5. Tier 1 feasibility (needs >= 2 allele-resolved modalities among eQTL / MPRA / CRISPR):
   eqtl   : 0 / 112 variants with an observed result
   mpra   : 22 / 112 variants with an observed result
   crispr : 0 / 112 variants with an observed result
   Tier 1 variants now: 0. Variants one extra source away from Tier 1: 22
   -> adding a second allele-resolved table (another MPRA/MAVE dataset, a CRISPR screen, or a second eQTL resource) covering these variants would promote them; see layer2_tier1_candidates_needing_second_source.csv


In [36]:
def test_multi_context_aggregation_order():
    """Observed positive in ANY context must override null or absent."""
    recs = [
        {"missingness_flag": "observed_null", "context": "Whole_Blood", "evidence_value": "{}"},
        {"missingness_flag": "observed_positive", "context": "Liver", "evidence_value": "{}"},
        {"missingness_flag": "absent", "context": "Brain", "evidence_value": "{}"},
    ]
    status, cat, _, _ = aggregate_modality_status(recs)
    assert status == "observed_positive", f"Expected positive, got {status}"
    assert cat == "POSITIVE"

    # Null must override absent
    recs_null = [
        {"missingness_flag": "observed_null", "context": "Whole_Blood", "evidence_value": "{}"},
        {"missingness_flag": "absent", "context": "Liver", "evidence_value": "{}"},
    ]
    status_null, cat_null, _, _ = aggregate_modality_status(recs_null)
    assert status_null == "observed_null", f"Expected null, got {status_null}"
    assert cat_null == "NULL_EXPERIMENTAL"
    print("PASS: multi-context aggregation order (positive > null > absent)")


def test_transient_error_handling():
    """Query failures must NOT trigger AlphaGenome and must be flagged as warnings."""
    v = Variant("11", 100, "A", "G", canon_key("11", 100, "A", "G"))
    ev_by_key = {
        v.key: [{
            "modality": "eqtl",
            "context": "Whole_Blood",
            "missingness_flag": "absent",
            "evidence_value": json.dumps({"reason": "query_failed", "error": "HTTP 500 Server Error"})
        }]
    }
    trigs, warns = compile_alphagenome_triggers(
        variants=[v],
        evidence_by_key=ev_by_key,
        completable_modalities=["eqtl"],
        target_tissues=["Whole_Blood"],
        head_mapping=MODALITY_META,
        gene_meta={"gene_symbol": "TEST"}
    )
    assert len(trigs) == 0, "Transient query failure must not generate an AlphaGenome trigger!"
    assert len(warns) == 1, "Transient query failure must be recorded in warnings!"
    print("PASS: transient query errors are flagged and isolated from AlphaGenome triggers")


def test_tested_nulls_never_trigger_alphagenome():
    """Observed negative results (null) must never be filled by AlphaGenome."""
    v = Variant("11", 200, "C", "T", canon_key("11", 200, "C", "T"))
    ev_by_key = {
        v.key: [{
            "modality": "eqtl",
            "context": "Whole_Blood",
            "missingness_flag": "observed_null",
            "evidence_value": json.dumps({"reason": "common_variant_not_significant", "maf": 0.25})
        }]
    }
    trigs, _ = compile_alphagenome_triggers(
        variants=[v],
        evidence_by_key=ev_by_key,
        completable_modalities=["eqtl"],
        target_tissues=["Whole_Blood"],
        head_mapping=MODALITY_META,
        gene_meta={"gene_symbol": "TEST"}
    )
    assert len(trigs) == 0, "Tested null must NEVER trigger AlphaGenome!"
    print("PASS: observed negative results (tested nulls) never trigger AlphaGenome")


def test_non_completable_modalities():
    """GWAS and gnomAD AF must NEVER be trigger targets for AlphaGenome."""
    v = Variant("11", 300, "G", "C", canon_key("11", 300, "G", "C"))
    ev_by_key = {
        v.key: [
            {"modality": "gwas", "missingness_flag": "absent", "evidence_value": json.dumps({"reason": "not_in_catalog"})},
            {"modality": "gnomad_af", "missingness_flag": "absent", "evidence_value": json.dumps({"reason": "not_found"})}
        ]
    }
    trigs, _ = compile_alphagenome_triggers(
        variants=[v],
        evidence_by_key=ev_by_key,
        completable_modalities=L2["completable_modalities"],
        target_tissues=L2["target_tissues"],
        head_mapping=MODALITY_META,
        gene_meta={"gene_symbol": "TEST"}
    )
    modalities_in_trigs = {t["modality"] for t in trigs}
    assert "gwas" not in modalities_in_trigs, "GWAS cannot be completed by AlphaGenome!"
    assert "gnomad_af" not in modalities_in_trigs, "gnomAD AF cannot be completed by AlphaGenome!"
    print("PASS: non-completable modalities (GWAS, gnomAD) strictly excluded from AlphaGenome")


def test_agreement_queue_strict_isolation():
    """Observed measurements go to agreement; absent evidence goes to triggers."""
    v = Variant("11", 400, "T", "G", canon_key("11", 400, "T", "G"))

    ev_by_key = {
        v.key: [
            {
                "modality": "mpra",
                "context": "K562",
                "missingness_flag": "observed_positive",
                "evidence_value": json.dumps({"skew": 1.2}),
            },
            {
                "modality": "crispr",
                "context": "K562",
                "missingness_flag": "absent",
                "evidence_value": json.dumps({
                    "reason": "no_data_source_configured"
                }),
            },
        ]
    }

    test_cell_types = ["K562"]

    trigs, _ = compile_alphagenome_triggers(
        variants=[v],
        evidence_by_key=ev_by_key,
        completable_modalities=["mpra", "crispr"],
        target_tissues=[],
        head_mapping=MODALITY_META,
        gene_meta={"gene_symbol": "TEST"},
        target_cell_types=test_cell_types,
    )

    agrees = compile_agreement_check_queue(
        variants=[v],
        evidence_by_key=ev_by_key,
        completable_modalities=["mpra", "crispr"],
        target_tissues=[],
        target_cell_types=test_cell_types,
    )

    trig_mods = {t["modality"] for t in trigs}
    agree_mods = {a["modality"] for a in agrees}

    assert "mpra" in agree_mods and "mpra" not in trig_mods, (
        "Observed MPRA must be in agreement queue, not trigger queue!"
    )
    assert "crispr" in trig_mods and "crispr" not in agree_mods, (
        "Absent CRISPR must be in trigger queue, not agreement queue!"
    )

    print("PASS: agreement check queue strictly segregated from trigger queue")


def test_stratification_tier_boundaries():
    """Verify correct assignment of Tier 1 (>=2), Tier 2 (1), and Tier 3 (0)."""
    v_rich = Variant("11", 501, "A", "T", canon_key("11", 501, "A", "T"))
    v_mod = Variant("11", 502, "A", "T", canon_key("11", 502, "A", "T"))
    v_sparse = Variant("11", 503, "A", "T", canon_key("11", 503, "A", "T"))

    ev_by_key = {
        # Rich: eQTL + MPRA tested; GWAS is position-level and does not count toward allele-specific tier
        v_rich.key: [
            {"modality": "gwas", "missingness_flag": "observed_positive",
             "evidence_value": json.dumps({"matching_level": "position", "allele_specific": False})},
            {"modality": "eqtl", "missingness_flag": "observed_null", "evidence_value": "{}"},
            {"modality": "mpra", "missingness_flag": "observed_positive", "evidence_value": "{}"},
        ],
        # Moderate: only eQTL tested
        v_mod.key: [
            {"modality": "gwas", "missingness_flag": "absent", "evidence_value": "{}"},
            {"modality": "eqtl", "missingness_flag": "observed_positive", "evidence_value": "{}"},
        ],
        # Sparse: all absent
        v_sparse.key: [
            {"modality": "gwas", "missingness_flag": "absent", "evidence_value": "{}"},
            {"modality": "eqtl", "missingness_flag": "absent", "evidence_value": "{}"},
        ],
    }
    df = compute_evidence_tiers([v_rich, v_mod, v_sparse], ev_by_key)
    tier_map = dict(zip(df["variant_key"], df["evidence_tier"]))

    key_rich = f"{v_rich.chrom}:{v_rich.pos}:{v_rich.ref}:{v_rich.alt}"
    key_mod = f"{v_mod.chrom}:{v_mod.pos}:{v_mod.ref}:{v_mod.alt}"
    key_sparse = f"{v_sparse.chrom}:{v_sparse.pos}:{v_sparse.ref}:{v_sparse.alt}"

    assert tier_map[key_rich] == "Tier 1 (Evidence-Rich)"
    assert tier_map[key_mod] == "Tier 2 (Moderate Evidence)"
    assert tier_map[key_sparse] == "Tier 3 (Evidence-Sparse)"
    print("PASS: stratification tier boundaries correctly assigned")


def test_manifest_provenance_recording():
    """Manifest payload must record all required fields, hashes, and distributions."""
    import tempfile
    with tempfile.TemporaryDirectory() as tmpdir:
        payload = {
            "gene_symbol": "HBB",
            "tier_distribution": {"Tier 1": 10, "Tier 3": 50},
            "alphagenome_triggers": {"total_jobs": 120}
        }
        path = write_manifest(tmpdir, "test_manifest.json", payload)
        assert os.path.exists(path)
        with open(path) as f:
            loaded = json.load(f)
        assert loaded["gene_symbol"] == "HBB"
        assert "generated_at_utc" in loaded
        assert loaded["tier_distribution"]["Tier 3"] == 50
    print("PASS: provenance manifest serialization and schema integrity")


def test_target_tissues_respect_layer1_scope():
    """v1 queued Liver jobs although Layer 1 only queried Whole_Blood. Must warn once and queue nothing."""
    v = Variant("11", 600, "A", "G", canon_key("11", 600, "A", "G"))
    ev_by_key = {v.key: [{"modality": "eqtl", "context": "Whole_Blood", "missingness_flag": "absent",
                          "evidence_value": json.dumps({"reason": "not_testable_rare_or_unknown_frequency"})}]}
    trigs, warns = compile_alphagenome_triggers([v], ev_by_key, ["eqtl"], ["Whole_Blood", "Liver"],
                                                MODALITY_META, {"gene_symbol": "TEST"})
    assert {t["context"] for t in trigs} == {"Whole_Blood"}, "Only the tissue Layer 1 queried may be completed"
    liver_warns = [w for w in warns if w.startswith("CONFIG:") and "Liver" in w]
    assert len(liver_warns) == 1, "Un-queried target tissue must raise exactly one CONFIG warning"
    print("PASS: un-queried target tissue -> CONFIG warning, no AlphaGenome jobs")


def test_non_target_context_does_not_lift_tier():
    """A Liver eQTL hit must not make an erythroid gene's variant look evidence-rich."""
    v = Variant("11", 601, "A", "G", canon_key("11", 601, "A", "G"))
    ev_by_key = {v.key: [
        {"modality": "eqtl", "context": "Liver", "missingness_flag": "observed_positive", "evidence_value": "{}"},
        {"modality": "mpra", "context": "HepG2", "missingness_flag": "observed_positive", "evidence_value": "{}"},
        {"modality": "gwas", "context": None, "missingness_flag": "absent", "evidence_value": "{}"},
    ]}
    row = compute_evidence_tiers([v], ev_by_key, regulatory_modalities=("gwas", "eqtl", "mpra", "crispr")).iloc[0]
    assert row["eqtl_status"] == "absent" and row["mpra_status"] == "absent"
    assert row["evidence_tier"] == "Tier 3 (Evidence-Sparse)"
    agrees = compile_agreement_check_queue([v], ev_by_key, ["eqtl", "mpra"])
    assert agrees == [], "Out-of-scope contexts must not enter the agreement queue"
    print("PASS: non-target tissues / cell types excluded from status, tiers and agreement checks")


def test_mpra_crispr_triggers_use_target_cell_types():
    v = Variant("11", 602, "C", "T", canon_key("11", 602, "C", "T"))
    ev_by_key = {v.key: [{"modality": "mpra", "context": None, "missingness_flag": "absent",
                          "evidence_value": json.dumps({"reason": "no_data_source_configured"})}]}
    trigs, _ = compile_alphagenome_triggers([v], ev_by_key, ["mpra", "crispr"], [], MODALITY_META,
                                            {"gene_symbol": "TEST"}, target_cell_types=["K562"])
    assert {t["context"] for t in trigs} == {"K562"}, "Jobs must be labelled with the target cell type, not a placeholder"
    assert {t["modality"] for t in trigs} == {"mpra", "crispr"}
    print("PASS: MPRA/CRISPR triggers are per target cell type (K562), never HepG2 / placeholder")


def test_trigger_output_types_and_serialization():
    v = Variant("11", 603, "G", "A", canon_key("11", 603, "G", "A"))
    ev_by_key = {v.key: [
        {"modality": "eqtl", "context": "Whole_Blood", "missingness_flag": "absent",
         "evidence_value": json.dumps({"reason": "not_testable_rare_or_unknown_frequency", "maf": None})},
        {"modality": "mpra", "context": None, "missingness_flag": "absent",
         "evidence_value": json.dumps({"reason": "no_data_source_configured"})},
    ]}
    trigs, _ = compile_alphagenome_triggers([v], ev_by_key, ["eqtl", "mpra", "crispr"], ["Whole_Blood"],
                                            MODALITY_META, {"gene_symbol": "TEST"})
    df = pd.DataFrame(trigs)
    # This unit test deliberately exercises the CRISPR schema even when the real project
    # has no CRISPR source configured. Temporarily enable it only for this isolated test.
    _old_completable = list(L2["completable_modalities"])
    try:
        L2["completable_modalities"] = ["eqtl", "mpra", "crispr"]
        validate_layer2_triggers_output(df)
    finally:
        L2["completable_modalities"] = _old_completable
    assert "target_head" not in df.columns
    assert set(df["alphagenome_binding_status"]) == {"unverified"}
    for s in df["alphagenome_output_types"]:
        assert set(s.split(";")) <= set(AG_OUTPUT_TYPES)
    for s in df["source_evidence_value"]:
        assert isinstance(json.loads(s), dict), "source_evidence_value must not be double JSON-encoded"
    assert set(df.loc[df.modality == "mpra", "alphagenome_proxy_quality"]) == {"indirect_proxy"}
    print("PASS: output types are from the paper's modality list, binding unverified, no double-encoded JSON")


def test_nan_context_is_normalized():
    recs = [{"missingness_flag": "absent", "context": float("nan"),
             "evidence_value": json.dumps({"reason": "no_data_source_configured"})}]
    _, _, _, by_ctx = aggregate_modality_status(recs)
    assert list(by_ctx) == ["default"], f"NaN context must normalize to 'default', got {list(by_ctx)}"
    assert _norm_ctx(None) == "default" and _norm_ctx("") == "default" and _norm_ctx("K562") == "K562"
    print("PASS: NaN / empty contexts from CSV normalize to 'default'")


def test_synthetic_outputs_are_segregated():
    assert resolve_out_dir("HBB", False) == os.path.join("data/outputs", "HBB")
    syn = resolve_out_dir("HBB", True)
    assert syn == os.path.join("data/outputs", "HBB", "synthetic_demo")
    assert os.path.dirname(syn) != os.path.dirname(os.path.join(resolve_out_dir("HBB", False), "x")) or syn.endswith("synthetic_demo")
    print("PASS: synthetic demo outputs go to synthetic_demo/, never the real output path")


def test_tier_and_label_warnings():
    import pandas as _pd
    df = _pd.DataFrame({"evidence_tier": ["Tier 3 (Evidence-Sparse)"] * 6, "label": ["pathogenic"] * 6})
    tw = tier_balance_warnings(df)
    assert any(w.startswith("Tier 1") and "0 variants" in w for w in tw)
    assert any(w.startswith("Tier 2") and "0 variants" in w for w in tw)
    lw = label_coverage_warnings(df)
    assert any("no 'benign'" in w for w in lw)
    assert label_coverage_warnings(df.drop(columns=["label"]))[0].startswith("No ClinVar")
    print("PASS: empty tiers, missing classes and missing labels are all surfaced")


def test_labels_matching_no_variant_does_not_crash():
    v = Variant("11", 700, "A", "G", canon_key("11", 700, "A", "G"))
    df = compute_evidence_tiers([v], {v.key: []}, clinvar_labels={("11", 1, "A", "G"): {"CLNSIG": "x", "label": "benign", "review_stars": 1}})
    assert "label" in df.columns and df["label"].isna().all()
    print("PASS: Layer 0 labels that match no variant no longer raise KeyError")





def test_mpra_crispr_per_cell_type_granularity():
    """The headline v3 fix: a hit in ONE target cell type must not suppress completion for a
    SECOND, genuinely never-tested target cell type. v2 used one aggregate status across all
    target_cell_types, so this was silently swallowed whenever more than one was configured."""
    v = Variant("11", 900, "A", "G", canon_key("11", 900, "A", "G"))
    ev_by_key = {v.key: [
        {"modality": "mpra", "context": "K562", "missingness_flag": "observed_positive",
         "evidence_value": json.dumps({"log2_allelic_skew": 2.1, "pvalue": 1e-7})},
    ]}
    trigs, _ = compile_alphagenome_triggers([v], ev_by_key, ["mpra"], [], MODALITY_META,
                                            {"gene_symbol": "TEST"}, target_cell_types=["K562", "HUDEP2"])
    contexts = {t["context"] for t in trigs}
    assert "K562" not in contexts, "K562 has a real hit -- must not be re-triggered"
    assert "HUDEP2" in contexts, "HUDEP2 was never tested at all -- MUST be triggered, v2 silently dropped this"
    print("PASS: MPRA/CRISPR trigger generation is per-cell-type, not one aggregate status for all targets")


def test_mpra_crispr_default_context_reason_preserved():
    """A context-less ('default') absent record -- the real shape Layer 1 emits for
    no_data_source_configured / not_tested_in_provided_table -- must still carry its real reason
    and evidence payload into the trigger, for every target cell type, not just a bare fallback."""
    v = Variant("11", 901, "C", "T", canon_key("11", 901, "C", "T"))
    ev_by_key = {v.key: [
        {"modality": "mpra", "context": None, "missingness_flag": "absent",
         "evidence_value": json.dumps({"reason": "no_data_source_configured"})},
    ]}
    trigs, _ = compile_alphagenome_triggers([v], ev_by_key, ["mpra"], [], MODALITY_META,
                                            {"gene_symbol": "TEST"}, target_cell_types=["K562", "HUDEP2"])
    assert {t["context"] for t in trigs} == {"K562", "HUDEP2"}
    assert all(t["trigger_reason"] == "no_data_source_configured" for t in trigs), \
        "the real reason from the default-context record must carry through, not a generic fallback"
    print("PASS: default-context MPRA/CRISPR records apply their real reason to every target cell type")


def test_mpra_crispr_never_tested_cell_type_is_not_a_config_mismatch():
    """A target cell type that never appears anywhere in the evidence (e.g. HUDEP2 configured as a
    target but the supplied MPRA table only covers K562) must still get a trigger -- unlike eQTL,
    MPRA/CRISPR have no 'Layer 1 iterated this tissue' signal, so there's no config-mismatch case
    to suppress it with. An earlier draft of this fix mistakenly added that suppression and it
    cancelled out the fix above; this test guards against reintroducing it."""
    v = Variant("11", 902, "G", "A", canon_key("11", 902, "G", "A"))
    ev_by_key = {v.key: [
        {"modality": "mpra", "context": "K562", "missingness_flag": "observed_null",
         "evidence_value": json.dumps({"reason": "tested_not_significant", "pvalue": 0.8})},
    ]}
    trigs, warns = compile_alphagenome_triggers([v], ev_by_key, ["mpra"], [], MODALITY_META,
                                                {"gene_symbol": "TEST"}, target_cell_types=["K562", "HUDEP2"])
    assert any(t["context"] == "HUDEP2" for t in trigs), \
        "HUDEP2 never appears in the evidence at all -- must still trigger, not be suppressed as a config mismatch"
    assert not any("CONFIG" in w and "HUDEP2" in w for w in warns)
    print("PASS: a cell type absent from the evidence entirely is treated as a gap, not a config mismatch")


def test_unknown_reason_is_never_completed_and_warns():
    """A reason string Layer 1 emits that Layer 2's REASON_CATEGORIES doesn't recognize must never
    silently be treated as a safe-to-complete biological gap -- v2's fallback was GAP_BIOLOGICAL,
    which would have queued (and paid for) an AlphaGenome job for what might be a technical problem
    under a new, unrecognized label."""
    assert classify_evidence_reason("absent", reason="some_future_reason_layer1_added") == "UNKNOWN_REASON"
    assert classify_evidence_reason("absent", reason=None) == "GAP_BIOLOGICAL", \
        "no reason at all (e.g. a minimal fixture) is still an ordinary gap, not flagged as unknown"

    v = Variant("11", 903, "A", "G", canon_key("11", 903, "A", "G"))
    ev_by_key = {v.key: [{"modality": "eqtl", "context": "Whole_Blood", "missingness_flag": "absent",
                          "evidence_value": json.dumps({"reason": "totally_new_reason_string"})}]}
    trigs, warns = compile_alphagenome_triggers([v], ev_by_key, ["eqtl"], ["Whole_Blood"], MODALITY_META,
                                                {"gene_symbol": "TEST"})
    assert len(trigs) == 0, "an unrecognized reason must NOT silently generate an AlphaGenome trigger"
    assert any("unrecognized reason" in w for w in warns), "a human must be able to see the vocabulary drift"
    print("PASS: unrecognized reason strings are never auto-completed and always surfaced as a warning")


def test_empty_layer1_evidence_raises_clear_error():
    """Layer 1 evidence that loads successfully but contains zero rows must raise a clear,
    actionable error -- v2's live-intake path did `variants[0]` unguarded and would have crashed
    with a bare IndexError."""
    empty_df = pd.DataFrame(columns=["chrom", "pos", "ref", "alt", "modality", "source_type",
                                     "evidence_value", "context", "missingness_flag"])
    empty_variants = to_variants(empty_df)
    try:
        derive_fallback_gene_meta(empty_variants, "HBB")
        assert False, "should have raised ValueError on zero candidate variants"
    except ValueError as e:
        assert "zero candidate variants" in str(e)
    print("PASS: zero-row Layer 1 evidence raises a clear, actionable ValueError instead of IndexError")


def test_gnomad_af_status_is_validated():
    """v2 validated 4 of the 5 status columns in availability_df and silently skipped
    gnomad_af_status -- a corrupted value there would have gone undetected."""
    v = Variant("11", 904, "C", "T", canon_key("11", 904, "C", "T"))
    ev_by_key = {v.key: [{"modality": "gnomad_af", "context": "gnomad_r4:genomes",
                         "missingness_flag": "observed_positive", "evidence_value": json.dumps({"af": 0.2})}]}
    df = compute_evidence_tiers([v], ev_by_key)
    validate_layer2_availability_output(df)  # must not raise on valid data
    df.loc[0, "gnomad_af_status"] = "not_a_real_status"
    try:
        validate_layer2_availability_output(df)
        assert False, "an invalid gnomad_af_status value should have been caught"
    except ValueError:
        pass
    print("PASS: gnomad_af_status is now validated along with the other four status columns")


def test_canon_key_matches_layer1_suffix_trimming():
    """v2's canon_key only trimmed a matching prefix; Layer 1's trims a matching suffix first, then
    a matching prefix, down to empty. This is the case that separated them."""
    assert canon_key("11", 1000, "CAT", "GAT") == canon_key("11", 1000, "C", "G"), \
        "a shared 'AT' suffix must be trimmed, matching Layer 1's normalization"
    # the non-minimal-indel case both versions already handled correctly, kept as a regression guard
    assert canon_key("11", 5227100, "G", "GA") == canon_key("11", 5227099, "AG", "AGA")
    print("PASS: canon_key fully normalizes shared suffixes, matching Layer 1's algorithm")


def test_layer1_input_validation_rejects_bad_flags():
    """validate_layer1_input must catch a corrupted/typo'd missingness_flag before it's silently
    treated as 'absent' by the reason classifier (anything not exactly observed_positive /
    observed_null currently falls through to absent handling)."""
    bad_df = pd.DataFrame([{
        "chrom": "11", "pos": 1, "ref": "A", "alt": "G", "modality": "eqtl",
        "source_type": "experimental", "evidence_value": "{}", "context": None,
        "missingness_flag": "observed_postive",  # typo
    }])
    try:
        validate_layer1_input(bad_df)
        assert False, "a typo'd missingness_flag should have been rejected"
    except ValueError as e:
        assert "invalid missingness_flag" in str(e)
    good_df = bad_df.copy()
    good_df.loc[0, "missingness_flag"] = "observed_positive"
    validate_layer1_input(good_df)  # must not raise
    print("PASS: validate_layer1_input rejects corrupted missingness_flag values before they're trusted")


def test_empty_gtex_result_is_not_a_tested_null():
    """A non-hit in GTEx significant-association results is not a tested null."""
    v = Variant("11", 900, "A", "G", canon_key("11", 900, "A", "G"))
    for gene_rows in (0, 3):
        eqtl_fixture = {
            "gencode_id": "ENSG00000244734.3",
            "resolve_error": None,
            "tissues": {
                "Whole_Blood": {"error": None, "n_rows": gene_rows, "by_key": {}}
            }
        }
        records = build_eqtl_records(
            eqtl_fixture, [v], ["Whole_Blood"], {v.key: 0.12}, 0.01
        )
        assert len(records) == 1
        assert records[0].missingness_flag == "absent"
        payload = parse_evidence_value(records[0].evidence_value)
        assert payload["reason"] == "no_significant_eqtl_association_returned_test_status_unverified"
    print("PASS: GTEx non-hits are absent/unverified, never unsupported tested-nulls")


# Run the full regression suite
ALL_TESTS = [
    test_multi_context_aggregation_order,
    test_transient_error_handling,
    test_tested_nulls_never_trigger_alphagenome,
    test_non_completable_modalities,
    test_agreement_queue_strict_isolation,
    test_stratification_tier_boundaries,
    test_manifest_provenance_recording,
    test_target_tissues_respect_layer1_scope,
    test_non_target_context_does_not_lift_tier,
    test_mpra_crispr_triggers_use_target_cell_types,
    test_trigger_output_types_and_serialization,
    test_nan_context_is_normalized,
    test_synthetic_outputs_are_segregated,
    test_tier_and_label_warnings,
    test_labels_matching_no_variant_does_not_crash,
    test_mpra_crispr_per_cell_type_granularity,
    test_mpra_crispr_default_context_reason_preserved,
    test_mpra_crispr_never_tested_cell_type_is_not_a_config_mismatch,
    test_unknown_reason_is_never_completed_and_warns,
    test_empty_layer1_evidence_raises_clear_error,
    test_gnomad_af_status_is_validated,
    test_canon_key_matches_layer1_suffix_trimming,
    test_layer1_input_validation_rejects_bad_flags,
    test_empty_gtex_result_is_not_a_tested_null,
]
print("\n" + "=" * 50)
print("RUNNING LAYER 2 REGRESSION & SANITY CHECK SUITE")
print("=" * 50)
for _t in ALL_TESTS:
    _t()
print(f"\nALL {len(ALL_TESTS)} REGRESSION & SANITY TESTS PASSED SUCCESSFULLY!\n")




RUNNING LAYER 2 REGRESSION & SANITY CHECK SUITE
PASS: multi-context aggregation order (positive > null > absent)
PASS: transient query errors are flagged and isolated from AlphaGenome triggers
PASS: observed negative results (tested nulls) never trigger AlphaGenome
PASS: non-completable modalities (GWAS, gnomAD) strictly excluded from AlphaGenome
PASS: agreement check queue strictly segregated from trigger queue
PASS: stratification tier boundaries correctly assigned
PASS: provenance manifest serialization and schema integrity
PASS: un-queried target tissue -> CONFIG warning, no AlphaGenome jobs
PASS: non-target tissues / cell types excluded from status, tiers and agreement checks
PASS: MPRA/CRISPR triggers are per target cell type (K562), never HepG2 / placeholder
PASS: output types are from the paper's modality list, binding unverified, no double-encoded JSON
PASS: NaN / empty contexts from CSV normalize to 'default'
PASS: synthetic demo outputs go to synthetic_demo/, never the real

## 9b. Regression tests for the v3 fixes (offline)


In [37]:
import tempfile, contextlib, os as _os
from unittest import mock


@contextlib.contextmanager
def _in_tmpdir():
    old = _os.getcwd()
    with tempfile.TemporaryDirectory() as d:
        _os.chdir(d)
        try:
            yield Path(d)
        finally:
            _os.chdir(old)


class _FakeResp:
    def __init__(self, status, payload=None):
        self.status_code, self._p, self.headers = status, payload, {}
    def json(self):
        if self._p is None:
            raise ValueError("no json")
        return self._p


def test_default_config_is_created_once_and_parses():
    with _in_tmpdir():
        assert ensure_project_config("configs/t.yaml") is True
        cfg = yaml.safe_load(Path("configs/t.yaml").read_text())
        assert cfg["layer2"]["target_cell_types"] == ["HEL92.1.7"]
        assert cfg["layer0"]["clinvar_snapshot_date"] == "20261019"
        assert set(cfg["layer1"]) <= set(DEFAULT_L1), "config layer1 keys must all be recognised by Layer 1"
        Path("configs/t.yaml").write_text("gene_symbol: EDITED\n")
        assert ensure_project_config("configs/t.yaml") is False
        assert "EDITED" in Path("configs/t.yaml").read_text(), "an existing config must never be overwritten"
    print("PASS: default config created once, parses, uses only recognised keys, never overwrites user edits")


def test_http_410_is_retried_then_reported():
    calls = {"n": 0}
    def always_410(*a, **k):
        calls["n"] += 1
        return _FakeResp(410)
    cache = HTTPCache(cache_dir=None, enabled=False)
    f = make_http_fetch(cache, max_retries=3, backoff=1.0)
    with mock.patch.object(f.session, "request", always_410), mock.patch("time.sleep"):
        data, err, _ = f("GET", "https://example.test/x")
    assert err == "http_410" and data is None and calls["n"] == 3, (err, calls)   # 1 try + 2 bounded retries
    seq = iter([_FakeResp(410), _FakeResp(200, {"ok": 1})])
    f2 = make_http_fetch(cache, max_retries=3, backoff=1.0)
    with mock.patch.object(f2.session, "request", lambda *a, **k: next(seq)), mock.patch("time.sleep"):
        data, err, _ = f2("GET", "https://example.test/y")
    assert err is None and data == {"ok": 1}
    print("PASS: HTTP 410 is retried a bounded number of times, then reported as http_410 (never cached)")


def test_gwas_response_shape_tolerance_and_guard():
    hal = {"_embedded": {"snps": [{"rs_id": "rs1"}]}, "_links": {"next": {"href": "https://x/next{?page}"}}}
    plain = {"content": [{"rs_id": "rs2"}], "page": {"number": 0}}
    empty_valid = {"_links": {}, "page": {"totalElements": 0}}
    weird = {"unexpected": "shape"}
    pages = {"https://x/start": hal, "https://x/next": {"_embedded": {"snps": [{"rs_id": "rs3"}]}, "_links": {}}}
    fetch = lambda m, url, params=None: (pages[url], None, False)
    items, err = fetch_paginated_hal("https://x/start", {}, "snps", fetch)
    assert err is None and [i["rs_id"] for i in items] == ["rs1", "rs3"]
    assert fetch_paginated_hal("https://x/p", {}, "snps", lambda m, u, params=None: (plain, None, False)) == ([{"rs_id": "rs2"}], None)
    assert fetch_paginated_hal("https://x/e", {}, "snps", lambda m, u, params=None: (empty_valid, None, False)) == ([], None)
    items, err = fetch_paginated_hal("https://x/w", {}, "snps", lambda m, u, params=None: (weird, None, False))
    assert err == "unrecognized_response_shape" and items == [], "an unknown payload must be an ERROR, not 'not in catalog'"
    print("PASS: GWAS parser handles HAL and plain list payloads, and flags unrecognised shapes instead of reporting absence")


def test_gwas_failure_propagates_to_layer2_and_gate():
    v = Variant("11", 5225600, "A", "G", canon_key("11", 5225600, "A", "G"))
    failing = lambda m, url, params=None, json_body=None: (None, "http_410", False)
    gwas = collect_gwas_evidence("11", [v.pos], fetch=failing, window_bp=25000)
    recs = build_gwas_records(gwas, [v], 5e-8)
    assert recs[0].missingness_flag == "absent" and "incomplete_region_query" in recs[0].evidence_value
    df = records_to_dataframe(recs)
    warns = scan_query_errors(df)
    assert len(warns) == 1 and "gwas" in warns[0] and "http_410" in warns[0], warns
    try:
        enforce_data_quality_gate(warns, halt=True)
        raise AssertionError("gate must halt on query failures")
    except RuntimeError as e:
        assert "halted" in str(e)
    assert enforce_data_quality_gate(warns, halt=False) == "DEGRADED"
    assert enforce_data_quality_gate([], halt=True) == "OK"
    print("PASS: a GWAS 410 is surfaced in Layer 2 (non-completable modality) and halts / marks DEGRADED")


def test_layer0_manifest_is_discovered_and_gene_meta_pinned():
    with _in_tmpdir():
        Path("layer0_outputs").mkdir()
        pd.DataFrame({"chrom": ["11"], "pos": [5227002], "ref": ["T"], "alt": ["C"], "label": ["benign"]}) \
            .to_csv("layer0_outputs/HBB_layer0_regulatory_truthset.csv", index=False)
        gm = {"gene_symbol": "HBB", "gene_id": "ENSG00000244734", "chrom": "11", "start": 5225464,
              "end": 5229395, "strand": -1}
        Path("layer0_outputs/HBB_layer0_manifest.json").write_text(
            json.dumps({"gene_metadata": gm, "clinvar_snapshot_date": "20261019"}))
        df, meta, info = load_layer0_output("HBB", None, fetch=lambda *a, **k: (_ for _ in ()).throw(AssertionError("no network allowed")))
        assert meta == gm and info["manifest_path"] and info["clinvar_snapshot_date"] == "20261019"
    print("PASS: Layer 0 manifest is found in the layer-0 notebook layout and pins gene metadata (no network)")


def test_scope_audit_catches_wrong_cell_type():
    ev_df = pd.DataFrame([
        {"chrom": "11", "pos": 1, "ref": "A", "alt": "G", "modality": "mpra", "source_type": "experimental",
         "evidence_value": "{}", "context": "K562", "missingness_flag": "observed_positive"},
        {"chrom": "11", "pos": 2, "ref": "A", "alt": "G", "modality": "mpra", "source_type": "experimental",
         "evidence_value": "{}", "context": None, "missingness_flag": "absent"}])
    warns, tbl = audit_context_scope(ev_df, ["Whole_Blood"], ["HEL92.1.7"])
    assert any("NONE of the mpra" in w for w in warns) and int(tbl.loc[tbl.modality == "mpra", "records_excluded"].iloc[0]) == 1
    warns_ok, _ = audit_context_scope(ev_df, ["Whole_Blood"], ["K562"])
    assert not warns_ok
    print("PASS: context-scope audit flags MPRA evidence that would be silently excluded by the target cell type")


def test_tier1_feasibility_lists_promotable_variants():
    av = pd.DataFrame({
        "variant_key": ["a", "b", "c", "d"],
        "eqtl_status": ["absent", "observed_positive", "absent", "absent"],
        "mpra_status": ["observed_positive", "observed_positive", "absent", "observed_null"],
        "crispr_status": ["absent", "absent", "absent", "absent"],
        "evidence_tier": ["Tier 2 (Moderate Evidence)", "Tier 1 (Evidence-Rich)", "Tier 3 (Evidence-Sparse)", "Tier 2 (Moderate Evidence)"]})
    c = tier1_feasibility(av, 2)
    assert list(c["variant_key"]) == ["a", "d"] and c.iloc[0]["has_modalities"] == "mpra"
    print("PASS: Tier 1 feasibility lists exactly the variants one extra source away from Tier 1")


def test_layer0_logic_offline():
    vep = [{"start": 5226000, "seq_region_name": "11", "allele_string": "T/C",
            "transcript_consequences": [{"gene_id": "ENSG00000244734", "transcript_id": "ENST00000335295",
                                         "consequence_terms": ["5_prime_UTR_variant"]},
                                        {"gene_id": "OTHER", "consequence_terms": ["missense_variant"]}]},
           {"start": 5226100, "seq_region_name": "11", "allele_string": "A/G",
            "transcript_consequences": [], "regulatory_feature_consequences": []}]
    vdf = parse_vep_results(vep, "ENST00000335295", "ENSG00000244734")
    assert list(vdf["vep_all_transcript_terms"][0]) == ["5_prime_UTR_variant"], "other genes' terms must be excluded"
    base = pd.DataFrame({"chrom": ["11", "11"], "pos": [5226000, 5226100], "ref": ["T", "A"], "alt": ["C", "G"],
                         "CLNSIG": ["Pathogenic", "Benign"], "CLNREVSTAT": ["x", "y"], "review_stars": [2, 1],
                         "label": ["pathogenic", "benign"], "interval_evidence": [[], []]})
    merged, final = build_regulatory_table(base, vdf)
    assert list(final["pos"]) == [5226000] and final.iloc[0]["regulatory_evidence"] == ["vep:5_prime_UTR_variant"]
    try:
        import pyranges  # noqa: F401
        flagged = flag_interval_overlap(base[["chrom", "pos", "ref", "alt"]].assign(label=["pathogenic", "benign"]),
                                        pd.DataFrame({"Chromosome": ["11"], "Start": [5225990], "End": [5226010], "ccre_label": ["promoter-like"]}))
        assert flagged["interval_evidence"].tolist() == [["promoter-like"], []]
        extra = " + cCRE interval overlap"
    except ImportError:
        extra = " (pyranges not installed: interval-overlap check skipped)"
    print("PASS: Layer 0 VEP parsing / regulatory classification" + extra)


def _make_bulk_zip(path):
    import zipfile
    header = ["DATE ADDED TO CATALOG", "PUBMEDID", "DISEASE/TRAIT", "CHR_ID", "CHR_POS", "MAPPED_GENE", "SNPS",
              "P-VALUE", "PVALUE_MLOG", "MAPPED_TRAIT", "STUDY ACCESSION"]
    rows = [
        ["2020-01-01", "1", "Fetal hemoglobin", "11", "5227002", "HBB", "rs1", "5E-8", "7.3", "fetal hemoglobin measurement", "GCST1"],
        ["2020-01-01", "2", "Hematocrit", "11; 11", "5227002; 5227100", "HBB", "rs1; rs2", "", "9", "hematocrit", "GCST2"],
        ["2020-01-01", "3", "Interaction", "11 x 12", "5227500 x 100", "HBB", "rs9 x rs8", "1E-30", "29", "x", "GCST3"],
        ["2020-01-01", "4", "Other chromosome", "12", "5227002", "X", "rs7", "1E-30", "29", "y", "GCST4"],
        ["2020-01-01", "5", "Weak", "11", "5300000", "HBB", "rs5", "3E-3", "2.5", "weak trait", "GCST5"],
    ]
    tsv = "\t".join(header) + "\n" + "\n".join("\t".join(r) for r in rows) + "\n"
    with zipfile.ZipFile(path, "w") as zf:
        zf.writestr("gwas-catalog-associations_ontology-annotated.tsv", tsv)


def test_gwas_bulk_file_parsing_and_records():
    with _in_tmpdir() as d:
        zp = str(d / "bulk.zip")
        _make_bulk_zip(zp)
        positions = [5227002, 5227100, 5228000, 5300000]
        res = collect_gwas_evidence_from_download("11", positions, zp, window_bp=25000)
        assert all(w["error"] is None for w in res["windows"]) and res["n_snps_in_region"] == 4, res["n_snps_in_region"]
        assert set(res["by_pos"]) == {5227002, 5227100, 5300000}
        e = res["by_pos"][5227002]
        assert e["rs_ids"] == ["rs1"] and len(e["assocs"]) == 2
        pv = sorted(a["pvalue"] for a in e["assocs"])
        assert abs(pv[0] - 1e-9) < 1e-12 and abs(pv[1] - 5e-8) < 1e-12, pv   # 2nd row's p derived from PVALUE_MLOG
        vs = [Variant("11", p, "A", "G", canon_key("11", p, "A", "G")) for p in positions]
        recs = {r.pos: r for r in build_gwas_records(res, vs, 5e-8)}
        assert recs[5227002].missingness_flag == "observed_positive" and recs[5227100].missingness_flag == "observed_positive"
        assert recs[5300000].missingness_flag == "observed_null"
        assert json.loads(recs[5228000].evidence_value)["reason"] == "not_in_catalog"
        assert "bulk download" in json.loads(recs[5227002].evidence_value)["source"]
        bad = str(d / "bad.zip"); Path(bad).write_text("not a zip")
        broken = collect_gwas_evidence_from_download("11", positions, bad)
        assert broken["windows"][0]["error"].startswith("bulk_file_unreadable")
    print("PASS: GWAS bulk-file parser (multi-SNP rows, other chromosomes, p from -log10, corrupt file) and records")


def test_gwas_auto_fallback_and_modes():
    pos = [5227002]
    failing = lambda m, u, params=None, json_body=None: (None, "http_410", False)
    good_bulk = {"windows": [{"chrom": "11", "start": 5227002, "stop": 5227002, "error": None, "n_snps": 1}],
                 "by_pos": {5227002: {"rs_ids": ["rs1"], "assocs": [{"traits": ["t"], "pvalue": 1e-9, "rs_id": "rs1"}], "error": None}},
                 "n_snps_in_region": 1, "source_label": "bulk"}
    probes = []
    r = collect_gwas_with_fallback("11", pos, failing, {"gwas_source": "auto"},
                                   bulk_fn=lambda c, p, L: (good_bulk, {"sha256": "x"}, None), probe_fn=lambda *a: probes.append(a))
    assert r["used"] == "download" and not r["warnings"] and r["notes"] and probes and r["file_info"] == {"sha256": "x"}
    r = collect_gwas_with_fallback("11", pos, failing, {"gwas_source": "auto"},
                                   bulk_fn=lambda c, p, L: (None, None, "bulk_download_failed:X"), probe_fn=lambda *a: None)
    assert r["used"] == "api" and any("also unavailable" in w for w in r["warnings"]) and any("REST API" in w for w in r["warnings"])
    called = []
    r = collect_gwas_with_fallback("11", pos, failing, {"gwas_source": "api"}, bulk_fn=lambda *a: called.append(1), probe_fn=lambda *a: None)
    assert not called and r["warnings"] and r["used"] == "api"
    ok_fetch = lambda m, u, params=None, json_body=None: ({"_embedded": {"snps": []}}, None, False)
    r = collect_gwas_with_fallback("11", pos, ok_fetch, {"gwas_source": "auto"}, bulk_fn=lambda *a: called.append(2), probe_fn=lambda *a: None)
    assert r["used"] == "api" and not r["warnings"] and called == []
    r = collect_gwas_with_fallback("11", pos, failing, {"gwas_source": "download"}, bulk_fn=lambda c, p, L: (None, None, "bulk_download_failed:X"))
    assert r["used"] == "download" and r["warnings"]
    try:
        collect_gwas_with_fallback("11", pos, failing, {"gwas_source": "nope"})
        raise AssertionError("invalid gwas_source must raise")
    except ValueError:
        pass
    print("PASS: GWAS auto mode falls back to the bulk file; api/download modes are honoured; unresolved failures stay warnings")


def run_audit_fix_tests():
    tests = [test_default_config_is_created_once_and_parses, test_http_410_is_retried_then_reported,
             test_gwas_response_shape_tolerance_and_guard, test_gwas_failure_propagates_to_layer2_and_gate,
             test_layer0_manifest_is_discovered_and_gene_meta_pinned, test_scope_audit_catches_wrong_cell_type,
             test_tier1_feasibility_lists_promotable_variants, test_layer0_logic_offline,
             test_gwas_bulk_file_parsing_and_records, test_gwas_auto_fallback_and_modes]
    print("=" * 50)
    print("RUNNING AUDIT-FIX REGRESSION TESTS")
    print("=" * 50)
    for t in tests:
        t()
    print(f"\nALL {len(tests)} AUDIT-FIX TESTS PASSED")


run_audit_fix_tests()


RUNNING AUDIT-FIX REGRESSION TESTS
PASS: default config created once, parses, uses only recognised keys, never overwrites user edits
PASS: HTTP 410 is retried a bounded number of times, then reported as http_410 (never cached)
PASS: GWAS parser handles HAL and plain list payloads, and flags unrecognised shapes instead of reporting absence
PASS: a GWAS 410 is surfaced in Layer 2 (non-completable modality) and halts / marks DEGRADED
PASS: Layer 0 manifest is found in the layer-0 notebook layout and pins gene metadata (no network)
PASS: context-scope audit flags MPRA evidence that would be silently excluded by the target cell type
PASS: Tier 1 feasibility lists exactly the variants one extra source away from Tier 1
PASS: Layer 0 VEP parsing / regulatory classification (pyranges not installed: interval-overlap check skipped)
PASS: GWAS bulk-file parser (multi-SNP rows, other chromosomes, p from -log10, corrupt file) and records
PASS: GWAS auto mode falls back to the bulk file; api/download

## 9. Additional real-input and GWAS interpretation checks


In [38]:
# This cell validates supplied input tables and the GWAS match semantics without network access.
from pathlib import Path
import pandas as pd, json

l0_path = Path(LAYER0_DEST)
mpra_path = Path(MPRA_DEST)
l0_check = pd.read_csv(l0_path, dtype={"chrom": str})
required_l0 = {"chrom", "pos", "ref", "alt"}
assert required_l0.issubset(l0_check.columns), f"Layer 0 missing columns: {required_l0 - set(l0_check.columns)}"
assert pd.to_numeric(l0_check["pos"], errors="coerce").notna().all(), "Layer 0 contains invalid positions."
assert not l0_check.duplicated(["chrom", "pos", "ref", "alt"]).any(), "Layer 0 has duplicate candidate keys."
print(f"PASS: supplied Layer 0 table schema and uniqueness ({len(l0_check)} candidates).")

if mpra_path.exists():
    mpra_check = pd.read_csv(mpra_path, dtype={"chrom": str})
    required_mpra = {"chrom", "pos", "ref", "alt", "cell_type", "log2_allelic_skew", "pvalue", "dataset_name"}
    assert required_mpra.issubset(mpra_check.columns), f"MPRA missing columns: {required_mpra - set(mpra_check.columns)}"
    assert pd.to_numeric(mpra_check["pos"], errors="coerce").notna().all(), "MPRA contains invalid positions."
    l0_keys = set(zip(l0_check.chrom.astype(str).str.replace("chr", "", regex=False),
                      pd.to_numeric(l0_check.pos).astype(int), l0_check.ref.astype(str).str.upper(),
                      l0_check.alt.astype(str).str.upper()))
    mpra_keys = set(zip(mpra_check.chrom.astype(str).str.replace("chr", "", regex=False),
                        pd.to_numeric(mpra_check.pos).astype(int), mpra_check.ref.astype(str).str.upper(),
                        mpra_check.alt.astype(str).str.upper()))
    print(f"PASS: supplied MPRA schema ({len(mpra_check)} rows); exact candidate-allele overlaps: {len(l0_keys & mpra_keys)}.")
else:
    print("MPRA file missing; it will not be treated as tested-null.")

# Synthetic API-shaped record checks are tests only, never added to evidence_df.
vtest = Variant("11", 100, "A", "G", canon_key("11", 100, "A", "G"))
fixture = {"windows": [{"chrom": "11", "start": 100, "stop": 100, "error": None}],
           "by_pos": {100: {"rs_ids": ["rsTEST"], "error": None,
                            "assocs": [{"traits": ["HbF"], "pvalue": 1e-9, "rs_id": "rsTEST"}]}}}
rtest = build_gwas_records(fixture, [vtest], 5e-8)[0]
payload = json.loads(rtest.evidence_value)
assert rtest.missingness_flag == "observed_positive"
assert payload["matching_level"] == "position" and payload["allele_specific"] is False
print("PASS: GWAS position-level evidence is explicitly marked non-allele-specific.")


# GWAS audit: the collector matches by position and explicitly does not claim allele-specificity.
if "evidence_df" in globals() and evidence_df is not None:
    gwas_rows = evidence_df[evidence_df["modality"].eq("gwas")].copy()
    gwas_rows["evidence_payload"] = gwas_rows["evidence_value"].map(parse_evidence_value)
    gwas_rows["matching_level"] = gwas_rows["evidence_payload"].map(lambda x: x.get("matching_level", "unknown"))
    gwas_rows["allele_specific"] = gwas_rows["evidence_payload"].map(lambda x: x.get("allele_specific", False))
    gwas_hits = gwas_rows[gwas_rows["missingness_flag"].eq("observed_positive")].copy()
    print(f"GWAS audit: {len(gwas_hits)} candidate-allele rows are position-level catalog matches; "
          "do not interpret them as allele-specific without a verified effect allele match.")
    display(gwas_hits[["chrom", "pos", "ref", "alt", "context", "missingness_flag",
                       "matching_level", "allele_specific", "evidence_value"]])

if "availability_df" in globals() and "evidence_tier" in availability_df.columns:
    tier1_audit = availability_df[availability_df["evidence_tier"].str.startswith("Tier 1", na=False)]
    print(f"Tier 1 audit: {len(tier1_audit)} variants; tier count excludes position-only GWAS as allele-specific support.")
    display(tier1_audit[["chrom", "pos", "ref", "alt", "gwas_status", "eqtl_status",
                         "mpra_status", "crispr_status", "n_tested_experimental", "evidence_tier"]])


PASS: supplied Layer 0 table schema and uniqueness (112 candidates).
PASS: supplied MPRA schema (1383 rows); exact candidate-allele overlaps: 22.
PASS: GWAS position-level evidence is explicitly marked non-allele-specific.
GWAS audit: 6 candidate-allele rows are position-level catalog matches; do not interpret them as allele-specific without a verified effect allele match.


,chrom,pos,ref,alt,context,missingness_flag,matching_level,allele_specific,evidence_value
22,11,5225678,C,A,GWAS Catalog,observed_positive,position,False,"{""allele_specific"": false, ""associations"": [{""..."
23,11,5225678,C,G,GWAS Catalog,observed_positive,position,False,"{""allele_specific"": false, ""associations"": [{""..."
24,11,5225678,C,T,GWAS Catalog,observed_positive,position,False,"{""allele_specific"": false, ""associations"": [{""..."
52,11,5225911,G,A,GWAS Catalog,observed_positive,position,False,"{""allele_specific"": false, ""associations"": [{""..."
88,11,5227100,T,C,GWAS Catalog,observed_positive,position,False,"{""allele_specific"": false, ""associations"": [{""..."
89,11,5227100,T,G,GWAS Catalog,observed_positive,position,False,"{""allele_specific"": false, ""associations"": [{""..."


Tier 1 audit: 0 variants; tier count excludes position-only GWAS as allele-specific support.


,chrom,pos,ref,alt,gwas_status,eqtl_status,mpra_status,crispr_status,n_tested_experimental,evidence_tier


## 9. Final reproducibility and input audit


In [39]:
from pathlib import Path
import pandas as pd, json, hashlib

print("PROJECT INPUTS")
for p in [LAYER0_DEST, LAYER0_MANIFEST_PATH, MPRA_DEST, Path(CONFIG_PATH)]:
    print(f"{'FOUND' if p.exists() else 'OPTIONAL/MISSING'}: {p}")
if 'evidence_df' in globals() and evidence_df is not None:
    print("\nLayer 1 rows:", len(evidence_df))
    print("Layer 1 unique candidate keys:", evidence_df[["chrom","pos","ref","alt"]].drop_duplicates().shape[0])
    print("Layer 1 records by modality:")
    display(evidence_df.groupby(["modality","missingness_flag"]).size().rename("records").reset_index())
if 'availability_df' in globals():
    print("\nLayer 2 candidates:", len(availability_df))
    print("Layer 2 trigger rows:", len(triggers_df))
    print("Layer 2 agreement rows:", len(agreement_df))
    print("AlphaGenome API calls made by this notebook: 0 (trigger queue only)")
if "DATA_QUALITY" in globals():
    print(f"\nDATA QUALITY: {DATA_QUALITY}")
print("\nInterpretation note: a failed/incomplete query is not biological absence; inspect warnings and each evidence_value.reason.")


PROJECT INPUTS
FOUND: /content/layer0_outputs/HBB_layer0_regulatory_truthset.csv
FOUND: layer0_outputs/HBB_layer0_manifest.json
FOUND: /content/data/inputs/hbb_locus_mpra_evidence.csv
FOUND: configs/hbb.yaml

Layer 1 rows: 560
Layer 1 unique candidate keys: 112
Layer 1 records by modality:


,modality,missingness_flag,records
0,crispr,absent,112
1,eqtl,absent,112
2,gnomad_af,absent,69
3,gnomad_af,observed_positive,43
4,gwas,absent,106
5,gwas,observed_positive,6
6,mpra,absent,90
7,mpra,observed_null,4
8,mpra,observed_positive,18



Layer 2 candidates: 112
Layer 2 trigger rows: 202
Layer 2 agreement rows: 22
AlphaGenome API calls made by this notebook: 0 (trigger queue only)

DATA QUALITY: OK

Interpretation note: a failed/incomplete query is not biological absence; inspect warnings and each evidence_value.reason.


## Troubleshooting and interpretation

- **Layer 2 halts with "evidence queries failed":** a source (usually GWAS) returned an error. Read the diagnostic probe printed by the Layer 1 collection cell. With the default `layer1.gwas_source: auto` the notebook already falls back to the official bulk release file, so this means both paths failed: check access to `ftp.ebi.ac.uk`, then rerun Layer 1 with `layer1.refresh_cache: true`, or force `gwas_source: download`. Set `layer2.halt_on_query_failure: false` only if you accept `DEGRADED` outputs.
- **"SCOPE:" warnings:** evidence exists in a cell type/tissue that is not a configured target, so it is excluded from tiers and triggers. Fix `layer2.target_cell_types` / `target_tissues` or the table's `cell_type` values.
- **Tier 1 empty or tiny:** Tier 1 needs two allele-resolved sources (eQTL, MPRA, CRISPR) on the same variant. See `layer2_tier1_candidates_needing_second_source.csv` for the variants one more source would promote. This needs new data, not different code.
- **GTEx reports zero rows:** the dataset is pinned to `gtex_v8` / GENCODE v26 in the config. Zero significant rows is recorded as *unverified*, never as a tested null.
- **GWAS:** position-level association only; it does not count toward tiers and does not establish allele-specific causality.
- **No CRISPR table:** CRISPR is labelled unconfigured (not tested-null) and no CRISPR AlphaGenome jobs are queued.
- **Missing input CSV:** upload the Layer 0 truth set and MPRA evidence file, or set `layer0.run_layer0: true`.
- **AlphaGenome:** `layer2_alphagenome_triggers.csv` is a queue, not model output; output-type bindings stay unverified until Layer 3.
- **Re-running:** rerun from the top after changing configuration or code so stale in-memory definitions cannot shadow new ones.

**HBB metadata provenance:** Ensembl REST identifies `ENSG00000244734` on chromosome 11, GRCh38, 1-based coordinates 5,225,464–5,229,395, strand −1 (https://rest.ensembl.org/lookup/symbol/homo_sapiens/HBB); the Alliance of Genome Resources reports the same locus (https://www.alliancegenome.org/gene/HGNC%3A4827). This validates the gene metadata, not the assembly of every input variant file; confirm the truth-set assembly separately. The ClinVar snapshot date recorded for a *supplied* Layer 0 CSV is user-reported (from the original run) and cannot be verified from the CSV.


## Appendix

The former archived Layer 0 cells are now live code in Sections 3b/4b (disabled by default), so this appendix no longer holds a markdown copy.


In [40]:
  m = pd.read_csv("data/inputs/hbb_locus_mpra_evidence.csv")
  print(m.groupby(["cell_type", "dataset_name"]).pos.agg(["min", "max", "count"]))

                                           min      max  count
cell_type dataset_name                                        
HEL92.1.7 Kircher2019_satMutMPRA_HBB   5227022  5227208    561
          Kircher2019_satMutMPRA_HBG1  5249805  5250078    822


In [41]:
!zip -r hbb_layers0-2.zip data/outputs layer0_outputs data/inputs configs
from google.colab import files; files.download("hbb_layers0-2.zip")

  adding: data/outputs/ (stored 0%)
  adding: data/outputs/HBB/ (stored 0%)
  adding: data/outputs/HBB/layer2_agreement_checks.csv (deflated 81%)
  adding: data/outputs/HBB/layer1_manifest.json (deflated 66%)
  adding: data/outputs/HBB/layer1_evidence.csv (deflated 93%)
  adding: data/outputs/HBB/layer2_tier1_candidates_needing_second_source.csv (deflated 85%)
  adding: data/outputs/HBB/layer2_alphagenome_triggers.csv (deflated 93%)
  adding: data/outputs/HBB/layer2_manifest.json (deflated 64%)
  adding: data/outputs/HBB/layer2_modality_availability.csv (deflated 93%)
  adding: layer0_outputs/ (stored 0%)
  adding: layer0_outputs/HBB_layer0_manifest.json (deflated 47%)
  adding: layer0_outputs/HBB_layer0_regulatory_truthset.csv (deflated 94%)
  adding: data/inputs/ (stored 0%)
  adding: data/inputs/hbb_locus_mpra_evidence.csv (deflated 74%)
  adding: configs/ (stored 0%)
  adding: configs/hbb.yaml (deflated 46%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>